In [ ]:
import os, sys
import pandas as pd
import numpy as np
import anndata
import glob
from pym3c.hic import mcds2frac
from scipy.stats import pearsonr
import cooler
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pylab as plt
import matplotlib as mpl
import seaborn as sns
# mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'
# mpl.rcParams['figure.dpi'] = 80
# mpl.rcParams['savefig.dpi']=300
from PyComplexHeatmap import *

## Parameters

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Subclass"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Subclass/Subclass.mcds"
pseudobulk_atac_adata_path="~/Projects/BG/atac/Pseudobulk.Subclass.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Subclass"
group_col = "Subclass"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood", "Class"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 12
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ['Nonneuron','Subpallium GABA', 'Glut Sero Dopa', 'Subpallium GABA-Glut']
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Subclass/domain"
outname = "2.Subclass.domain_correlation.ipynb"


In [ ]:
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"figures")
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)

print(figdir,group_col)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
keep_chroms=[chrom for chrom in chrom_sizes.index if chrom not in ['chrX','chrY']]
chrom_sizes = chrom_sizes[keep_chroms]
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[metadata[group_col].apply(lambda x:not x.endswith('-Unk'))] #remove Exc-Unk
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
group_colors=D[group_col].Hex.to_dict()
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent={}
if not stratify_col is None:
    D1=D[stratify_col].Hex.to_dict()
    stratify_colors={k: D1[k] for k in D1 if k in stratify2group}
else:
    stratify_colors={}
print(group_colors)
print(parent_colors)
print(stratify_colors)
print(group2parent)

In [ ]:
os.chdir(os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/"))

In [ ]:
! pwd

## Plot correlation between chromatin interaction and mCG, mCH and ATAC at different resolution

In [ ]:
! ls */*/*.multiomics_correlation_across_bins.tsv

In [ ]:
# calculate pearson correlation for each cell type across all bins
D={}
for infile in glob.glob("*/*/*.multiomics_correlation_across_bins.tsv"):
    module=infile.split('/')[0]
    df1=pd.read_csv(infile,sep='\t',index_col=0).loc[:,['CG_corr','CH_corr','ATAC_corr']]
    df1.columns=[col.replace('_corr','') for col in df1.columns.tolist()]
    
    row_info=pd.Series(group2parent)
    row_info=row_info.loc[df1.index.tolist()]
    colors={}
    for parent in row_info.values:
        if parent not in row_info.tolist():
            continue
        colors[parent]=parent_colors[parent]
    row_ha_dict={
        parent_col:anno_simple(row_info,colors=colors,legend=True),
    }
    row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
             verbose=0,label_side='bottom',label_kws=dict(visible=False))
    show_rownames=True if module=='loop' else False
    left_annotation=row_ha if module=='compartment' else None
    cm = ClusterMapPlotter(
                    data=df1, left_annotation=left_annotation,vmin=-0.35,vmax=0.7,
                    show_rownames=show_rownames, show_colnames=True,col_cluster=False,row_cluster=True,
                    row_dendrogram=False, col_dendrogram=False,
                    row_split=row_info,xticklabels=True,yticklabels=True,
                    xticklabels_kws={'labelrotation':-30,'labelcolor':'black','labelsize':10,'bottom':True},
                    yticklabels_kws=dict(right=True),
                    row_split_gap=0.5,rasterized=True,#linecolor='white',linewidth=0.05,
                    plot=False,cmap='coolwarm',label='Correlation',annot=True,
                    legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
                    verbose=0,linewidth=0.08,linecolor='gold')
    D[module]=cm
cmlist=[D['compartment'],D['domain'],D['loop']]

In [ ]:
from PyComplexHeatmap.utils import despine
plt.figure(figsize=(6,5))
ax,legend_axes=composite(cmlist=cmlist, main=0,legend_hpad=120,col_gap=1,legend_gap=10)
module2res={'compartment':'100kb','domain':'25kb','loop':'10kb'}
for k in D:
    D[k].ax.set_title(module2res[k],fontsize=14)
# ax.set_ylabel(str(tmp3c.shape[0])+" Diff Boundaries",color='black',fontsize=14)
ax.set_axis_on()
despine(ax=ax, left=True, bottom=True,top=True,right=True)
ax.tick_params(axis="both",which="both",left=False,
                right=False,labelleft=False,labelright=False,
                top=False,bottom=False,labeltop=False,labelbottom=False)
# ax.set_title("Title",y=1.03,fontdict={'fontweight':'bold','color':'red'})
plt.savefig(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.pdf'), 
            transparent=True,bbox_inches='tight',dpi=300)
plt.show()

## Plot boxplot for domain size, insulation score, RNA and ATAC counts

In [ ]:
boundary=anndata.read_h5ad("domain/Subclass/cell_boundary.h5ad")
boundary.obs

In [ ]:
import xarray as xr
res=25000
ins_count = xr.open_dataset('Subclass_domain/25K_domain.insulation.nc') #pseudobulk level insulation score
# ins_count['ratio'] = ins_count['__xarray_dataarray_variable__']
judge = ins_count.get("bin_chrom").isin(['chrX','chrY','chrM','chrL'])
ins_count = ins_count.sel({'bin': ~judge})
ins = ins_count['__xarray_dataarray_variable__'].to_pandas().T #ratio
ins.columns=[col.replace('_',' ') for col in ins.columns.tolist()]
ins=ins.loc[:,boundary.obs.Subclass.unique().tolist()]

df_bin = ins_count[['bin_chrom', 'bin_start', 'bin_end']].to_pandas()
df_bin.columns = df_bin.columns.str.split('_').str[1]
df_bin.index = df_bin['chrom'] + '_' + (df_bin['start'] // res).astype(str)
ins.index=df_bin.index
print(np.nanmin(ins.values),np.nanmax(ins.values))
ins.head() #between 0-1

In [ ]:
adata=anndata.read_h5ad(os.path.expanduser("/home/x-wding2/Projects/BG/atac/Pseudobulk.Subclass.gene.h5ad"))
adata.X=adata.raw.X
use_genes=adata.var.query("gene_type=='protein_coding'").index.tolist()
cell_counts=adata.obs['cell_count'].to_dict()
atac_count=adata[:,use_genes].to_df()
atac_count=atac_count.apply(lambda x:x / cell_counts[x.name],axis=1)
atac_count=atac_count.stack().reset_index()
atac_count.columns=['Subclass','Gene','CountsPerCell']
atac_count=atac_count.loc[atac_count.CountsPerCell > 0]
atac_count

In [ ]:
adata=anndata.read_h5ad(os.path.expanduser("~/Projects/BICAN/adata/HMBA_v2/Pseudobulk.Subclass.h5ad"))
adata.X=adata.raw.X
use_genes=adata.var.query("gene_type=='protein_coding'").index.tolist()
cell_counts=adata.obs['cell_count'].to_dict()
rna_count=adata[:,use_genes].to_df()
rna_count=rna_count.apply(lambda x:x / cell_counts[x.name],axis=1)
rna_count=rna_count.stack().reset_index()
rna_count.columns=['Subclass','Gene','CountsPerCell']
rna_count=rna_count.loc[rna_count.CountsPerCell > 0]
rna_count

In [ ]:
# load pseudobulk boundary
bulk_boundary=anndata.read_h5ad('Subclass_domain/25K_domain.boundary.h5ad')
bulk_boundary.X = bulk_boundary.X.astype(int)
bulk_boundary = bulk_boundary[:, ~bulk_boundary.var['chrom'].isin(['chrX', 'chrY', 'chrM', 'chrL'])]
bulk_boundary.var.index=bulk_boundary.var['chrom'].astype(str) + '_' + (bulk_boundary.var['start'] // res).astype(str)
print(bulk_boundary.shape)
df_bulk_bound=bulk_boundary.to_df().T
df_bulk_bound.columns=[col.replace('_',' ') for col in df_bulk_bound.columns.tolist()]
df_bulk_bound=df_bulk_bound.stack()
# df_bulk_bound.columns=['ID',group_name,'boundary']
df_bulk_bound

In [ ]:
insulation_score=ins.stack().reset_index()
insulation_score.columns=['ID','Subclass',"Insulation score"] #group_name=group_col if stratify_col is None else 'GroupID'
insulation_score['boundary']=insulation_score.loc[:,['ID','Subclass']].apply(lambda x:tuple(x.tolist()),axis=1).map(df_bulk_bound.to_dict())
insulation_score=insulation_score.loc[insulation_score.boundary==2] # only calculate the median insulation score for bins with boundary in pseudobulk level
# order=data.groupby(group_name)['Insulation score'].median().sort_values().index.tolist()
insulation_score.head()

In [ ]:
# log2(short/long)
D=pd.read_csv(os.path.expanduser("~/Projects/BG/HIC/contact_distance/log2_short_long_ratio.tsv"),sep='\t',index_col=0)['log2(short/long)'].to_dict()
metadata['log2(short/long)']=metadata.index.to_series().map(D)
metadata.loc[:,['Subclass','log2(short/long)']]

In [ ]:
# Calculate loop distance (contact distance) and loop Q (strength)
indir="/home/x-wding2/Projects/BG/HIC/BG_Subclass/Subclass_loop"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
res=10000
m='Q'

if not os.path.exists(os.path.join(figdir,"merged_loop.tsv")):
    df_loop=[]
    for ct in boundary.obs[group_col].unique():
        sname=ct.replace(' ','_')
        infile=os.path.join("/anvil/projects/x-mcb130189/Wubin/BG/HIC/BG_Subclass/Subclass_loop",sname,sname,f"{sname}.loop.bedpe")
        if not os.path.exists(infile):
            continue
        with open(infile,'r') as f:
            N=len(f.readlines())
        if N==0:
            continue
        df1=pd.read_csv(infile,sep='\t',index_col=None, header=None,usecols=[0,1,2,3,4,5],names=['chrom1','start1','end1','chrom2','start2','end2'])
        # load loop strength (Q)
        tmp = []
        sname=ct.replace(' ','_')
        if '+' in sname:
            group=sname.split('+')[0]
        else:
            group=sname
        cool_file = cooler.Cooler(os.path.join(indir,f'{group}/{sname}/{sname}.{m}.cool')).matrix(balance=False, sparse=True)
        for chrom in df1.chrom1.unique().tolist():
            mat = cool_file.fetch(chrom).tocsr()
            tmp.append(mat[(df1.loc[df1['chrom1']==chrom, 'start1'].values // res, df1.loc[df1['chrom1']==chrom, 'start2'].values // res)].A1)
        df1['loop_strength']=np.concatenate(tmp)
        df1['Subclass']=ct

        df_loop.append(df1)
    df_loop = pd.concat(df_loop, axis=0)
    df_loop['loop_distance']=(df_loop.start2 + df_loop.end2) / 2 - (df_loop.start1 + df_loop.end1) / 2
    df_loop=df_loop.loc[df_loop.loop_distance.notna()]
    df_loop.to_csv(os.path.join(figdir,"merged_loop.tsv"),sep='\t',index=False)
else:
    df_loop=pd.read_csv(os.path.join(figdir,"merged_loop.tsv"),sep='\t')
df_loop.loop_distance=(df_loop.loop_distance / 1000000) # x Mb
df_loop

In [ ]:
# No. of diff boundaries
diff_boundary=pd.read_csv(os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/domain/Subclass/diff_boundary.tsv"),sep='\t',index_col=0)
# diff_boundary.probdiff.min()
# diff_boundary['ins_peak'].unique()
diff_boundary['Group'].value_counts()

In [ ]:
# No. of diff loop
diff_loop=pd.read_csv(os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/loop/Subclass/diff_loops.tsv"),sep='\t',index_col=0)
diff_loop['Group'].value_counts()

In [ ]:
# obs=pd.read_csv(os.path.expanduser("~/Projects/BICAN/adata/HMBA_v2/obs.tsv"),sep='\t',index_col=0)
# D=pd.read_csv(os.path.expanduser("~/Projects/BICAN/adata/HMBA_v2/activity_scores.tsv"),sep='\t',index_col=0)['predicted_activity'].to_dict()
# obs['predicted_activity']=obs.index.to_series().map(D)
# obs=obs.loc[obs.predicted_activity.notna()]
# order1=obs.groupby('Group')['predicted_activity'].median().sort_values().index.tolist()
# D=pd.read_excel(os.path.expanduser(palette_path),
#                 sheet_name=None, index_col=0)
# colors=D['Group'].Hex.to_dict()
# order=[o for o in order1 if o in colors]
# fig, ax = plt.subplots(figsize=(7, 3), dpi=300)
# sns.boxplot(data=obs, x='Group', y='predicted_activity',
#             order=order, showfliers=False, palette=colors, ax=ax,linewidth=0.5)
# ax.grid(axis='y',which='major',linestyle='--')
# ax.set_yticks([0.3,0.4,0.5,0.6,0.8,1])
# ax.text(0.5, 0.9, "Predicted Neuronal Activity",ha='center', va='top', color='darkred', fontsize=10,transform=ax.transAxes) #lower right / lower left
# ax.yaxis.tick_right()
# ax.yaxis.set_label_position('right')
# ax.xaxis.label.set_visible(False)
# ax.yaxis.label.set_visible(False)
# ax.set_xticklabels(order, rotation=-45,ha='left',fontsize=8,va='center',rotation_mode='anchor') #rotation=45,ha='left',va='top'
# for tick in ax.get_xticklabels():
#     color=colors[tick.get_text()]
#     tick.set_color(color)
# plt.tight_layout()
# plt.savefig(os.path.join(figdir,f'Group.predicted_activity.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)

In [ ]:
# Plot domain size and insulation score together
from seaborn import despine
group_col='Subclass'
order=boundary.obs.groupby(group_col)['domain_size'].median().sort_values().index.tolist()
ncols=len(order)

fig, axes = plt.subplots(8, 1, figsize=(7, 5.6), sharex='all',
                         height_ratios=[15,15,10,10,15,15,15,15],
                         dpi=300,gridspec_kw=dict(hspace=0.01))

ax=axes[0]
ax=sns.boxplot(data=rna_count, x=group_col, y='CountsPerCell',order=order,
            showfliers=False, 
                  palette=group_colors,ax=ax,linewidth=0.5)
# ax.set_xticklabels(order, rotation=90)
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([0,2,4])
# ax.set_ylim([0,4])
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--')
despine(ax=ax,top=False,bottom=True,right=False)
# ax.set_title("RNA (mean counts)",y=0.68,x=0.8)
ax.text(0.72, 0.76, "RNA (mean counts for expressed genes)",ha='center', va='top', color='red',fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

ax=axes[1]
ax=sns.boxplot(data=atac_count, x=group_col, y='CountsPerCell',order=order,
            showfliers=False, 
                  palette=group_colors,ax=ax,linewidth=0.5)
# ax.set_xticklabels(order, rotation=90)
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([0,2,4])
# ax.set_ylim([0,4])
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--')
despine(ax=ax,top=False,bottom=True,right=False)
# ax.set_ylabel("Insulation score")
# ax.set_title("ATAC (mean counts)",y=0.68,x=0.8)
ax.text(0.72, 0.9, "ATAC (mean counts for expressed genes)",ha='center', va='top', color='orange', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

vc=diff_loop['Group'].value_counts() #.apply(lambda x:np.log10(x))
ax=axes[2]
ax=sns.barplot(vc.reindex(index=order),palette=group_colors,ax=ax,zorder=3,
               linewidth=0.5,edgecolor='black')
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([100,5000,10000,20000])
ax.set_ylim([0,20000])
ax.tick_params(axis='y', labelsize=6)
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--',zorder=1)
despine(ax=ax,top=False,bottom=True,right=False)
ax.text(0.72, 0.8, "No. of diff loops",ha='center', va='top', color='purple', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

vc=diff_boundary['Group'].value_counts() #.apply(lambda x:np.log2(x))
ax=axes[3]
ax=sns.barplot(vc.reindex(index=order),palette=group_colors,ax=ax,zorder=3,
              linewidth=0.5,edgecolor='black')
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([5,100,200,500])
ax.set_ylim([0,500])
ax.tick_params(axis='y', labelsize=6)
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--',zorder=1)
despine(ax=ax,top=False,bottom=True,right=False)
ax.text(0.72, 0.8, "No. of diff boundaries",ha='center', va='top', color='chocolate', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

df=metadata.loc[:,['Subclass','log2(short/long)']].dropna()
ax=axes[4]
ax=sns.violinplot(data=df, x=group_col, y='log2(short/long)',order=order,
            # showfliers=False, 
                  palette=group_colors,ax=ax,linewidth=0.5)
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([-1,0,1,2,3])
ax.set_ylim([-1,4])
ax.tick_params(axis='y', labelsize=8)
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--')
despine(ax=ax,top=False,bottom=True,right=False)
ax.text(0.72, 0.97, "log2(short/long contact distance)",ha='center', va='top', color='deepskyblue', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

# ax=axes[5] # loop strength are similar in different cell types
# ax=sns.boxplot(data=df_loop, x=group_col, y='loop_strength',order=order,
#             showfliers=False, palette=group_colors,ax=ax,linewidth=0.5)
# ax.xaxis.label.set_visible(False)
# ax.yaxis.label.set_visible(False)
# # ax.set_yticks([0.1,0.5,1,2])
# ax.tick_params(axis='y', labelsize=8)
# ax.yaxis.tick_right()
# ax.yaxis.set_label_position('right')
# ax.grid(which='major',axis='y',linestyle='--')
# despine(ax=ax,top=False,bottom=True,right=False)
# ax.text(0.8, 0.97, "Loop strength",ha='center', va='top', color='skyblue', fontsize=10,transform=ax.transAxes) #lower right / lower left
# ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

ax=axes[5]
ax=sns.boxplot(data=df_loop, x=group_col, y='loop_distance',order=order,
            showfliers=False, 
                  palette=group_colors,ax=ax,linewidth=0.5)
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([0.1,0.5,1,2])
# ax.set_ylim([0,2])
ax.tick_params(axis='y', labelsize=8)
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.grid(which='major',axis='y',linestyle='--')
despine(ax=ax,top=False,bottom=True,right=False)
ax.text(0.72, 0.97, "Loop distance (Mb)",ha='center', va='top', color='magenta', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

ax=axes[6]
ax=sns.violinplot(data=insulation_score, x=group_col, y='Insulation score',order=order,
            # showfliers=False, 
                  palette=group_colors,ax=ax,linewidth=0.5)
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_yticks([0.1,0.2,0.3])
ax.set_ylim([0,0.3])
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
# ax.set_ylabel("Median boundary\ninsulation score",fontsize=12)
ax.grid(which='major',axis='y',linestyle='--')
# ax.set_title("Insulation score",y=0.68,x=0.8)
despine(ax=ax,top=False,bottom=True,right=False)
ax.text(0.72, 0.97, "Boundary insulation score",ha='center', va='top', color='blue', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)

ax = axes[7]
df=boundary.obs.copy()
df['domain_size'] = df['domain_size'] / 1000000
sns.boxplot(data=df, x=group_col, y='domain_size',order=order, 
               showfliers=False, palette=group_colors, ax=ax,linewidth=0.5)
ax.grid(axis='y',which='major',linestyle='--')
# ax.set_title("Domain Size",y=0.68,x=0.8)
ax.set_yticks([0.5,0.75,1])
# ax.set_ylim([0,1])
ax.text(0.72, 0.97, "Domain Size (Mb)",ha='center', va='top', color='green', fontsize=10,transform=ax.transAxes) #lower right / lower left
ax.yaxis.tick_right()
ax.yaxis.set_label_position('right')
ax.xaxis.label.set_visible(False)
ax.yaxis.label.set_visible(False)
ax.set_xticklabels(order, rotation=-45,ha='left',fontsize=8,va='center',rotation_mode='anchor') #rotation=45,ha='left',va='top'
for tick in ax.get_xticklabels():
    try:
        color=parent_colors[group2parent[tick.get_text()]]
    except:
        color=group_colors[tick.get_text()]
    tick.set_color(color)
plt.tight_layout()
plt.savefig(os.path.join(figdir,f'{group_col}.multiomics_stats.violinplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)

## Plot domain Heatmap (schicluster)

In [ ]:
from scipy import ndimage as nd
import matplotlib
from seaborn import despine
from matplotlib.patches import Polygon
# from PyComplexHeatmap import *
indir=os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/Subclass_loop")
domain_bed_dir=os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/domain/Subclass/domain")
loop_bedpe_dir=os.path.expanduser("~/Projects/BG/Browser/Tracks/Subclass/loop.bedpe/")
gene = 'SLC17A6'
var_meta="~/Ref/hg38/annotations/gene_2000_2000.bed.gz"
resolution=10*1000
lslop, rslop = 1e6, 1e6 #2Mb upstream and downstream

genemeta=pd.read_csv(os.path.expanduser(var_meta), sep='\t',
							usecols=[0,1,2, 4], header=None,
							names=['chrom', 'start','end','gene_name']).drop_duplicates()
gene_info = genemeta.query("gene_name==@gene")
chrom=gene_info.chrom.tolist()[0]
start=gene_info.start.min()
end=gene_info.end.max()

bin_start, bin_end = int((start - lslop) // resolution), int((end + rslop) // resolution)
print(chrom,bin_start,bin_end)
cmap='afmhot_r'

In [ ]:
celltypes=["F_GABA",'CN_ST18_GABA','Microglia']
nrows=len(celltypes)
vmin=0
vmax=0.012
fig, axes = plt.subplots(nrows, 1, figsize=(8,nrows*1), dpi=300, 
                         sharex='all',gridspec_kw=dict(hspace=0.01))
for i,ct in enumerate(celltypes):
    ax=axes[i]
    infile=os.path.expanduser(f"{indir}/{ct}/{ct}/{ct}.Q.cool")
    cool = cooler.Cooler(infile)
    Q = cool.matrix(balance=False, sparse=True).fetch(chrom).tocsr()
    # print(Q.shape)
    tmp = Q[bin_start:bin_end, bin_start:bin_end].toarray()
    # tmp=Q.toarray()
    dst = nd.rotate(tmp, 45, order=0, reshape=True, prefilter=False, cval=0)
    
    # vmax=np.round(np.percentile(dst,99),3)
    # fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    img = ax.imshow(dst, cmap=cmap,vmin=0,vmax=vmax) #vmin=0, vmax=0.012
    h = len(dst)
    ylim=[0.5*h, 0.4*h]
    ax.set_title(ct,x=0.5,y=0.99,transform=ax.transAxes,ha='center',va='center')
    ax.set_ylim(*ylim) #ylim values are from largest to smallest, figure direction is from top to bottom,
    # print(ylim)
    ax.set_xlim([0, h])
    despine(ax=ax,top=True,bottom=True,left=True,right=True)
    ax.tick_params(
        axis="both",which="both",left=False,right=False,labelleft=False,labelright=False,
        top=False,bottom=False,labeltop=False,labelbottom=False
    )
    
# ax=plt.gca()
ax=axes[nrows // 2]
cax = fig.add_axes(rect=[1,ax.get_position().y0,ax.get_position().width*0.03,ax.get_position().height],xmargin=0, ymargin=0) # from top to bottom
# unit is fractions of figure width and height, left,bottom,width,height
m = plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=0, vmax=vmax), cmap=cmap)
cbar = ax.figure.colorbar(m, cax=cax,label="Interaction\nfreqency",ticks=[vmin, vmax],
                         location='left')
cax.set_ylim([vmin, vmax])
plt.tight_layout()
plt.savefig(os.path.join(figdir,'Subclass.domain.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

## Pseudobulk level domain size

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
celltypes=["CN ONECUT1 GABA","CN LHX8 GABA",'M Dopa',"F Glut", #"F M Glut","F Glut",
           'STR Hybrid MSN','STR D2 MSN','STR D1 MSN',
           'Astrocyte','Oligodendrocyte','Lymphocyte']

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from pycirclize.utils import ColorCycler

# Sample chromosome sizes (e.g., human hg38, subset for simplicity)
chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[chrom_sizes.ID!=25]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# chrom_sizes
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes['chr4']=chrom_sizes['chr4']

# Sample TAD data for two cell types (replace with your BED files)
# diff_boundary=pd.read_csv(os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/domain/Subclass/diff_boundary.tsv"),sep='\t',index_col=0)
# diff_boundary=diff_boundary.loc[:,['chrom','start','end','probdiff','insfc','Group']]
# diff_boundary_dict=diff_boundary.Group.to_dict()

celltypes=['CN GABA-Glut','M Dopa','STR Hybrid MSN','Astrocyte','Lymphocyte']
tad_dfs={}
for ct in celltypes:
    ct_name=ct.replace(' ','_')
    df=pd.read_csv(f"{domain_bed_dir}/{ct_name}.bed",sep='\t',header=None,names=['chrom','start','end'])
    df.index=df['chrom'] + '_' + (df['start'] // 25000).astype(str)
    # df['diff_boundary_group']=df.index.to_series().map(diff_boundary_dict)
    tad_dfs[ct]=df


# loop_dfs={}
# for ct in celltypes:
#     ct_name=ct.replace(' ','_')
#     df=pd.read_csv(f"{loop_bedpe_dir}/{ct_name}.loop.bedpe",sep='\t',header=None,names=['chrom1','start1','end1','chrom2','start2','end2','score'])
#     loop_dfs[ct]=df

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
circos.text("Domain", size=15)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)
# Plot chromosome name
for sector in circos.sectors:
    sector.text(sector.name, size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

# Cell Type TADs
y=92
for ct in celltypes:
    print(ct)
    color=group_colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.1)
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=6,color=color,ha='center',va='top')
        track.axis(fc='white')
        # track.text(ct, r=y-2.5, size=8, color="black")
        tads = tad_dfs[ct].loc[tad_dfs[ct]['chrom']==sector.name]
        for _, tad in tads.iterrows():
            # ec=color if tad['diff_boundary_group']==ct else 'white'
            track.rect(tad["start"], tad["end"], fc='grey', ec='black', lw=0.1,rasterized=True)
    y=y-6

# # Save the plot
circos.savefig(os.path.join(figdir,'Subclass.comprison.chr4.domain.ciscos_plot.pdf'), dpi=300, figsize=(5,5))
fig = circos.plotfig()

## Plot domain size on single cell (1 cell) level to be consistent with domain size

In [ ]:
boundary=anndata.read_h5ad("domain/Subclass/cell_boundary.h5ad")
boundary.var['ID']=boundary.var.apply(lambda x:'-'.join([str(i) for i in x]),axis=1)
usecells=boundary.obs.groupby('Subclass').apply(lambda x:x.sample(1).index.tolist()).sum()
boundary=boundary[usecells,:]
subclasses=boundary.obs['Subclass'].tolist()
boundary_df=boundary.to_df()
boundary_df.columns=boundary.var['ID'].tolist()
boundary_df.index=subclasses
boundary_df=boundary_df.T
boundary_df

In [ ]:
# convert boundary to domain
R=[]
for col in boundary_df.columns.tolist(): # cell type level domain doundaries, not on single cell level
    df=boundary_df.loc[boundary_df[col]>0,col]
    # 1 means either start or end of a domain,
    # 2 means both start and end of two domains
    bounds=df.index.tolist()
    values=df.tolist()
    i=0
    while i <= len(bounds)-2:
        chrom1,start1,end1=bounds[i].split('-')
        chrom2,start2,end2=bounds[i+1].split('-')
        if chrom1!=chrom2:
            continue
        start=(int(start1)+int(end1)) / 2
        end=(int(start2)+int(end2)) / 2
        R.append([chrom1,int(start),int(end),col])
        boundary_n=values[i+1]
        if boundary_n==2:
            i+=1
        else:
            i+=2
df=pd.DataFrame(R,columns=['chrom','start','end','Group'])
df

In [ ]:
500000000 / 1e6

In [ ]:
df.loc[df.chrom=='chr4','Group'].value_counts()

In [ ]:
# domain size at single cell level
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from pycirclize.utils import ColorCycler

# Sample chromosome sizes (e.g., human hg38, subset for simplicity)
chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[chrom_sizes.ID!=25]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# chrom_sizes
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes['chr4']=chrom_sizes['chr4']

celltypes=['CN GABA-Glut','M Dopa','STR Hybrid MSN','Astrocyte','Lymphocyte']
tad_dfs={}
for ct in celltypes:
    df1=df.loc[df.Group==ct]
    df1.index=df1['chrom'] + '_' + (df1['start'] // 25000).astype(str)
    tad_dfs[ct]=df1

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
circos.text("Domain", size=15)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)
# Plot chromosome name
for sector in circos.sectors:
    sector.text(sector.name, size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

# Cell Type TADs
y=92
for ct in celltypes:
    print(ct)
    color=group_colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.1)
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=6,color=color,ha='center',va='top')
        track.axis(fc='white')
        # track.text(ct, r=y-2.5, size=8, color="black")
        tads = tad_dfs[ct].loc[tad_dfs[ct]['chrom']==sector.name]
        for _, tad in tads.iterrows():
            # ec=color if tad['diff_boundary_group']==ct else 'white'
            track.rect(tad["start"], tad["end"], fc='grey', ec='black', lw=0.1,rasterized=True)
    y=y-6

# # Save the plot
circos.savefig(os.path.join(figdir,'Subclass.one_cell.chr4.domain.ciscos_plot.pdf'), dpi=300, figsize=(5,5))
fig = circos.plotfig()

## Merge compartment into megadomain/superdomain

In [ ]:
import cooler
level="Class"
cool = cooler.Cooler(os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/Subclass_raw/merged.mcool::resolutions/100000"))
bins_df = cool.bins()[:]
# bins_df.index = bins_df['chrom'].astype(str) + '-' + (bins_df['start'] // res).astype(str)
# bins_df

D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
colors=D[level].Hex.to_dict()
group='merged'
group_name=group.replace(' ','_')
diff_comp_dir=os.path.expanduser(f"~/Projects/BG/HIC/BG_{level}/compartment/{level}DiffComp")
infile=os.path.join(diff_comp_dir,
                f'{group_name}/DifferentialResult/{group_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
df_comp = pd.read_csv(infile, sep='\t')
# res=100000
# df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') for col in df_comp.columns.tolist()]
df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
usecols=[col for col in df_comp.columns.tolist() if col in colors]
df_comp=df_comp.loc[:,['chr','start','end']+usecols]
df_comp.set_index(['chr','start','end'],inplace=True)
# df_comp

idx=bins_df.apply(lambda x:tuple(x.tolist()),axis=1)
for col in df_comp.columns.tolist():
    bins_df[col]=idx.map(df_comp[col].to_dict()).fillna(0)
bins_df=bins_df.loc[~ bins_df['chrom'].isin(['chrX','chrM','chrY'])]
bins_df.chrom=bins_df.chrom.astype(str)
bins_df.set_index(['chrom','start','end'],inplace=True)
df_comp=bins_df.copy()
    
R=[]
for ct in df_comp.columns.tolist():
    df=df_comp[ct].to_frame().reset_index()
    df.rename(columns={ct:'score'},inplace=True)
    df['AB']=df.score.apply(lambda x:'A' if x > 0 else 'B' if x < 0 else 'NA')
    df['group'] = (df['AB'] != df['AB'].shift()) | (df['chrom'] != df['chrom'].shift())
    # shift: delete the last row and move every row afterward, equal to previous row after shift
    df['group'] = df['group'].cumsum()
    # df['diff_gap']=(df.start-df.start.shift(periods=1)).fillna(100000)!=100000
    # df['diff_gap']=df['diff_gap'].cumsum()
    # df=df.groupby(['chrom','AB','group','diff_gap'],as_index=False).agg({'start':'min','end':'max','score':'mean'})
    df=df.groupby(['chrom','AB','group']).agg({'start':'min','end':'max','score':'mean'}).reset_index()
    df.sort_values('group',inplace=True)
    df['CellType']=ct
    R.append(df.loc[:,['chrom','start','end','CellType','AB','score']])
data=pd.concat(R)
# data.start=data.start.astype(int)
# data.end=data.end.astype(int)
data.sort_values(['chrom','start','end','CellType','AB'],inplace=True)
data.to_csv(os.path.join(figdir,f'{level}.superdomain.tsv'),sep='\t',index=False)
data

## Plot 100kb compartments scores & H3K9me3

In [ ]:
group='merged'
# level="Subclass"
# chrom="chr4" # chr10 for Class and chr4 for Subclass
level="Class"
chrom="chr10"
res=100000
marker="H3K9me3" # H3K27ac, H3K9me3, H3K27me3
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
colors=D[level].Hex.to_dict()
group_name=group.replace(' ','_')
diff_comp_dir=os.path.expanduser(f"~/Projects/BG/HIC/BG_{level}/compartment/{level}DiffComp")
infile=os.path.join(diff_comp_dir,
                f'{group_name}/DifferentialResult/{group_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
df_comp = pd.read_csv(infile, sep='\t')
df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') for col in df_comp.columns.tolist()]
df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
usecols=[col for col in df_comp.columns.tolist() if col in colors]
df_comp=df_comp.loc[:,['chr','start','end']+usecols]
df_comp

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
indir=os.path.expanduser("~/Projects/BG/PairedTag/BGC_paired-Tag_fragment/")
obs=pd.read_csv(os.path.join(indir,f"{marker}.obs.tsv"),sep='\t',index_col=0)
vc=obs[level].value_counts()
keep_celltypes=vc[vc>=500].index.tolist()
if level == 'Class':
    celltypes1=["M Dopa","F M Glut",#'CN GABA-Glut',
               "F M GABA", 'CN LGE GABA','CN MGE GABA','CN CGE GABA',
           'Astro-Epen','Vascular','OPC-Oligo','Immune']
elif level=='Subclass':
    # celltypes1=["CN ONECUT1 GABA","CN LHX8 GABA",'M Dopa',"F Glut","F M GATA3 GABA", 
    #        'STR D1 MSN','STR D2 MSN','STR Hybrid MSN','CN ST18 GABA','Astrocyte',
    #        'Oligodendrocyte','OPC','Microglia','Lymphocyte']
    celltypes1=[
           'STR D1 MSN','STR D2 MSN','STR Hybrid MSN',"F M GATA3 GABA", 'M Dopa',"CN ONECUT1 GABA","CN LHX8 GABA","F Glut",'CN ST18 GABA','Astrocyte',
           'Oligodendrocyte','OPC','Microglia','Lymphocyte']
else:
    celltypes1=df_comp.columns.tolist()[3:]
celltypes=[ct for ct in celltypes1 if ct in keep_celltypes]
print(celltypes)

In [ ]:
from pym3c.adata import merge_adata_regions
peak=merge_adata_regions(
		os.path.expanduser(f"~/Projects/BG/PairedTag/{marker}.Pseudobulk.{level}.5kb.h5ad"),bin_size=5000,use_raw=True,
		res=res,filter_chroms=True,boundary=False,
		exclude_chroms=['chrY','chrM','chrX'])
peak=peak.T
peak.insert(0,'chrom',peak.index.to_series().apply(lambda x:x.split('_')[0]))
peak.insert(1,'start',peak.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
peak

In [ ]:
# peak_dir=os.path.expanduser(f"~/Projects/BG/PairedTag/{level}.{marker}.peaks")
# print(res,peak_dir)
vmax=0
for ct in celltypes:
    df=df_comp.loc[:,ct]
    a,b=df.quantile(0.05),df.quantile(0.95)
    print(ct,a,b)
    m=max(abs(a),abs(b))
    if m > vmax:
        vmax=m
vmax=round(vmax,2)
print(vmax)

In [ ]:
peak_vmin=1
peak_vmax=1
for ct in celltypes:
    df=peak.loc[:,ct]
    a,b=df.quantile(0.05),df.quantile(0.95)
    print(ct,a,b)
    if a < peak_vmin:
        peak_vmin=a
    if b > peak_vmax:
        peak_vmax=b
peak_vmin=round(peak_vmin,2)
peak_vmax=round(peak_vmax,2)
print(peak_vmin,peak_vmax)

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from matplotlib.patches import Patch,Rectangle

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
# circos.text("Compartments\nscores (100kb)", size=15)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((97, 100), cytoband_file)
# Plot chromosome name
circos.text(chrom, r=98.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
#     sector.text(sector.name.replace('chr',''), size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6,#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=95
vmin=-1 * vmax
gap=0.5 if marker != 'H3K9me3' else 1
for ct in celltypes[::-1]:
    print(ct)
    comp=df_comp.loc[:,['chr','start','end',ct]]
    # infile=os.path.join(peak_dir,f"{ct.replace(' ','_')}.macs3.peak.tsv")
    # if not os.path.exists(infile):
    #     continue
    # peak=pd.read_csv(infile,sep='\t',usecols=['chrom','start','end','score'])
    color=colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-4.5, y), r_pad_ratio=0.05)
        # size=6 if ct=='OPC-Oligo' else 5 if ct=='Oligodendrocyte' else 5.5 if ct=='Lymphocyte' else 6.5
        size=6
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center-1, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        if ct=='STR D2 MSN' or ct=='OPC-Oligo':
            track.yticks([vmin,vmax], [str(vmin),str(vmax)],vmin=vmin,vmax=vmax,label_size=4)
        comp1 = comp.loc[comp['chr']==sector.name]
        for _, row in comp1.iterrows():
            signal=row[ct] if row[ct] < vmax and row[ct] > vmin else vmax if row[ct] >= vmax else vmin
            if row[ct] > 0:
                track.fill_between(x=[row['start'],row['end']], y1=[signal,signal], y2=0, vmin=vmin,vmax=vmax, 
                                   color="red",rasterized=True) #alpha=0.6, 
            else:
                track.fill_between(x=[row['start'],row['end']], y1=[signal,signal], y2=0, vmin=vmin,vmax=vmax, 
                                   color="blue",rasterized=True) # 

        # add track for histone marker
        track = sector.add_track((y-6.75, y-4.5), r_pad_ratio=0)
        track.axis(fc='white')
        peak1 = peak.loc[peak['chrom']==sector.name]
        if peak1.shape[0]==0:
            continue
        if ct=='F M GATA3 GABA' or ct == 'Astro-Epen':
            track.yticks([peak_vmin,peak_vmax], [str(peak_vmin),str(peak_vmax)],vmin=peak_vmin,vmax=peak_vmax,label_size=4)
        for _, row in peak1.iterrows():
            end=min(row['start']+res,sector.size)
            signal=row[ct] if row[ct] < peak_vmax and row[ct]>peak_vmin else peak_vmax if row[ct] > peak_vmax else peak_vmin
            track.fill_between(x=[row['start'],end], y1=[signal,signal], y2=peak_vmin, vmin=peak_vmin,vmax=peak_vmax, 
                               color=color,rasterized=True) #
    # y=y-5.7
    y=y-6.75-gap

# circos.savefig(os.path.join(figdir,'Subclass.compartment_socres.ciscos_plot.pdf'), dpi=300, figsize=(5,5))
# fig = circos.plotfig()

handles = [
    Patch(facecolor="red", edgecolor='black',linewidth=0.1,label="A Compartment"),
    Patch(facecolor="blue", edgecolor='black',linewidth=0.1,label="B Compartment"),
    # Patch(facecolor="white", edgecolor='black',linewidth=0.1,label=f"{marker} (logCPM)"),
]
width1=5 if level=='Class' else 6
width=4.5 if marker=='H3K9me3' and level=='Subclass' else width1
fig = circos.plotfig(dpi=300, figsize=(width,width))
# circos.ax.set_rasterized(True)
lgd = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.5), loc="center", 
                     fontsize=8, title=f"100kb {marker}\n& compartment\nscores",
                     title_fontsize=12,frameon=False)
lgd.get_title().set_ha('center')
fig.savefig(os.path.join(figdir,f'{level}.{chrom}.{marker}.compartment_socres.ciscos_plot.pdf'))
# display(circos.figure)

## Plot superdomain (A/B compartments)

In [ ]:
group='merged'
# level="Subclass"
# chrom="chr4"
level="Class"
chrom="chr10"
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
colors=D[level].Hex.to_dict()
df_comp=pd.read_csv(os.path.join(figdir,f'{level}.superdomain.tsv'),sep='\t')
df_comp=df_comp.loc[df_comp.AB.notna()]
df_comp

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
if level=='Subclass':
    celltypes=[
               'STR D1 MSN','STR D2 MSN','STR Hybrid MSN',"F M GATA3 GABA", 'M Dopa',"CN ONECUT1 GABA","CN LHX8 GABA","F Glut",'CN ST18 GABA','Astrocyte',
               'Oligodendrocyte','OPC','Microglia','Lymphocyte']
else:
    celltypes=["M Dopa","F M Glut",'CN GABA-Glut',"F M GABA", 
           'CN LGE GABA','CN MGE GABA','CN CGE GABA',
           'Astro-Epen','Vascular','OPC-Oligo','Immune']
print(celltypes)

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from matplotlib.patches import Patch,Rectangle

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
circos.add_cytoband_tracks((97, 100), cytoband_file)
# Plot chromosome name
circos.text(chrom, r=98.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
#     sector.text(sector.name.replace('chr',''), size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6,#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=95
for ct in celltypes[::-1]:
    print(ct)
    comp=df_comp.loc[df_comp.CellType==ct,['chr','start','end','score']]
    color=colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.05)
        size=6 if ct!='Lymphocyte' else 5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center-1, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        comp1 = comp.loc[comp['chr']==sector.name]
        for _, row in comp1.iterrows():
            signal=row['score']
            if signal > 0:
                color='red'
            else:
                color='blue'
            track.rect(row["start"], row['end'], fc=color, ec='none', lw=0.01,rasterized=True)
    y=y-5.7
fig = circos.plotfig(dpi=300, figsize=(8,8))
fig.savefig(os.path.join(figdir,f'{level}.{chrom}.superdomain.ciscos_plot.pdf'))
# display(circos.figure)

In [ ]:
# for Subclass
metadata=pd.read_csv(os.path.expanduser("~/Projects/BG/clustering/100kb/annotations.tsv"),sep='\t',index_col=0)
subclass2neighborhood=metadata.loc[:,['Subclass','Neighborhood']].drop_duplicates().set_index('Subclass').Neighborhood.to_dict()
vc=df_comp.loc[:,['CellType','AB']].value_counts()
df_comp['length']=df_comp.end-df_comp.start
avg_size=df_comp.groupby(['CellType','AB']).length.sum() / vc
avg_size=avg_size.unstack(level=1).sort_values('B')
avg_size.rename(columns={'A':'A.avg_size','B':'B.avg_size'},inplace=True)
vc=vc.unstack(level=1)
avg_size['#A']=vc.loc[avg_size.index.tolist(),'A'].tolist()
avg_size['#B']=vc.loc[avg_size.index.tolist(),'B'].tolist()
avg_size.sort_values('#B',inplace=True)
avg_size['Neighborhood']=avg_size.index.to_series().map(subclass2neighborhood)
avg_size['Neu']=avg_size.Neighborhood != 'Nonneuron'
avg_size.to_csv(os.path.join(figdir,f'{level}.superdomain.sizes.tsv'))
avg_size

In [ ]:
import scipy.stats as stats
# one way ANOVA across Neighborhood
R=[]
for col in ['A.avg_size','B.avg_size','#A','#B']:
    values=[avg_size.loc[avg_size.Neighborhood==neighborhood,col].values for neighborhood in avg_size.Neighborhood.unique()]
    f_stat, p_value = stats.f_oneway(*tuple(values))
    R.append([col,f_stat,p_value])
df=pd.DataFrame(R,columns=['Var','f_stat','pvalue'])
df

In [ ]:
import scipy.stats as stats
R=[]
for col in ['A.avg_size','B.avg_size','#A','#B']:
    statistic, p_value = stats.mannwhitneyu(avg_size.loc[avg_size.Neu==True,col].values,avg_size.loc[avg_size.Neu==False,col].values,
                                           )
    R.append([col,statistic,p_value])
    
df=pd.DataFrame(R,columns=['Var','statistic','pvalue'])
df

In [ ]:
df=pd.melt(avg_size.reset_index(),id_vars=['CellType','Neighborhood','Neu'],value_vars=['A.avg_size','B.avg_size'],var_name='AB',value_name='Avg_size')
df['Superdomain']=df.AB.replace({'A.avg_size':'A','B.avg_size':'B'})
df.Neu=df.Neu.replace({True:'Neuron',False:'Nonneuron'})
plt.figure(figsize=(3,4))
ax=sns.boxplot(df,x='Superdomain',y='Avg_size',hue='Neu',palette=dict(Neuron='orangered',Nonneuron='dodgerblue'))
ax.text(0.5,0.1,"pvalue\nA: 0.052\nB: 0.027",transform=ax.transAxes,fontsize=12)
plt.savefig(os.path.join(figdir,'Superdomain.avg_size.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
df=pd.melt(avg_size.reset_index(),id_vars=['CellType','Neighborhood','Neu'],value_vars=['#A','#B'],var_name='AB',value_name='Number')
df['Superdomain']=df.AB.replace({'#A':'A','#B':'B'})
df.Neu=df.Neu.replace({True:'Neuron',False:'Nonneuron'})
plt.figure(figsize=(3,4))
ax=sns.boxplot(df,x='Superdomain',y='Number',hue='Neu',palette=dict(Neuron='orangered',Nonneuron='dodgerblue'),legend=False)
ax.text(0.5,0.95,"pvalue\nA: 0.052\nB: 0.027",transform=ax.transAxes,fontsize=12,ha='left',va='top')
plt.savefig(os.path.join(figdir,'Superdomain.Number.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

## Correlation between Histone marker & compartment scores

In [ ]:
from pym3c.adata import merge_adata_regions
from scipy import stats
level="Subclass"
res=100000
R=[]
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
colors=D[level].Hex.to_dict()
group='merged'
group_name=group.replace(' ','_')
diff_comp_dir=os.path.expanduser(f"~/Projects/BG/HIC/BG_{level}/compartment/{level}DiffComp")
infile=os.path.join(diff_comp_dir,
                f'{group_name}/DifferentialResult/{group_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
df_comp = pd.read_csv(infile, sep='\t')
df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') for col in df_comp.columns.tolist()]
df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
usecols=[col for col in df_comp.columns.tolist() if col in colors]
df_comp=df_comp.loc[:,['chr','start','end']+usecols]
df_comp.set_index(['chr','start','end'],inplace=True)
# df_comp

R=[]
for marker in ['H3K27ac', 'H3K9me3', 'H3K27me3']:
    peak=merge_adata_regions(
    		os.path.expanduser(f"~/Projects/BG/PairedTag/{marker}.Pseudobulk.{level}.5kb.h5ad"),bin_size=5000,use_raw=True,
    		res=res,filter_chroms=True,boundary=False,
    		exclude_chroms=['chrY','chrM','chrX'])
    peak=peak.T
    peak.insert(0,'chrom',peak.index.to_series().apply(lambda x:x.split('_')[0]))
    peak.insert(1,'start',peak.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
    peak.insert(2,'end',peak.start + res)
    peak.set_index(['chrom','start','end'],inplace=True)
    
    use_rows=list(set(df_comp.index.tolist()) & set(peak.index.tolist()))
    df_comp=df_comp.loc[use_rows]
    peak=peak.loc[use_rows]
    peak_celltypes=peak.columns.tolist()
    for ct in df_comp.columns.tolist():
        if ct not in peak_celltypes:
            continue
        r,p=stats.pearsonr(df_comp[ct].values,peak[ct].values)
        R.append([marker,ct,r,p])
data=pd.DataFrame(R,columns=['HistoneMarker','CellType','Correlation','P-value'])
data.sort_values('Correlation',ascending=False,inplace=True)
data.to_csv(os.path.join(figdir,f'{level}.compartment_socres.correlation.tsv'),sep='\t',index=False)
data

In [ ]:
data=pd.read_csv(os.path.join(figdir,f'{level}.compartment_socres.correlation.tsv'),sep='\t')
metadata=pd.read_csv(os.path.expanduser("~/Projects/BG/clustering/100kb/annotations.tsv"),sep='\t',index_col=0)
subclass2neighborhood=metadata.loc[:,['Subclass','Neighborhood']].drop_duplicates().set_index('Subclass').Neighborhood.to_dict()
data['Neighborhood']=data.CellType.map(subclass2neighborhood)
data

In [ ]:
df1=data.groupby(['Neighborhood','HistoneMarker']).Correlation.mean().unstack()
df1.to_csv(os.path.join(figdir,f'{level}.compartment_socres.correlation.groupby.tsv'),sep='\t')
df1

## A/B compartments & Gene expression for Subclass

In [ ]:
# adata_path="~/Projects/BICAN/adata/HMBA_v2/Pseudobulk.Subclass.TPM.h5ad" #HMBA
adata_path="~/Projects/BICAN/adata/HMBA_v2/Pseudobulk.Subclass.h5ad" # logCPM
adata=anndata.read_h5ad(os.path.expanduser(adata_path))
data=adata.to_df().stack().reset_index()
frac=adata.to_df(layer='frac').stack().reset_index()
data.columns=['Subclass','Gene','logCPM']
frac.columns=['Subclass','Gene','frac']
data['frac']=data.loc[:,['Subclass','Gene']].apply(lambda x:tuple(x.tolist()),axis=1).map(frac.set_index(['Subclass','Gene']).frac.to_dict())
data['#Cells']=data.Subclass.map(adata.obs.cell_count.to_dict())
for col in ['chrom','beg','end','strand','gene_type']:
    data[col]=data.Gene.map(adata.var[col].to_dict())
data=data.loc[data.chrom.notna(),['chrom','beg','end','strand','gene_type','Subclass','logCPM','frac','Gene','#Cells']]
data.beg=data.beg.astype(int)
data.end=data.end.astype(int)
data['ID']=data.chrom.map(str)+':'+data.beg.map(str)+'-'+data.end.map(str)
print(data.shape,data.ID.nunique())
data

In [ ]:
data.Gene.nunique()

In [ ]:
data.loc[:,['Subclass','ID']].drop_duplicates().shape

In [ ]:
df_comp=pd.read_csv(os.path.join(figdir,f'Subclass.superdomain.tsv'),sep='\t')
df_comp.AB=df_comp.AB.fillna('NA')
df_comp

In [ ]:
import pybedtools
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))
bed1 = pybedtools.BedTool.from_dataframe(data)
bed2 = pybedtools.BedTool.from_dataframe(df_comp)
intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
df=intersection.to_dataframe(names=data.columns.tolist()+['chr2','start2','end2','CellType','AB','score'])
df=df.loc[df.Subclass==df.CellType]
df.to_csv(os.path.join(figdir,f'Subclass.superdomain.gene_expression.tsv'),sep='\t',index=False)
df

In [ ]:
# define neuronal genes and non-neuronal genes
df_deg=pd.read_csv(os.path.expanduser("~/Projects/BICAN/adata/HMBA_v2/merged_deg.one_vs_rest.tsv"),sep='\t')
neuronal_genes=df_deg.loc[(df_deg.Group=='Neighborhood') & (df_deg.groupA=='Nonneuron') & (df_deg.FC <= 0.5) & (df_deg.auc_score >= 0.8)].Gene.unique().tolist()
nonneuronal_genes=df_deg.loc[(df_deg.Group=='Neighborhood') & (df_deg.groupA=='Nonneuron') & (df_deg.FC >= 2) & (df_deg.auc_score >= 0.8)].Gene.unique().tolist()
print(len(neuronal_genes),len(nonneuronal_genes))

In [ ]:
promoter_distance=1500
df_input=pd.read_csv(os.path.join(figdir,'Subclass.superdomain.gene_expression.tsv'),sep='\t')
df_input.AB=df_input.AB.fillna('NA')
df_input.drop(['ID','chr2','CellType'],axis=1,inplace=True)
df_input=df_input.drop_duplicates()
df_input['Promoter']=df_input.apply(lambda x:1 if x.strand=='+' and x.start-x.start2 >= -1*promoter_distance else 1 if x.strand=='-' and x.end - x.end2 <= promoter_distance else 0,axis=1)
df_input.sort_values(['Subclass','Gene','Promoter'],ascending=[True,True,False],inplace=True)
# df_input
df_input=df_input.drop(['start2','end2','Promoter'],axis=1).groupby(['chrom','start','end','strand','Subclass','logCPM','frac','Gene','#Cells'],
                          as_index=False).agg({'AB':lambda x:'-'.join(x.unique().tolist()),'score':np.mean})
# AB means promoter in A, BA means promoter in B
df_input['length']=df_input.end.astype(int) - df_input.start.astype(int)
df_input['detected']=df_input.logCPM.apply(lambda x:True if x >= 1 else False)
df_input['NeuronalGene']=df_input.Gene.apply(lambda x:True if x in neuronal_genes else False)
metadata=pd.read_csv(os.path.expanduser("~/Projects/BG/clustering/100kb/annotations.tsv"),sep='\t',index_col=0)
subclass2neighborhood=metadata.loc[:,['Subclass','Neighborhood']].drop_duplicates().set_index('Subclass').Neighborhood.to_dict()
df_input.to_csv(os.path.join(figdir,f'Subclass.superdomain.gene_expression.grouped.tsv'),sep='\t',index=False)
df_input

In [ ]:
df_input=pd.read_csv(os.path.join(figdir,f'Subclass.superdomain.gene_expression.grouped.tsv'),sep='\t')
df_input.AB=df_input.AB.fillna('NA')
df_input.AB=df_input.AB.replace({
                                'B-NA-A':'Other','NA-A':'Other','NA-B':'Other','NA-B-A':'Other','NA-A-B':'Other',
                                'B-NA':'Other','A-NA':'Other',
                                'A-B-NA':'Other','B-A-NA':'Other','A-NA-B':'Other'})
df_input.AB.unique()

In [ ]:
# # plot distribution plots for selected cell types
# for celltype in ['STR D1 MSN','STR D2 MSN','STR Hybrid MSN']:
#     df=df_input.loc[(df_input.Subclass==celltype)]
#     break


In [ ]:
df=df_input.loc[df_input.length>0,['Gene','length','NeuronalGene']].drop_duplicates()
plt.figure(figsize=(2,3))
ax=sns.violinplot(df_input, y="length", x='NeuronalGene',order=[True,False],
            palette={'True':'crimson','False':'dodgerblue'},
            # common_norm=False,
           )
ax.set_ylim([0,1e6])
plt.savefig(os.path.join(figdir,'GeneLengthForNeuronalGene.violinplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)

In [ ]:
df_input['Neighborhood']=df_input.Subclass.map(subclass2neighborhood)
df_input['Neu']=df_input.Neighborhood != 'Nonneuron'
df_input.Neu=df_input.Neu.replace({True:'Neuron',False:'Nonneuron'})
df_input.head()

In [ ]:
import scipy.stats as stats
# Test whether neuronal genes are enriched in A compartments
df1=df_input.groupby(['Neu','NeuronalGene'])['AB'].value_counts().unstack().reset_index()
R=[]
# mannwhitneyu; wilcoxon
for neighborhood in ['Neuron','Nonneuron']:
    df2=df1.loc[(df1.Neu==neighborhood)]
    groups=df2.columns.tolist()[2:]
    N=df2.loc[:,groups].sum(axis=1).sum() # total number of genes in different cell types
    for comp in groups:
        n=df2[comp].sum() # total number of genes in A/B/AB compartments
        df3=df2.loc[df2.NeuronalGene==True] # neuronal genes
        G=df3.loc[:,groups].sum(axis=1).sum() # total number of neuronal genes
        x=df3[comp].sum() # total number of neuronal genes in A/B/AB compartments
        p_value=hypergeometric(x, N, n, G, alternative='greater')
        R.append([neighborhood,comp,p_value])
    
df=pd.DataFrame(R,columns=['Neu','AB','pvalue'])
df.sort_values('pvalue',inplace=True)
df
# hypergeometric(x, N, n, G, alternative='greater') #一共N个动物，其中有n只狗，从中取出G个，有x只狗的p值

In [ ]:
df['-log10(pvalue)']=df.pvalue.apply(lambda x:-np.log10(x)).apply(lambda x:0 if x <=0 else x)
plot_data=df.pivot(index='Neu',columns='AB',values='-log10(pvalue)')
plt.figure(figsize=(4,2))
ClusterMapPlotter(data=plot_data,cmap='Reds',vmin=-np.log10(0.05),vmax=200,
                  annot=True,show_colnames=True,show_rownames=True,row_cluster=True,col_cluster=True,
                  col_names_side='top',row_names_side='left',xticklabels_kws=dict(labelrotation=45),label="-log10(pvalue)")

In [ ]:
sns.displot(df_input, x="logCPM", kind="kde",#hue='Subclass',#hue_order=['A','B','A-B','B-A','NA'],
         # 'A-NA':'salmon', 'A-B':'orangered', 'Other':'gray',  'B-NA':'lightskyblue'
            common_norm=False,
           )

In [ ]:
# sns.displot(df_input.loc[df_input.logCPM > 1], x="logCPM", kind="kde",hue='AB',hue_order=['A','B','A-B','B-A','NA'],
#             palette={'A':'crimson','B':'dodgerblue','A-B':'gold','NA':'darkorange','B-A':'lightseagreen'},
#          # 'A-NA':'salmon', 'A-B':'orangered', 'Other':'gray',  'B-NA':'lightskyblue'
#             common_norm=False,
#            )

In [ ]:
# sns.displot(df_input.loc[df_input.logCPM > 1], x="length", kind="kde",hue='AB',hue_order=['A','B','A-B','B-A','NA'],
#             palette={'A':'crimson','B':'dodgerblue','A-B':'gold','NA':'darkorange','B-A':'lightseagreen'},
#          # 'A-NA':'salmon', 'A-B':'orangered', 'Other':'gray',  'B-NA':'lightskyblue'
#             common_norm=False,log_scale=2,
#            )

In [ ]:
# sns.displot(df.loc[df.logCPM > 1], x="score", y="logCPM", hue="AB", kind="kde",rug=False,
#             hue_order=['A','B','A-B','B-A','NA'],
#             palette={'A':'crimson','B':'dodgerblue','A-B':'gold','NA':'darkorange','B-A':'lightseagreen'},
#            fill=False)

In [ ]:
# sns.displot(df.loc[df.logCPM > 1], x="score", y="length", hue="AB", kind="kde",rug=True,
#             hue_order=['A','B','A-B','B-A','NA'],
#             palette={'A':'crimson','B':'dodgerblue','A-B':'gold','NA':'darkorange','B-A':'lightseagreen'},
#            fill=False,log_scale=(None,2))

In [ ]:
df_input.loc[df_input.AB=='NA'].Gene.value_counts()

In [ ]:
catetories=df_input.AB.unique().tolist()
catetories

In [ ]:
# Number of genes detected or not detected
plot_data=df_input.groupby(['Subclass','AB']).detected.value_counts().reset_index()
plot_data['Neighborhood']=plot_data.Subclass.map(subclass2neighborhood)
plot_data['Neu']=plot_data.Neighborhood != 'Nonneuron'
plot_data.Neu=plot_data.Neu.replace({True:'Neuron',False:'Nonneuron'})
plot_data.head()

In [ ]:
import scipy.stats as stats
# Number of detected and not detected genes
df1=plot_data.pivot(index=['Neu','Subclass','detected'],columns='AB',values='count').reset_index()
R=[]
# mannwhitneyu; wilcoxon
for neighborhood in ['Neuron','Nonneuron']:
    df2=df1.loc[(df1.Neu==neighborhood)]
    existed_groups=df2.columns.tolist()
    groups=[c for c in catetories if c in existed_groups]
    N=df2.loc[:,groups].sum(axis=1).sum() # total number of genes
    for comp in groups:
        n=df2[comp].sum() # total number of genes in A/B/AB compartments
        for detected in [True,False]:
            df3=df2.loc[df2.detected==detected]
            G=df3.loc[:,groups].sum(axis=1).sum() # total number of genes that are detected (or not detected)
            x=df3[comp].sum() # total number of detected (or not detected) genes in A/B/AB compartments
            p_value=hypergeometric(x, N, n, G, alternative='greater')
            R.append([neighborhood,comp,detected,p_value])
    
df=pd.DataFrame(R,columns=['Neu','AB','Detected','pvalue'])
df.sort_values('pvalue',inplace=True)
pvalue_dict=df.set_index(['Neu','AB','Detected']).pvalue.to_dict()
print(pvalue_dict)
df

# hypergeometric(x, N, n, G, alternative='greater') #一共N个动物，其中有n只狗，从中取出G个，有x只狗的p值

In [ ]:
plot_data['log2(count)']=plot_data['count'].apply(lambda x:np.log2(x))
palette={'A':'crimson','B':'dodgerblue','AB':'gold','NA':'darkorange',
         'A-NA':'salmon', 'A-B':'orangered', 'Other':'gray', 'B-A':'lightseagreen', 'B-NA':'lightskyblue'}
for neighborhood in ['Neuron','Nonneuron']:
    plot_data1=plot_data.loc[plot_data.Neu==neighborhood]
    groups=[c for c in catetories if c in plot_data1.AB.tolist()]
    plt.figure(figsize=(2.8,3.5))
    order=[True,False]
    # ax=sns.boxplot(plot_data.loc[plot_data.Neu=='Neuron'],x='detected',y='log2(count)',hue='AB',
    #                palette=dict(A='orangered',B='dodgerblue',AB='gold'),legend=True,
    #                order=order,hue_order=['A','B','AB'])
    ax=sns.swarmplot(plot_data1,x='detected',y='log2(count)',hue='AB',
                   palette=palette,s=5,alpha=0.8,
                     legend=True,
                   order=order,hue_order=groups,edgecolor='black',linewidth=0.1)
    # p1=df.loc[df.Var=='Neuron','pvalue'].iloc[0]
    # p2=df.loc[df.Var=='Nonneuron','pvalue'].iloc[0]
    # ax.text(0.5,0.6,f"pvalue\nNeu: {p1:.2e}\nNonN: {p2:.2e}",transform=ax.transAxes,fontsize=12,ha='center')
    ax.xaxis.label.set_visible(False)
    ax.set_ylabel("No. of genes (log2 count)",fontsize=14)
    ax.set_xticklabels(['Expressed\n(logCPM > 1)','Not Expressed\n(logCPM <= 1)'],fontsize=12)
    ax.set_title(neighborhood,fontsize=14)
    ax.legend(bbox_to_anchor=(1.1, 0.5), loc='center left', borderaxespad=0,markerscale=2,frameon=True,facecolor='lightgrey')
    plt.savefig(os.path.join(figdir,f'{neighborhood}.NumberOfGenesDetectedInCompartments.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

In [ ]:
# Average length of genes detected or not detected
plot_data=df_input.groupby(['Subclass','AB','detected'],as_index=False).agg({'length':'mean','logCPM':'mean'})
plot_data['Neighborhood']=plot_data.Subclass.map(subclass2neighborhood)
plot_data['Neu']=plot_data.Neighborhood != 'Nonneuron'
plot_data.Neu=plot_data.Neu.replace({True:'Neuron',False:'Nonneuron'})
plot_data['avg.length (Kb)']=plot_data['length'] / 1000
plot_data.drop('length',axis=1,inplace=True)
plot_data.rename(columns={'logCPM':'avg.logCPM'},inplace=True)
plot_data.head()

In [ ]:
import scipy.stats as stats
# avg.length
df1=plot_data.pivot(index=['Neu','Subclass','detected'],columns='AB',values='avg.length (Kb)').reset_index()
R=[]
for detected in [True,False]:
    for neighborhood in ['Neuron','Nonneuron']:
        df2=df1.loc[(df1.Neu==neighborhood) & (df1.detected==detected)]
        groups=[c for c in catetories if c in df2.columns.tolist()]
        values=[df2[comp].values for comp in groups]
        statistic, p_value=stats.kruskal(*tuple(values))
        R.append([detected,neighborhood,statistic,p_value])
    
df=pd.DataFrame(R,columns=['Detected','Neu','statistic','pvalue'])
pvalue_dict=df.set_index(['Neu','Detected']).pvalue.to_dict()
print(pvalue_dict)
df

In [ ]:
# hue_order=['A','B','AB']
hue_order=catetories
for detected in [True,False]:
    plt.figure(figsize=(2.8,3.5))
    order=['Neuron','Nonneuron']
    plot_data1=plot_data.loc[plot_data.detected==detected]
    # ax=sns.boxplot(plot_data.loc[plot_data.Neu=='Neuron'],x='detected',y='log2(count)',hue='AB',
    #                palette=dict(A='orangered',B='dodgerblue',AB='gold'),legend=True,
    #                order=order,hue_order=['A','B','AB'])
    ax=sns.swarmplot(plot_data1,x='Neu',y='avg.length (Kb)',hue='AB',
                   palette=palette,
                     legend=True,
                   order=order,hue_order=hue_order,edgecolor='black',linewidth=0.1)
    p1=pvalue_dict[('Neuron',detected)]
    p2=pvalue_dict[('Nonneuron',detected)]
    # ax.text(0.5,0.6,f"pvalue\nNeu: {p1:.2e}\nNonN: {p2:.2e}",transform=ax.transAxes,fontsize=12,ha='center')
    ax.xaxis.label.set_visible(False)
    ax.set_ylabel("Mean gene length (Kb)",fontsize=14)
    xticklabels=[f"Neuron\n(p={p1:.2e})",f"Nonneuron\n(p={p2:.2e})"]
    ax.set_xticklabels(xticklabels,fontsize=12)
    title='Expressed Genes (logCPM > 1)' if detected else "Not Expressed Genes"
    ax.set_title(title,fontsize=14)
    ax.legend(bbox_to_anchor=(1.1, 0.5), loc='center left', borderaxespad=0,markerscale=2,frameon=True,facecolor='lightgrey')
    prefix='ExpressedGenes' if detected else "NotExpressedGenes"
    plt.savefig(os.path.join(figdir,f'{prefix}.MeanGeneLength.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

In [ ]:
plot_data.head()

In [ ]:
import scipy.stats as stats
# avg.logCPM
df1=plot_data.loc[plot_data.detected].pivot(index=['Neu','Subclass'],columns='AB',values='avg.logCPM').reset_index()
# df1=plot_data.pivot(index=['Neu','Subclass'],columns='AB',values='avg.logCPM').reset_index()
R=[]
for neighborhood in ['Neuron','Nonneuron']:
    df2=df1.loc[df1.Neu==neighborhood]
    groups=[c for c in catetories if c in df2.columns.tolist()]
    values=[df2[comp].values for comp in groups]
    statistic, p_value=stats.kruskal(*tuple(values))
    R.append([neighborhood,statistic,p_value])
    
df=pd.DataFrame(R,columns=['Neu','statistic','pvalue'])
pvalue_dict=df.set_index(['Neu']).pvalue.to_dict()
print(pvalue_dict)
df

In [ ]:
# hue_order=['A','B','AB']
hue_order=catetories
plt.figure(figsize=(2.8,3.5))
order=['Neuron','Nonneuron']
ax=sns.swarmplot(plot_data.loc[plot_data.detected==True],x='Neu',y='avg.logCPM',hue='AB',
               palette=palette,
                 legend=True,
               order=order,hue_order=hue_order,edgecolor='black',linewidth=0.1)
# ax=sns.swarmplot(plot_data,x='Neu',y='avg.logCPM',hue='AB',
#                palette=palette,
#                  legend=True,
#                order=order,hue_order=hue_order,edgecolor='black',linewidth=0.1)
p1=pvalue_dict['Neuron']
p2=pvalue_dict['Nonneuron']
# ax.text(0.5,0.6,f"pvalue\nNeu: {p1:.2e}\nNonN: {p2:.2e}",transform=ax.transAxes,fontsize=12,ha='center')
ax.xaxis.label.set_visible(False)
ax.set_ylabel("Mean expression (logCPM)",fontsize=14)
xticklabels=[f"Neuron\n(p={p1:.2e})",f"Nonneuron\n(p={p2:.2e})"]
ax.set_xticklabels(xticklabels,fontsize=12)
# ax.set_title("All Expressed Genes",fontsize=14)
ax.legend(bbox_to_anchor=(1.1, 0.5), loc='center left', borderaxespad=0.1,markerscale=2,frameon=True,facecolor='lightgrey')
plt.savefig(os.path.join(figdir,'MeanGenelogCPM.boxplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
pybedtools.cleanup()

## Enrichment analysis for the A/B compartments genes in each cell type

```R
# Enrichment analysis in R
# mkdir -p ABGenesEnrichment
suppressMessages(suppressWarnings(source("~/Scripts/R/RunClusterProfiler.R",echo = FALSE, verbose = FALSE)))
data <- read.csv("Subclass.superdomain.gene_expression.grouped.tsv",sep="\t")
data$AB[data$AB==""] <- "NA"
data$AB[is.na(data$AB)] <- "NA"
OrgDb="org.Hs.eg.db"
organism="hsa"
# Select all expressed genes?
data <- data[data$logTPM > 0,]

outdir="ExpressedABGenesEnrichment"
if(! dir.exists(outdir)){
    dir.create(outdir)
}
categories=c('A','B','A-B','B-A','NA')
for(cell_type in unique(data$Subclass)){
    print(cell_type)
    df <- data[data$Subclass==cell_type,]
    if (dim(df)[1] < 3) {next}
    #for(category in unique(df$AB)){
    for(category in categories){
        print(category)
        df2 <- df[df$AB==category,]
        gene.df <- bitr(unique(df2$Gene),fromType="SYMBOL",toType="ENTREZID",
                        OrgDb = OrgDb)
        if(length(unique(gene.df$ENTREZID))<3){next}
        RunGoEnrich(
            genes=unique(gene.df$ENTREZID),OrgDb = OrgDb,
            readable=F,
            outname=paste(outdir,paste(category,cell_type,'BP',sep='_'),sep='/'),
            ont = 'BP',showCategory=15,title=cell_type
        )
        RunGoEnrich(genes=unique(gene.df$ENTREZID),OrgDb = OrgDb,
                    readable=F,
                    outname=paste(outdir,paste(category,cell_type,'MF',sep='_'),sep='/'),
                    ont = 'MF',showCategory=15,
                    title=cell_type)
        RunKeggEnrich(genes=unique(gene.df$ENTREZID),organism = organism,OrgDb = OrgDb,
                      outname=paste(outdir,paste(category,cell_type,'KEGG',sep='_'),sep='/'),
                      showCategory=15,
                      title=cell_type)
    }
}


# all cell types
categories=c('A','B','A-B','B-A','NA','A-NA','B-NA','NA-A','NA-B')
for(category in categories){
    print(category)
    df2 <- data[data$AB==category,]
    gene.df <- bitr(unique(df2$Gene),fromType="SYMBOL",toType="ENTREZID",
                    OrgDb = OrgDb)
    if(length(unique(gene.df$ENTREZID))<3){next}
    RunGoEnrich(
        genes=unique(gene.df$ENTREZID),OrgDb = OrgDb,
        readable=F,
        outname=paste(outdir,paste(category,'BP',sep='_'),sep='/'),
        ont = 'BP',showCategory=15,title=category
    )
    RunGoEnrich(genes=unique(gene.df$ENTREZID),OrgDb = OrgDb,
                readable=F,
                outname=paste(outdir,paste(category,'MF',sep='_'),sep='/'),
                ont = 'MF',showCategory=15,
                title=category)
    RunKeggEnrich(genes=unique(gene.df$ENTREZID),organism = organism,OrgDb = OrgDb,
                  outname=paste(outdir,paste(category,'KEGG',sep='_'),sep='/'),
                  showCategory=15,
                  title=category)
}
```

## Compartment Switch

In [ ]:
df_input=pd.read_csv(os.path.join(figdir,f'Subclass.superdomain.gene_expression.grouped.tsv'),sep='\t')
df_input.AB=df_input.AB.fillna('NA')
df_input.drop(['chrom','start','end','strand','#Cells','length','detected'],axis=1,inplace=True)
df_input.sort_values(['Gene','AB'],inplace=True)
df_input

In [ ]:
df_input.AB=df_input.AB.apply(lambda x:x if x in ['A','B','A-B','B-A','NA'] else 'Other')
print(df_input.AB.unique())
df_input

In [ ]:
plt.figure(figsize=(3.5,3.5))
g=sns.displot(df_input, x="logCPM", kind="kde",common_norm=False,linewidth=2)
ax=g.axes[0][0]
ax.vlines(x=1,ymin=0,ymax=1,linestyle='--',color='red')
plt.savefig(os.path.join(figdir,'logCPM.distribution.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
df1=df_input.loc[df_input.logCPM > 1].groupby('Subclass')['AB'].value_counts(normalize=True).unstack()
df1.sort_values('A',inplace=True)
df1

In [ ]:
plt.figure()
gap=0.05
linewidth=0.1
palette={'A':'crimson','B':'dodgerblue','NA':'darkorange','A-B':'orangered','Other':'gray','B-A':'lightseagreen', 
         # 'A-NA':'salmon',   'B-NA':'lightskyblue'
        }
ax=df1.plot.bar(stacked=True,align='edge', width=1-gap,edgecolor='black',
               linewidth=linewidth,figsize=(8,3.5),
               color=palette)
ax.set_xlim(0,df1.shape[0])
ax.set_ylim(0,1)
labels=[tick.get_text() for tick in ax.get_xticklabels()]
ax.set_xticks(ticks=np.arange(0.5,df1.shape[0],1),fontsize=10,
              labels=labels,rotation=-45,rotation_mode='anchor',horizontalalignment='left',verticalalignment='center')
ax.xaxis.label.set_visible(False)
ax.tick_params(
    axis="y", #both
    which="both",left=False,right=False,labelleft=False,labelright=False,
    top=False,labeltop=False,#bottom=False,labelbottom=False
    )
ax.legend(labelspacing=0.1)
ax.set_title("Chromatin compartment composition of the expressed genes across cell types",fontsize=12)
for tick in ax.get_xticklabels():
    tick.set_color(group_colors.get(tick.get_text(),'black'))
plt.savefig(os.path.join(figdir,'CellType.AB.composition.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
metadata=pd.read_csv(os.path.expanduser("~/Projects/BG/clustering/100kb/annotations.tsv"),sep='\t',index_col=0)
subclass2neighborhood=metadata.loc[:,['Subclass','Neighborhood']].drop_duplicates().set_index('Subclass').Neighborhood.to_dict()
df_input['Neighborhood']=df_input.Subclass.map(subclass2neighborhood)
df_input.head()

In [ ]:
# Parse gene biotype from GENCODE GTF and keep only protein_coding genes
import re
gtf_path = os.path.expanduser("~/Projects/Ref/hg38/annotations/gencode.v43.annotation.gtf")
_gn_re = re.compile(r'gene_name "([^"]+)"')
_gt_re = re.compile(r'gene_type "([^"]+)"')
records = []
with open(gtf_path) as fh:
    for line in fh:
        if line.startswith('#'):
            continue
        fields = line.rstrip('\n').split('\t')
        if len(fields) < 9 or fields[2] != 'gene':
            continue
        attrs = fields[8]
        m_name = _gn_re.search(attrs)
        m_type = _gt_re.search(attrs)
        if m_name and m_type:
            records.append((m_name.group(1), m_type.group(1)))
gene2type = dict(records)   # dedup by last-seen; gene_name unique in gencode 'gene' rows
print(f"Parsed {len(gene2type)} genes; protein_coding = {sum(v=='protein_coding' for v in gene2type.values())}")

df_input['gene_type'] = df_input.Gene.map(gene2type)
_before = df_input.Gene.nunique()
df_input = df_input.loc[df_input.gene_type == 'protein_coding'].copy()
print(f"df_input genes: {_before} -> {df_input.Gene.nunique()} (protein_coding only); rows: {df_input.shape[0]}")
df_input.head()

In [ ]:
compare_in="Neuron"
# compare_in="Nonneuron"
# compare_in="All"
if compare_in=="Neuron":
    _mask = df_input.Neighborhood != 'Nonneuron'
elif compare_in=="Nonneuron":
    _mask = df_input.Neighborhood == 'Nonneuron'
elif compare_in=="All":
    _mask = pd.Series(True, index=df_input.index)
else:
    raise ValueError(f"unknown compare_in={compare_in!r}")
cell_types = df_input.loc[_mask].Subclass.unique().tolist()
comp_switch = df_input.loc[_mask].groupby(['Gene','AB']).agg({'Subclass':lambda x:x.unique().tolist()}).unstack(level=1)
comp_switch.columns=comp_switch.columns.droplevel(0)
comp_switch.insert(0,'n_switch',comp_switch.notna().sum(axis=1).tolist())

comp_switch.sort_values('n_switch',ascending=False,inplace=True)
comp_switch

In [ ]:
adata_path="~/Projects/BICAN/adata/HMBA_v2/Pseudobulk.Subclass.h5ad" #HMBA
adata=anndata.read_h5ad(os.path.expanduser(adata_path))
data=adata.to_df().T.loc[:,cell_types]
data

In [ ]:
# test whether gene show differential expression in different compartment groups
from scipy import stats
def kruskal_test(x,data):
    # gene='PLD5'
    gene=x.name
    D=x.dropna().to_dict()
    values=[data.loc[gene,D[k]].values for k in D]
    try:
        statistic, p_value=stats.kruskal(*tuple(values))
    except:
        statistic=np.nan
        p_value=1
    mean_logCPM={k:np.mean(v) for k,v in zip(D.keys(),values)}
    return mean_logCPM,statistic,p_value
    
groups=comp_switch.columns.tolist()[1:]
comp_switch['kruskal_test']=comp_switch.loc[:,groups].apply(lambda x:kruskal_test(x,data),axis=1)
comp_switch.insert(1,'mean_logCPM',comp_switch.kruskal_test.apply(lambda x:x[0]).tolist())
comp_switch.insert(2,'kruskal_statistic',comp_switch.kruskal_test.apply(lambda x:x[1]).tolist())
comp_switch.insert(3,'kruskal_pvalue',comp_switch.kruskal_test.apply(lambda x:x[2]).tolist())
comp_switch         

In [ ]:
comp_switch.drop('kruskal_test',axis=1,inplace=True)
def cal_fold_change(D):
    values=[D[k] for k in D]
    vmin=max(min(values),0.001)
    return max(values) / vmin
def cal_delta(D):
    values=[D[k] for k in D]
    vmin=max(min(values),0.001)
    return max(values) - vmin
comp_switch.insert(2,'FC',comp_switch.mean_logCPM.apply(cal_fold_change))
comp_switch.insert(3,'delta',comp_switch.mean_logCPM.apply(cal_delta))
comp_switch.head()

In [ ]:
# add Pearson correlation between gene expression and compartment score
df_exp=df_input.pivot(index='Gene',columns='Subclass',values='logCPM')
df_score=df_input.pivot(index='Gene',columns='Subclass',values='score')
D1={}
D2={}
print(cell_types)
for gene in df_exp.index.tolist():
    r,pvalue=stats.pearsonr(x=df_exp.loc[gene,cell_types].values,y=df_score.loc[gene,cell_types].values)
    D1[gene]=r
    D2[gene]=pvalue
comp_switch['pearson_r']=comp_switch.index.to_series().map(D1)
comp_switch['pearson_p']=comp_switch.index.to_series().map(D2)
comp_switch.sort_values(['FC','kruskal_pvalue','kruskal_statistic'],ascending=[False,True,False],inplace=True)
comp_switch.head()

In [ ]:
comp_switch.to_csv(os.path.join(figdir,f'Subclass.compartment_switch_in_{compare_in}.tsv'),sep='\t')

In [ ]:
# visualize genes with compartment switch
compare_in="Neuron"
# compare_in="Nonneuron"
# compare_in="All"
r_cutoff=0.7 #if compare_in=="Neuron" else 0.5
fc_cutoff=1.5 if compare_in=="Neuron" else 2
comp_switch=pd.read_csv(os.path.join(figdir,f'Subclass.compartment_switch_in_{compare_in}.tsv'),sep='\t')
comp_switch.mean_logCPM=comp_switch.mean_logCPM.apply(eval)
comp_switch['vmax']=comp_switch.mean_logCPM.apply(lambda x:np.max([x[k] for k in x]))
comp_switch['vmin']=comp_switch.mean_logCPM.apply(lambda x:np.min([x[k] for k in x]))
for c in ['A','B','A-B','B-A','NA']: #
    if c not in comp_switch.columns.tolist():
        continue
    comp_switch[c]=comp_switch[c].apply(lambda x:eval(x) if not pd.isna(x) else [])
if compare_in=="Neuron":
    comp_switch=comp_switch.loc[ (comp_switch.pearson_p <= 0.05
                      ) & (comp_switch.pearson_r.abs() >= r_cutoff) & (comp_switch.vmax > 1) & (comp_switch.FC >= fc_cutoff)] #  & (comp_switch.vmin < 0.5) & (comp_switch.n_switch >= 2) &
else:
    comp_switch=comp_switch.loc[ (comp_switch.pearson_p <= 0.05
                      ) & (comp_switch.pearson_r.abs() >= r_cutoff) & (comp_switch.vmax > 1) & (comp_switch.FC >= fc_cutoff) & (comp_switch.vmin < 0.5) & (comp_switch.n_switch == 2)] #   &
# if compare_in == 'Nonneuron':
#     a=comp_switch.loc[comp_switch.mean_logCPM.apply(lambda x:x.get('A',0) > 1 and x.get('B',1) < x.get('A',1))].Gene.unique().tolist()
#     b=comp_switch.loc[comp_switch.mean_logCPM.apply(lambda x:x.get('B',0) > 1 and x.get('B',1) > x.get('A',1))].Gene.unique().tolist()
#     comp_switch['ExpressedIn']=''
#     comp_switch.loc[comp_switch.Gene.isin(a),'ExpressedIn']='A'
#     comp_switch.loc[comp_switch.Gene.isin(b),'ExpressedIn']='B'
#     comp_switch['No.A']=comp_switch['A'].apply(lambda x:len(x))
#     comp_switch['No.B']=comp_switch['B'].apply(lambda x:len(x))
#     comp_switch=comp_switch.loc[((comp_switch['No.A']==1) & (comp_switch.ExpressedIn=='A')) | ((comp_switch['No.B']==1) & (comp_switch.ExpressedIn=='B'))]
#     comp_switch=comp_switch.loc[((comp_switch.ExpressedIn=='A') & (comp_switch['No.A'] < comp_switch['No.B'])) | ((comp_switch.ExpressedIn=='B') & (comp_switch['No.B'] < comp_switch['No.A']))]
#     comp_switch=comp_switch.loc[(comp_switch.ExpressedIn=='B') & (comp_switch['No.B'] < comp_switch['No.A'])]

# comp_switch=comp_switch.loc[comp_switch.ExpressedIn=='B']comp_switch

comp_switch.sort_values('A',ascending=True,inplace=True)
print(comp_switch.shape)

In [ ]:
df_rows=comp_switch #.loc[comp_switch.FC >= 2]
# df_rows=df_rows.loc[(df_rows.A.apply(lambda x:len(x) > 0)) & (df_rows.B.apply(lambda x:len(x) > 0))]
df_rows.set_index('Gene',inplace=True)
df_rows['Direction']=df_rows.pearson_r.apply(lambda x:'pos' if x >0 else 'neg')
print(df_rows.shape)
df_rows

In [ ]:
print(df_rows.index.tolist())

In [ ]:
metadata = pd.read_csv(os.path.expanduser("~/Projects/BG/clustering/100kb/annotations.tsv"),sep='\t',index_col=0)
print(metadata.shape)
keep_cells=metadata.loc[~ (metadata['Group'].isin(['doublet','Glut','Unk','Low quality cells']))].index.tolist() #,
metadata=metadata.loc[keep_cells]
df_cols = metadata.loc[:,['Subclass','Class','Neighborhood']].drop_duplicates().set_index('Subclass')
sort_cols=df_cols.columns.tolist()[::-1]
df_cols.sort_values(sort_cols,inplace=True)
keep_rows=[row for row in df_cols.index.tolist() if row in df_input.Subclass.unique().tolist()]
df_cols=df_cols.loc[keep_rows]
# Restrict columns to the Neighborhood that compare_in selected the genes from,
# otherwise the chosen genes (expressed in `compare_in`) leave a big blank band
# in the other Neighborhoods. compare_in == "All" keeps everything.
if compare_in == "Nonneuron":
    df_cols = df_cols.loc[df_cols.Neighborhood == "Nonneuron"]
elif compare_in == "Neuron":
    df_cols = df_cols.loc[df_cols.Neighborhood != "Nonneuron"]
df_cols.head()

In [ ]:
df_input.AB=df_input.AB.replace({'A-B':'A','B-A':'B'})
df_input['abs_score']=df_input.score.abs()
df_input.head()

In [ ]:
# read color palette
level='Subclass'
palette_path=os.path.expanduser("~/Projects/BG/metadata/BG_color_palette.xlsx")
color_palette = {}
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
for key in ['Neighborhood','Subclass','Group','Class','Region']:
    if key not in D:
        continue
    color_palette[key]=D[key].Hex.to_dict()
ct_colors={}
for k in df_cols.index.tolist():
    ct_colors[k]=color_palette[level][k.replace('_',' ')]
print(ct_colors)

In [ ]:
from PyComplexHeatmap import *
vmax=10 if compare_in != "Nonneuron" else 8
col_ha = HeatmapAnnotation(
                           label=anno_label(df_cols.index.to_series(), merge=True,rotation=45,
                                            fontsize=10,height=2,colors=group_colors,relpos=(0,0),
                                           arrowprops = dict(visible=False)),
                           Subclass=anno_simple(df_cols.index.to_series(),legend=False,add_text=False,
                                               height=2,colors=group_colors),
                           # Class=anno_simple(df_cols.Class,colors=color_palette['Class'],legend=False,add_text=False,
                           #                       height=3),
                           verbose=0,label_side='right',label_kws={'horizontalalignment':'left'})

row_ha = HeatmapAnnotation(
                           Correlation=anno_simple(df_rows.pearson_r,legend=True,
                                               height=2,cmap='jet'),label_side='top',#label_kws={'horizontalalignment':'left'}
                           axis=0,)

figsize={'Neuron':(6,12),'Nonneuron':(3,8),'All':(8,16)}.get(compare_in,(6,10))
plt.figure(figsize=figsize)
cm = DotClustermapPlotter(data=df_input.loc[(df_input.Gene.isin(df_rows.index.tolist())) & (df_input.Subclass.isin(df_cols.index.tolist()))], 
                          x='Subclass',y='Gene',value='logCPM',c='logCPM',s='frac',
                          y_order=df_rows.index.tolist(),
                          x_order=df_cols.index.tolist(),
                          row_cluster=False,col_cluster=False,hue='AB',
                          cmap={'A':'Reds','B':'Blues'}, # 'A-B':'Reds','B-A':'Blues', 'A-B':'red','B-A':'blue'
                          colors={'A':'red','B':'blue'},
                          top_annotation=col_ha,left_annotation=row_ha,
                          # row_split=df_rows.Owner,
                          # row_split_order=[s for s in df_cols.index.tolist() if s in df_rows.Owner.unique()],
                          marker='o',vmin=0,vmax=vmax,
                          # marker={'EnrPval':'P','DepPval':'X'},
                          show_rownames=True,show_colnames=False,#col_dendrogram=True,
                          col_names_side='top',#row_names_side='right',
                          yticklabels_kws={'labelrotation': 0, 'labelcolor': 'black','labelsize':12,'right':True,},
                          xticklabels_kws={'labelrotation': -45, 'labelcolor': 'black','labelsize':12,'bottom':True,},
                          # #left_annotation=left_ha,#right_annotation=right_ha,
                          spines=False,alpha=0.7,#max_s=200,
                          # col_split=df_cols.Neighborhood,col_split_gap=0.5,col_split_order=df_cols.Neighborhood.unique().tolist(),
                          verbose=1,legend_vgap=10,legend_width=30,legend_anchor='ax',
                          dot_legend_kws=dict(labelspacing=0.2),
                          ylabel="Compartment score-associated DEGs",#ylabel_side="bottom",
                          ylabel_kws=dict(fontsize=14,fontweight='bold'),
                          # xlabel_bbox_kws=dict(facecolor=facecolor)
                         )
for cbar in cm.cbars:
    if not hasattr(cbar,'ax'):
        continue
    label=cbar.ax.yaxis.get_label().get_text()
    if label in ['A','B','A-B','B-A']:
        cbar.ax.set_ylabel(f"logCPM in {label}")
plt.savefig(os.path.join(figdir,f'Subclass.compartment_switch_in_{compare_in}.logCPM.highly_correlated.pdf'),transparent=True,bbox_inches='tight',dpi=300)
plt.show()

## Plot ATAC (100kb peaks at Subclass level)

In [ ]:
from pym3c.adata import merge_adata_regions
res=100000
# data=anndata.read_h5ad(os.path.expanduser("~/Projects/BG/atac/Pseudobulk.Subclass.5kb.h5ad")).to_df()
data=merge_adata_regions(
		os.path.expanduser("~/Projects/BG/atac/Pseudobulk.Subclass.5kb.h5ad"),bin_size=5000,use_raw=True,
		res=res,filter_chroms=True,boundary=False,
		exclude_chroms=['chrY','chrM','chrX'])
data=data.T
# data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split(':')[0]))
# data.insert(1,'start',data.index.to_series().apply(lambda x:x.split(':')[1].split('-')[0]).astype(int))
data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split('_')[0]))
data.insert(1,'start',data.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
data

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"

celltypes=[
           'STR D1 MSN','STR D2 MSN','STR Hybrid MSN',"F M GATA3 GABA", 'M Dopa',"CN ONECUT1 GABA","CN LHX8 GABA","F Glut",'CN ST18 GABA','Astrocyte',
           'Oligodendrocyte','OPC','Microglia','Lymphocyte']

In [ ]:
print(res)
for ct in celltypes:
    df=data.loc[data[ct]>0,ct]
    print(ct,df.quantile(0.05),df.quantile(0.95))

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from pycirclize.utils import ColorCycler

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes['chr4']=chrom_sizes['chr4']

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
circos.text(f"{int(res // 1000)}kb ATAC\nlog(CPM)", size=15)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)
# Plot chromosome name
circos.text('chr4', r=97.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
    # sector.text(sector.name.replace('chr',''), size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6,#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=92
vmin=2
vmax=4.8
for ct in celltypes:
    print(ct)
    peaks=data.loc[data[ct] >= vmin,['chrom','start',ct]]
    color=group_colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.05)
        size=5 if ct=='Oligodendrocyte' else 5.5 if ct=='Lymphocyte' else 6.5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        if ct=='M Dopa':
            track.yticks([vmin,vmax], [str(vmin),str(vmax)],vmin=vmin,vmax=vmax,label_size=6)
        peak = peaks.loc[peaks['chrom']==sector.name]
        for _, row in peak.iterrows():
            end=min(row['start']+res,sector.size)
            signal=row[ct] if row[ct] < vmax else vmax
            track.fill_between(x=[row['start'],end], y1=[signal,signal], y2=vmin, vmin=vmin,vmax=vmax, color=color,rasterized=True) #alpha=0.6,rasterized=True
            # center=int((row["start"]+end)*0.5)
            # track.bar([center], height=[signal], width=res, color=color,vmin=vmin,vmax=vmax) #edgecolor, linewidth, rasterized=True
            # track.rect(row["start"], end, fc=fc, ec=fc, lw=0.08,rasterized=True)
    y=y-5.7

circos.savefig(os.path.join(figdir,f'Subclass.{int(res // 1000)}kb_peaks.ciscos_plot.pdf'), dpi=300, figsize=(6,6))
fig = circos.plotfig()
# display(circos.figure)

## Plot ATAC (100kb peaks at Class level)

In [ ]:
from pym3c.adata import merge_adata_regions
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
class_colors=D['Class'].Hex.to_dict()

res=100000
# data=anndata.read_h5ad(os.path.expanduser("~/Projects/BG/atac/Pseudobulk.Subclass.5kb.h5ad")).to_df()
data=merge_adata_regions(
		os.path.expanduser("~/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"),bin_size=5000,use_raw=True,
		res=res,filter_chroms=True,boundary=False,
		exclude_chroms=['chrY','chrM','chrX'])
data=data.T
# data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split(':')[0]))
# data.insert(1,'start',data.index.to_series().apply(lambda x:x.split(':')[1].split('-')[0]).astype(int))
data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split('_')[0]))
data.insert(1,'start',data.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
data

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
celltypes=["M Dopa","F M Glut",'CN GABA-Glut',"F M GABA", 
           'CN LGE GABA','CN MGE GABA','CN CGE GABA',
           'Astro-Epen','Vascular','OPC-Oligo','Immune']

In [ ]:
print(res)
for ct in celltypes:
    df=data.loc[data[ct]>0,ct]
    print(ct,df.quantile(0.05),df.quantile(0.95))

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from pycirclize.utils import ColorCycler

chrom="chr10"
chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
circos.text(f"{int(res // 1000)}kb ATAC\nlog(CPM)", size=15)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)
# Plot chromosome name
circos.text(chrom, r=97.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
    # sector.text(sector.name.replace('chr',''), size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6,#label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=92
vmin=2.1
vmax=4.8
for ct in celltypes:
    print(ct)
    peaks=data.loc[data[ct] >= vmin,['chrom','start',ct]]
    color=class_colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.05)
        size=6 if ct=='OPC-Oligo' else 6.5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        if ct=='M Dopa':
            track.yticks([vmin,vmax], [str(vmin),str(vmax)],vmin=vmin,vmax=vmax,label_size=6)
        peak = peaks.loc[peaks['chrom']==sector.name]
        for _, row in peak.iterrows():
            end=min(row['start']+res,sector.size)
            signal=row[ct] if row[ct] < vmax else vmax
            track.fill_between(x=[row['start'],end], y1=[signal,signal], y2=vmin, vmin=vmin,vmax=vmax, color=color,rasterized=True) #alpha=0.6,rasterized=True
            # center=int((row["start"]+end)*0.5)
            # track.bar([center], height=[signal], width=res, color=color,vmin=vmin,vmax=vmax) #edgecolor, linewidth, rasterized=True
            # track.rect(row["start"], end, fc=fc, ec=fc, lw=0.08,rasterized=True)
    y=y-5.7

circos.savefig(os.path.join(figdir,f'Class.{chrom}.{int(res // 1000)}kb_peaks.ciscos_plot.pdf'), dpi=300, figsize=(5,5))
fig = circos.plotfig()
# display(circos.figure)

## Plot DNA methylation (5kb mCG raw methylation at Subclass level)

In [ ]:
from ALLCools.mcds import MCDS
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
pseudobulk_mcds_path="~/Projects/BG/pseudobulk/Subclass/Subclass.mcds"
var_dim='5kb'
mc_type='CGN'
res=5000
mcds = MCDS.open(os.path.expanduser(pseudobulk_mcds_path), var_dim=var_dim)
mc_df = mcds[f'{var_dim}_da'].sel(count_type='mc', mc_type=mc_type).to_pandas().T
mc_df.columns = [col.replace('_', ' ') for col in mc_df.columns.tolist()]
cov_df = mcds[f'{var_dim}_da'].sel(count_type='cov', mc_type=mc_type).to_pandas().T
cov_df.columns = [col.replace('_', ' ') for col in cov_df.columns.tolist()]
# rows are bins, columns are cell_types
df_frac = calculate_posterior_mc_frac(mc_df.values, cov_df.values, normalize_per_cell=False)[0]  # bigger range with normalize_per_cell
data = pd.DataFrame(df_frac, index=mc_df.index.tolist(), columns=mc_df.columns.tolist())

data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split('_')[0]))
data.insert(1,'start',data.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
data

In [ ]:
data.iloc[:,2:].max().max()

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
celltypes=["CN LHX8 GABA","F Glut","F M GATA3 GABA", 
           'STR D1 MSN','STR D2 MSN','STR Hybrid MSN','CN ST18 GABA','Astrocyte',
           'Oligodendrocyte','OPC','Microglia','Lymphocyte']

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from matplotlib.patches import Patch,Rectangle

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes['chr4']=chrom_sizes['chr4']

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
# circos.text("5kb CpG Mean\nmethylation", size=15,r=20)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((95, 100), cytoband_file)
# Plot chromosome name
circos.text('chr4', r=97.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
    # sector.text(sector.name, size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6, #label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=92
for ct in celltypes[::-1]:
    print(ct)
    color=group_colors[ct.replace('_',' ')]
    for sector in circos.sectors:
        track = sector.add_track((y-5, y), r_pad_ratio=0.05)
        size=5 if ct=='Oligodendrocyte' else 5.5 if ct=='Lymphocyte' else 6.5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        df_cg=data.loc[(data['chrom']==sector.name),['chrom','start',ct]]
        # track.heatmap(df_cg.fillna(0.5), width=5000, vmin=0, vmax=1, cmap='Blues_r')
        for _, row in df_cg.iterrows():
            if pd.isna(row[ct]):
                fc='black'
            elif row[ct] > 0.7: # hypermethylated: white color
                continue
            elif row[ct] > 0.5:
                fc='lightskyblue'
            elif row[ct] > 0.3:
                fc='royalblue'
            else:
                fc='#3e27a9' # color
            end=min(row['start']+res,sector.size)
            track.rect(row["start"], end, fc=fc, ec=fc, lw=0.08,rasterized=True)
    y=y-5.7

handles = [
    Patch(color="black", label="NA"),
    Patch(facecolor="white", edgecolor='black',linewidth=0.1,label="0.7 - 1"),
    Patch(color="lightskyblue", label="0.5 - 0.7"),
    Patch(color="royalblue", label="0.3 - 0.5"),
    Patch(color="#3e27a9", label="0 - 0.3"),
]
fig = circos.plotfig(dpi=300, figsize=(7,7))
# circos.ax.set_rasterized(True)
_ = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.5), loc="center", fontsize=8, 
                     title="  5kb CG\nmethylation",title_fontsize=12,labelspacing=0.1)
fig.savefig(os.path.join(figdir,'Subclass.5kb_mCG.ciscos_plot.pdf'))

# from matplotlib.colors import LinearSegmentedColormap
# my_cmap = LinearSegmentedColormap.from_list(
#             "test", [(0, "#3e27a9"), (0.3,'royalblue'), (0.5,'deepskyblue'), (0.7,'lightskyblue'),  (1, 'white')]
#         )
# circos.colorbar(bounds=(0.35, 0.45, 0.3, 0.01), vmin=0, vmax=1, orientation="horizontal", cmap=my_cmap)
# circos.savefig(os.path.join(figdir,'Subclass.5kb_mCG.ciscos_plot.pdf'),dpi=300, figsize=(5,5))

# display(circos.figure)

In [ ]:
# Chi-square test: test whether the heterochromatin regions loss more methylation

In [ ]:
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
cytoband=pd.read_csv(cytoband_file,sep='\t')
cytoband=cytoband.loc[cytoband['#chrom']=='chr4']
cytoband.head()

In [ ]:
data['gieStain']=np.nan
for _,row in cytoband.iterrows():
    data.loc[(data.start >= row['chromStart']) & (data.start + 5000 <= row['chromEnd']),'gieStain']=row['gieStain']
data

In [ ]:
df=data.loc[data.chrom=='chr4',celltypes].applymap(lambda x:np.nan if pd.isna(x) else 'hyper' if x > 0.7 else 'intermedia' if x > 0.3 else 'hypo')
df['gieStain']=data.loc[df.index.tolist(),'gieStain'].tolist()
df

In [ ]:
from scipy import stats
R=[]
for ct in celltypes:
    df1=df.loc[df[ct].notna(),[ct,'gieStain']]
    vc=df1.groupby(ct)['gieStain'].value_counts().unstack()
    vc['Inactivate']=vc['gpos100'] #+vc['gpos75']
    vc['Activate']=vc['gneg']+vc['gpos25'] #+vc['gpos50']
    vc=vc.loc[:,['Inactivate','Activate']].T
    meth_types=vc.columns.tolist()
    for meth_type in meth_types:
        vc1=vc.copy()
        vc1[f'Non-{meth_type}']=vc1.apply(lambda x:x.sum() - x[meth_type],axis=1)
        test_data=vc1.loc[['Inactivate','Activate'],[meth_type,f'Non-{meth_type}']]
        print(test_data)
        r=stats.chi2_contingency(test_data.values)
        R.append([ct,meth_type,r.statistic,r.pvalue])
df_res=pd.DataFrame(R,columns=['CellType','Variable','Statistic','P-value'])
df_res.sort_values('Statistic',ascending=False,inplace=True)
df_res.to_excel("5kb_heterochromatin_loss_methylation_chi2_square.xlsx")

In [ ]:
df_res.loc[df_res.Variable=='intermedia']

## Plot DNA methylation (5kb mCG raw methylation at Class level)

In [ ]:
from ALLCools.mcds import MCDS
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
class_colors=D['Class'].Hex.to_dict()
chrom="chr10"

pseudobulk_mcds_path="~/Projects/BG/pseudobulk/Class/Class.mcds"
var_dim='5kb'
mc_type='CGN'
res=5000
mcds = MCDS.open(os.path.expanduser(pseudobulk_mcds_path), var_dim=var_dim)
mc_df = mcds[f'{var_dim}_da'].sel(count_type='mc', mc_type=mc_type).to_pandas().T
mc_df.columns = [col.replace('_', ' ') for col in mc_df.columns.tolist()]
cov_df = mcds[f'{var_dim}_da'].sel(count_type='cov', mc_type=mc_type).to_pandas().T
cov_df.columns = [col.replace('_', ' ') for col in cov_df.columns.tolist()]
# rows are bins, columns are cell_types
df_frac = calculate_posterior_mc_frac(mc_df.values, cov_df.values, normalize_per_cell=False)[0]  # bigger range with normalize_per_cell
data = pd.DataFrame(df_frac, index=mc_df.index.tolist(), columns=mc_df.columns.tolist())

data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split('_')[0]))
data.insert(1,'start',data.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
data

In [ ]:
data.iloc[:,2:].max().max()

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
celltypes=["M Dopa","F M Glut",'CN GABA-Glut',"F M GABA", 
           'CN LGE GABA','CN MGE GABA','CN CGE GABA',
           'Astro-Epen','Vascular','OPC-Oligo','Immune']

In [ ]:
df_comp=pd.read_csv(os.path.join(figdir,'Class.superdomain.tsv'),sep='\t')
# df_comp=df_comp.loc[df_comp['chr']==chrom]
df_comp

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from matplotlib.patches import Patch,Rectangle

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
# circos = Circos.initialize_from_bed(chr_bed_file, space=1,start=10, end=350)
# circos.text("5kb CpG Mean\nmethylation", size=15,r=20)
# Add cytoband tracks from cytoband file
circos.add_cytoband_tracks((97, 100), cytoband_file)
# Plot chromosome name
circos.text(chrom, r=98.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
    # sector.text(sector.name, size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6, #label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=95
for ct in celltypes[::-1]:
    print(ct)
    color=class_colors[ct.replace('_',' ')]
    gap=0
    for sector in circos.sectors:
        if ct==celltypes[-1]:
            comp=df_comp.loc[df_comp.CellType==ct,['chr','start','end','score']]
            track = sector.add_track((y-2, y), r_pad_ratio=0.05)
            track.axis(fc='white')
            comp1 = comp.loc[comp['chr']==sector.name]
            for _, row in comp1.iterrows():
                signal=row['score']
                if signal > 0:
                    color1='red'
                else:
                    color1='blue'
                track.rect(row["start"], row['end'], fc=color1, ec='none', lw=0.01,rasterized=True)
            gap+=2
        y0=y-gap
        y1=y0-5
        track = sector.add_track((y1, y0), r_pad_ratio=0.05)
        size=5.2 if ct=='CN GABA-Glut' else 6.5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        df_cg=data.loc[(data['chrom']==sector.name),['chrom','start',ct]]
        # track.heatmap(df_cg.fillna(0.5), width=5000, vmin=0, vmax=1, cmap='Blues_r')
        for _, row in df_cg.iterrows():
            if pd.isna(row[ct]):
                fc='black'
            elif row[ct] > 0.7: # hypermethylated: white color
                continue
            elif row[ct] > 0.5:
                fc='limegreen'
            elif row[ct] > 0.3:
                fc='green'
            else:
                fc='#3e27a9' # color
            end=min(row['start']+res,sector.size)
            track.rect(row["start"], end, fc=fc, ec=fc, lw=0.08,rasterized=True)
        gap+=5
        if ct==celltypes[0]:
            comp=df_comp.loc[df_comp.CellType==ct,['chr','start','end','score']]
            y0=y-gap
            y1=y0-2
            track = sector.add_track((y1, y0), r_pad_ratio=0.05)
            gap+=2
            track.axis(fc='white')
            comp1 = comp.loc[comp['chr']==sector.name]
            for _, row in comp1.iterrows():
                signal=row['score']
                if signal > 0:
                    color1='red'
                else:
                    color1='blue'
                track.rect(row["start"], row['end'], fc=color1, ec='none', lw=0.01,rasterized=True)
    y=y-gap-0.7

handles = [
    Patch(color="black", label="NA"),
    Patch(facecolor="white", edgecolor='black',linewidth=0.1,label="0.7 - 1"),
    Patch(color="limegreen", label="0.5 - 0.7"),
    Patch(color="green", label="0.3 - 0.5"),
    Patch(color="#3e27a9", label="0 - 0.3"),
]
fig = circos.plotfig(dpi=300, figsize=(5,5))
# circos.ax.set_rasterized(True)
_ = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.5), loc="center", fontsize=8, 
                     title="  5kb CG\nmethylation",title_fontsize=12,labelspacing=0.1,frameon=False)
fig.savefig(os.path.join(figdir,f'Class.{chrom}.5kb_mCG.ciscos_plot.pdf'))

# from matplotlib.colors import LinearSegmentedColormap
# my_cmap = LinearSegmentedColormap.from_list(
#             "test", [(0, "#3e27a9"), (0.3,'royalblue'), (0.5,'deepskyblue'), (0.7,'lightskyblue'),  (1, 'white')]
#         )
# circos.colorbar(bounds=(0.35, 0.45, 0.3, 0.01), vmin=0, vmax=1, orientation="horizontal", cmap=my_cmap)
# circos.savefig(os.path.join(figdir,'Subclass.5kb_mCG.ciscos_plot.pdf'),dpi=300, figsize=(5,5))

# display(circos.figure)

In [ ]:
# Chi-square test: test whether the heterochromatin regions loss more methylation

In [ ]:
df_meth=data.loc[data.chrom=='chr10'].set_index(['chrom','start']).loc[:,celltypes].applymap(lambda x:np.nan if pd.isna(x) else 'hyper' if x > 0.7 else 'intermedia' if x > 0.3 else 'hypo').reset_index()
df_meth.insert(2,'end',df_meth.start+5000)
df_meth.start=df_meth.start+1
df_meth

In [ ]:
comp=df_comp.loc[df_comp['chr']=='chr10'].pivot(index=['chr','start','end'],columns='CellType',values='AB').reset_index()
comp

In [ ]:
from scipy import stats
R=[]
for ct in celltypes:
    df1=df_meth.loc[df_meth[ct].notna(),['chrom','start','end',ct]].rename(columns={ct:'meth'})
    df2=comp.loc[:,['chr','start','end',ct]].dropna().rename(columns={ct:'AB'})
    bed1 = pybedtools.BedTool.from_dataframe(df1)
    bed2 = pybedtools.BedTool.from_dataframe(df2)
    intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
    df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','AB'])
    vc=df.groupby('AB')['meth'].value_counts().unstack()
    for meth_type in vc.columns.tolist():
        vc1=vc.copy()
        vc1[f'Non-{meth_type}']=vc1.apply(lambda x:x.sum() - x[meth_type],axis=1)
        test_data=vc1.loc[:,[meth_type,f'Non-{meth_type}']]
        print(test_data)
        r=stats.chi2_contingency(test_data.values)
        odd_ratio=(vc1.loc['A',meth_type] * vc1.loc['B',f'Non-{meth_type}']) / (vc1.loc['B',meth_type] * vc1.loc['A',f'Non-{meth_type}'])
        R.append([ct,meth_type,r.statistic,odd_ratio,r.pvalue])
df_res=pd.DataFrame(R,columns=['CellType','Variable','Statistic','OddRatio','P-value'])
df_res.sort_values('Statistic',ascending=False,inplace=True)
df_res.to_excel("5kb_AB_loss_methylation_chi2_square.xlsx")
pybedtools.cleanup(remove_all=True)

In [ ]:
df_res.sort_values('OddRatio') # enrichment of Variable in A compartments

## Plot DNA methylation (5kb mCH raw methylation at Class level)

In [ ]:
from ALLCools.mcds import MCDS
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
class_colors=D['Class'].Hex.to_dict()
chrom="chr10"

pseudobulk_mcds_path="~/Projects/BG/pseudobulk/Class/Class.mcds"
var_dim='5kb'
mc_type='CHN'
res=5000
mcds = MCDS.open(os.path.expanduser(pseudobulk_mcds_path), var_dim=var_dim)
mc_df = mcds[f'{var_dim}_da'].sel(count_type='mc', mc_type=mc_type).to_pandas().T
mc_df.columns = [col.replace('_', ' ') for col in mc_df.columns.tolist()]
cov_df = mcds[f'{var_dim}_da'].sel(count_type='cov', mc_type=mc_type).to_pandas().T
cov_df.columns = [col.replace('_', ' ') for col in cov_df.columns.tolist()]
# rows are bins, columns are cell_types
df_frac = calculate_posterior_mc_frac(mc_df.values, cov_df.values, normalize_per_cell=False)[0]  # bigger range with normalize_per_cell
data = pd.DataFrame(df_frac, index=mc_df.index.tolist(), columns=mc_df.columns.tolist())

data.insert(0,'chrom',data.index.to_series().apply(lambda x:x.split('_')[0]))
data.insert(1,'start',data.index.to_series().apply(lambda x:x.split('_')[1]).astype(int) * res)
data

In [ ]:
data.iloc[:,2:].max().max()

In [ ]:
data.iloc[:,2].quantile(0.2),data.iloc[:,2].quantile(0.5),data.iloc[:,2].quantile(0.8)

In [ ]:
data.iloc[:,5].quantile(0.2),data.iloc[:,5].quantile(0.5),data.iloc[:,5].quantile(0.8),data.iloc[:,5].median()

In [ ]:
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
celltypes=["M Dopa","F M Glut",'CN GABA-Glut',"F M GABA", 
           'CN LGE GABA','CN MGE GABA','CN CGE GABA',
           'Astro-Epen','Vascular','OPC-Oligo','Immune']

In [ ]:
df_comp=pd.read_csv(os.path.join(figdir,'Class.superdomain.tsv'),sep='\t')
# df_comp=df_comp.loc[df_comp['chr']==chrom]
df_comp

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import load_eukaryote_example_dataset
from matplotlib.patches import Patch,Rectangle

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

# Initialize Circos with chromosome sizes
circos = Circos(keep_chrom_sizes, space=1,start=15, end=345)
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")

circos.add_cytoband_tracks((97, 100), cytoband_file)
# Plot chromosome name
circos.text(chrom, r=98.5, size=10,color='black',ha='center',va='top')
for sector in circos.sectors:
    # sector.text(sector.name, size=10)
    # Plot outer track with xticks if only plot one chromosome
    major_ticks_interval = 10000000
    minor_ticks_interval = 5000000
    if sector.size >= major_ticks_interval:
        sector.get_track("cytoband").xticks_by_interval(
            major_ticks_interval, label_formatter=lambda v: f"{int(v / 10 ** 6)} Mb",label_size=6, #label_orientation="vertical",
        )
        sector.get_track("cytoband").xticks_by_interval(minor_ticks_interval, tick_length=1, show_label=False)

y=95
for ct in celltypes[::-1]:
    print(ct)
    color=class_colors[ct.replace('_',' ')]
    gap=0
    for sector in circos.sectors:
        if ct==celltypes[-1]:
            comp=df_comp.loc[df_comp.CellType==ct,['chr','start','end','score']]
            track = sector.add_track((y-2, y), r_pad_ratio=0.05)
            track.axis(fc='white')
            comp1 = comp.loc[comp['chr']==sector.name]
            for _, row in comp1.iterrows():
                signal=row['score']
                if signal > 0:
                    color1='red'
                else:
                    color1='blue'
                track.rect(row["start"], row['end'], fc=color1, ec='none', lw=0.01,rasterized=True)
            gap+=2
        y0=y-gap
        y1=y0-5
        track = sector.add_track((y1, y0), r_pad_ratio=0.05)
        size=5.2 if ct=='CN GABA-Glut' else 6.5
        if sector.name == circos.sectors[0].name:
            circos.text(ct, r=track.r_center, size=size,color=color,ha='center',va='top')
        track.axis(fc='white')
        df_cg=data.loc[(data['chrom']==sector.name),['chrom','start',ct]]
        # track.heatmap(df_cg.fillna(0.5), width=5000, vmin=0, vmax=1, cmap='Blues_r')
        for _, row in df_cg.iterrows():
            if pd.isna(row[ct]):
                fc='black'
            elif row[ct] > 0.07: # hypermethylated: white color
                continue
            elif row[ct] > 0.05:
                fc='limegreen'
            elif row[ct] > 0.03:
                fc='green'
            else:
                fc='#3e27a9' # color
            end=min(row['start']+res,sector.size)
            track.rect(row["start"], end, fc=fc, ec=fc, lw=0.08,rasterized=True)
        gap+=5
        if ct==celltypes[0]:
            comp=df_comp.loc[df_comp.CellType==ct,['chr','start','end','score']]
            y0=y-gap
            y1=y0-2
            track = sector.add_track((y1, y0), r_pad_ratio=0.05)
            gap+=2
            track.axis(fc='white')
            comp1 = comp.loc[comp['chr']==sector.name]
            for _, row in comp1.iterrows():
                signal=row['score']
                if signal > 0:
                    color1='red'
                else:
                    color1='blue'
                track.rect(row["start"], row['end'], fc=color1, ec='none', lw=0.01,rasterized=True)
    y=y-gap-0.7

handles = [
    Patch(color="black", label="NA"),
    Patch(facecolor="white", edgecolor='black',linewidth=0.1,label=">=0.07"),
    Patch(color="limegreen", label="0.05 - 0.07"),
    Patch(color="green", label="0.03 - 0.05"),
    Patch(color="#3e27a9", label="0 - 0.03"),
]
fig = circos.plotfig(dpi=300, figsize=(5,5))
# circos.ax.set_rasterized(True)
_ = circos.ax.legend(handles=handles, bbox_to_anchor=(0.5, 0.5), loc="center", fontsize=8, 
                     title="  5kb CH\nmethylation",title_fontsize=12,labelspacing=0.1,frameon=False)
fig.savefig(os.path.join(figdir,f'Class.{chrom}.5kb_mCH.ciscos_plot.pdf'))

In [ ]:
df_meth=data.loc[data.chrom=='chr10'].set_index(['chrom','start']).loc[:,celltypes].applymap(lambda x:np.nan if pd.isna(x) else 'hyper' if x > 0.7 else 'intermedia' if x > 0.3 else 'hypo').reset_index()
df_meth.insert(2,'end',df_meth.start+5000)
df_meth.start=df_meth.start+1
df_meth

In [ ]:
comp=df_comp.loc[df_comp['chr']=='chr10'].pivot(index=['chr','start','end'],columns='CellType',values='AB').reset_index()
comp

In [ ]:
from scipy import stats
R=[]
for ct in celltypes:
    df1=df_meth.loc[df_meth[ct].notna(),['chrom','start','end',ct]].rename(columns={ct:'meth'})
    df2=comp.loc[:,['chr','start','end',ct]].dropna().rename(columns={ct:'AB'})
    bed1 = pybedtools.BedTool.from_dataframe(df1)
    bed2 = pybedtools.BedTool.from_dataframe(df2)
    intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
    df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','AB'])
    vc=df.groupby('AB')['meth'].value_counts().unstack()
    for meth_type in vc.columns.tolist():
        vc1=vc.copy()
        vc1[f'Non-{meth_type}']=vc1.apply(lambda x:x.sum() - x[meth_type],axis=1)
        test_data=vc1.loc[:,[meth_type,f'Non-{meth_type}']]
        print(test_data)
        r=stats.chi2_contingency(test_data.values)
        odd_ratio=(vc1.loc['A',meth_type] * vc1.loc['B',f'Non-{meth_type}']) / (vc1.loc['B',meth_type] * vc1.loc['A',f'Non-{meth_type}'])
        R.append([ct,meth_type,r.statistic,odd_ratio,r.pvalue])
df_res=pd.DataFrame(R,columns=['CellType','Variable','Statistic','OddRatio','P-value'])
df_res.sort_values('Statistic',ascending=False,inplace=True)
df_res.to_excel("5kb_AB_loss_methylation_chi2_square.xlsx")
pybedtools.cleanup(remove_all=True)

In [ ]:
df_res.sort_values('OddRatio') # enrichment of Variable in A compartments

## Plot raw contacts at Subclass level

In [ ]:
contact_table_path="/home/x-wding2/Projects/BG/HIC/contact_table_filtered.downsampled.tsv"
metadata_path="/home/x-wding2/Projects/BG/clustering/100kb/annotations.tsv"
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
chrom="chr4"
# celltypes=["CN ONECUT1 GABA","CN LHX8 GABA",'M Dopa',"F Glut", #"F M Glut","F Glut",
#            'STR Hybrid MSN','STR D2 MSN','STR D1 MSN',
#            'Astrocyte','Oligodendrocyte','Lymphocyte'] + ['CN GABA-Glut','M Dopa','STR Hybrid MSN','Astrocyte','Lymphocyte']
celltypes=[
           'STR D1 MSN','STR D2 MSN','STR Hybrid MSN',"F M GATA3 GABA", 'M Dopa',"CN ONECUT1 GABA","CN LHX8 GABA","F Glut",'CN ST18 GABA','Astrocyte',
           'Oligodendrocyte','OPC','Microglia','Lymphocyte']

In [ ]:
metadata=pd.read_csv(metadata_path,sep='\t',index_col=0)
D=pd.read_csv(contact_table_path,sep='\t',header=None,names=['cell','path']).set_index('cell').path.to_dict()
metadata['path']=metadata.index.to_series().map(D)
metadata=metadata.loc[metadata.path.notna()]
print(metadata['Subclass'].value_counts().loc[celltypes].min())
# randomly select 100 cells from each subclass
usecells=metadata.groupby('Subclass').apply(lambda x:x.sample(100).index.tolist() if x.shape[0] >=100 else x.index.tolist()).sum()
df_meta=metadata.loc[usecells,['Subclass','path']]
df_meta=df_meta.loc[df_meta.Subclass.isin(celltypes)]
# df_meta.set_index('Subclass',inplace=True)
print(df_meta.shape)
df_meta.head()

In [ ]:
df_meta['Subclass'].value_counts().loc[celltypes]

In [ ]:
# read raw contacts (100 cells per subclass)
R=[]
for _,row in df_meta.iterrows():
    ct=row['Subclass']
    # print(ct,row.path)
    df=pd.read_csv(row.path,sep='\t',usecols=[1,2,5,6],header=None)
    df.columns=['chr1','start1','chr2','start2']
    # only select chr4
    df=df.loc[(df.chr1 == df.chr2) & (df.chr1==chrom)]
    df['CellType']=ct
    df.rename(columns={'chr1':'chrom'},inplace=True)
    R.append(df.loc[:,['chrom','start1','start2','CellType']].drop_duplicates())
df_raw_contact=pd.concat(R)
print(df_raw_contact.shape)
# df_raw_contact=df_raw_contact.loc[df_raw_contact.start1 < df_raw_contact.start2]
df_raw_contact

In [ ]:
from pycirclize.utils import load_eukaryote_example_dataset
chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
cytoband=pd.read_csv(cytoband_file,sep='\t')
cytoband=cytoband.loc[cytoband['#chrom']==chrom]
cytoband.head()

In [ ]:
df_raw_contact['gieStain1']=np.nan
df_raw_contact['gieStain2']=np.nan
for _,row in cytoband.iterrows():
    df_raw_contact.loc[df_raw_contact.start1.between(row['chromStart'],row['chromEnd']),'gieStain1']=row['gieStain']
    df_raw_contact.loc[df_raw_contact.start2.between(row['chromStart'],row['chromEnd']),'gieStain2']=row['gieStain']
df_raw_contact.head()

In [ ]:
df_raw_contact.gieStain1.unique()

In [ ]:
def hypergeometric(x, N, n, G, alternative='greater'):
    """
        https://www.programcreek.com/python/example/121129/scipy.stats.hypergeom.sf
    Parameters
        一共N个动物，其中有n只狗，从中取出G个，有x只狗的p值
    ----------
    x : int
        number of `good` elements observed in the sample
    N : int
        population size
    n : int
       sample size
    G : int
       hypothesized number of good elements in population
    alternative : {'greater', 'less', 'two-sided'}
       alternative hypothesis to test (default: 'greater')
    Returns
    -------
    float
       estimated p-value
    """
    from scipy.stats import hypergeom
    if n < x:
        raise ValueError("Cannot observe more good elements than the sample size")
    if N < n:
        raise ValueError("Population size cannot be smaller than sample")
    if N < G:
        raise ValueError("Number of good elements can't exceed the population size")
    if G < x:
        raise ValueError("Number of observed good elements can't exceed the number in the population")

    assert alternative in ("two-sided", "less", "greater")
    if n < x:
        raise ValueError("Cannot observe more successes than the population size")

    plower = hypergeom.cdf(x, N, G, n)
    pupper = hypergeom.sf(x - 1, N, G, n)
    if alternative == 'greater':
        return pupper
    elif alternative == 'less':
        return plower
    else:  # two-sided
        # For two-sided, sum probabilities of outcomes with equal or lower probability
        pmf_x = hypergeom.pmf(x, N, G, n)  # Probability of exactly x successes
        all_x = np.arange(max(0, n + G - N), min(n, G) + 1)  # Possible x values
        pmf_all = hypergeom.pmf(all_x, N, G, n)  # PMF for all possible x
        pvalue = np.sum(pmf_all[pmf_all <= pmf_x + 1e-10])  # Sum probs for outcomes as extreme as x
        return min(pvalue, 1.0)  # Ensure p-value doesn't exceed 1 due to numerical precision

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
from pycirclize.utils import ColorCycler

chrom_sizes = pd.read_csv(os.path.expanduser(chrom_size),sep='\t',index_col=0,header=None,names=['chrom','size'])
D={'X':23,'Y':24,'M':25}
chrom_sizes['ID']=chrom_sizes.index.to_series().apply(lambda x:x.replace('chr','')).apply(lambda x:int(x) if x not in ['X','Y','M'] else D[x])
chrom_sizes=chrom_sizes.loc[~ chrom_sizes.ID.isin([23,24,25])]
chrom_sizes.sort_values('ID',ascending=True,inplace=True)
chrom_sizes = chrom_sizes['size'].to_dict()
# keep_chrom_sizes=chrom_sizes
keep_chrom_sizes={}
keep_chrom_sizes[chrom]=chrom_sizes[chrom]

ncols=3
nrows=int(np.ceil(len(celltypes) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.5*ncols, 2.5*nrows), sharex='all',
                         dpi=300,gridspec_kw=dict(hspace=0.01),subplot_kw=dict(projection="polar"),)

for i,ct in enumerate(celltypes):
    ax=axes.flatten()[i]
    # circos.plotfig(ax=ax)
    circos = Circos(keep_chrom_sizes, space=1,start=10, end=350)
    chr_bed_file, cytoband_file, _ = load_eukaryote_example_dataset("hg38")
    circos.add_cytoband_tracks((95, 100), cytoband_file)

    link=df_raw_contact.loc[(df_raw_contact.CellType==ct) & (df_raw_contact.chrom==chrom)]
    link['distance']=(link.start2-link.start1).abs()
    N=link.shape[0]
    n=link.loc[(link.gieStain1.isin(['gpos100','gpos75'])) | (link.gieStain2.isin(['gpos100','gpos75']))].shape[0]
    link=link.loc[link.distance >= 10e6] # only show long range contact (> 5Mb)
    color=group_colors[ct.replace('_',' ')]
    # circos.text("Long range contact\n(> 10Mb) on chr4", r=sector.get_track("cytoband").r_center, 
    #             size=7,color='black',ha='center',va='top')
    G=link.shape[0]
    x=link.loc[(link.gieStain1.isin(['gpos100','gpos75'])) | (link.gieStain2.isin(['gpos100','gpos75']))].shape[0]
    frac=x / N
    p=hypergeometric(x, N, n, G, alternative='greater')
    print(ct,frac,x, N, n, G,p) #\np-value={p:.2e}
    circos.text(f"{ct} ({frac * 100:.1f}%)", r=100, 
                size=9,color=color,ha='center',va='bottom')
    # only visualize 10% of the long range contacts
    # link=link.sample(int(G * 0.1))
    # only visualize 10,000 long range contacts
    link=link.sample(10000)
    for _,row in link.iterrows():
        if row['gieStain1'] in ['gpos100','gpos75'] or row['gieStain2'] in ['gpos100','gpos75']:
            ec='black'
            lw=0.03
            alpha=0.2
        else:
            ec=color
            lw=0.02
            alpha=0.1
        circos.link((row['chrom'],row['start1']-1,row['start1']), (row['chrom'],row['start2']-1,row['start2']), 
                    color=color,rasterized=True,ec=ec,lw=lw,alpha=alpha)

    # circos.savefig(os.path.join(figdir,f"{ct.replace(' ','_')}.raw_contacts.ciscos_plot.pdf"), dpi=300, figsize=(2.5,2.5))
    ax.set_rasterized(True)
    fig = circos.plotfig(ax=ax)
if i < nrows*ncols:
    for j in range(i+1,nrows*ncols):
        ax=axes.flatten()[j]
        ax.set_axis_off()

plt.tight_layout()
plt.savefig(os.path.join(figdir,f'Subclass.{chrom}.raw_contacts.ciscos_plot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos

colors=dict(acen='royalblue',gpos100='darkviolet',gpos75='crimson',gpos50='deeppink',gpos25='pink',gneg='dodgerblue')
# colors=dict(Long='crimson',Short='lightgreen',Heterochromatin='blue',other='dodgerblue')

ncols=3
nrows=int(np.ceil(len(celltypes) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.5*ncols, 2.5*nrows), sharex='all',
                         dpi=300,gridspec_kw=dict(hspace=0.2),subplot_kw=dict(projection="polar"),)

for i,ct in enumerate(celltypes):
    ax=axes.flatten()[i]
    link=df_raw_contact.loc[(df_raw_contact.CellType==ct)]
    link['distance']=(link.start2-link.start1).abs()
    N=link.shape[0]
    # n=link.loc[(link.gieStain1.isin(['gpos100','gpos75'])) | (link.gieStain2.isin(['gpos100','gpos75']))].shape[0]
    # G=link.shape[0]
    # x=link.loc[(link.gieStain1.isin(['gpos100','gpos75'])) | (link.gieStain2.isin(['gpos100','gpos75']))].shape[0]
    # frac=x / N
    # print(ct,frac,x) #\np-value={p:.2e}

    link=link.loc[link.distance >= 40e6,['gieStain1','gieStain2']] #.sample(10000) # downsample to 10,000 contacts
    # link=link.loc[link.distance < 5e6,['gieStain1','gieStain2']] #.sample(10000) # downsample to 10,000 contacts
    data=link.value_counts().unstack() ## only show long range contact (> 10Mb)
    # data=data.apply(lambda x:x / x.sum())
    # data=data.apply(lambda x:x / x.sum(),axis=1)

    circos = Circos.chord_diagram(
        data,
        space=5,
        cmap=colors,
        label_kws=dict(size=6),
        order=list(colors.keys()),
        link_kws=dict(ec="black", lw=0.5, direction=1),
    )
    color=group_colors[ct]
    circos.text(f"{ct} ({frac * 100:.1f}%)", r=120, 
            size=9,color=color,ha='center',va='bottom')

    # circos.savefig(os.path.join(figdir,f"{ct.replace(' ','_')}.raw_contacts.ciscos_plot.pdf"), dpi=300, figsize=(2.5,2.5))
    # ax.set_rasterized(True)
    fig = circos.plotfig(ax=ax)
if i < nrows*ncols:
    for j in range(i+1,nrows*ncols):
        ax=axes.flatten()[j]
        ax.set_axis_off()

plt.tight_layout()
plt.savefig(os.path.join(figdir,f'Subclass.{chrom}.long.raw_contacts.chord_diagram.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
link

In [ ]:
data

## Plot raw contacts chord_diagram at Class level (Inter compartments)

In [ ]:
contact_table_path="/home/x-wding2/Projects/BG/HIC/contact_table_filtered.downsampled.tsv"
metadata_path="/home/x-wding2/Projects/BG/clustering/100kb/annotations.tsv"
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
chrom="chr10"
celltypes=["M Dopa","F M Glut",'CN GABA-Glut',"F M GABA", 
           'CN MGE GABA','CN LGE GABA','CN CGE GABA',
           'Astro-Epen','OPC-Oligo','Immune','Vascular',]

In [ ]:
metadata=pd.read_csv(metadata_path,sep='\t',index_col=0)
D=pd.read_csv(contact_table_path,sep='\t',header=None,names=['cell','path']).set_index('cell').path.to_dict()
metadata['path']=metadata.index.to_series().map(D)
metadata=metadata.loc[metadata.path.notna()]
print(metadata['Class'].value_counts().loc[celltypes].min())
# randomly select 100 cells from each class
usecells=metadata.groupby('Class').apply(lambda x:x.sample(100).index.tolist() if x.shape[0] >=100 else x.index.tolist()).sum()
df_meta=metadata.loc[usecells,['Class','path']]
df_meta=df_meta.loc[df_meta.Class.isin(celltypes)]
# df_meta.set_index('Subclass',inplace=True)
print(df_meta.shape)
df_meta.head()

In [ ]:
df_meta['Class'].value_counts().loc[celltypes]

In [ ]:
# read raw contacts (100 cells per Class)
R=[]
for _,row in df_meta.iterrows():
    ct=row['Class']
    # print(ct,row.path)
    df=pd.read_csv(row.path,sep='\t',usecols=[1,2,5,6],header=None)
    df.columns=['chr1','start1','chr2','start2']
    # only select chr10
    df=df.loc[(df.chr1 == df.chr2) & (df.chr1==chrom)]
    df['CellType']=ct
    df.rename(columns={'chr1':'chrom'},inplace=True)
    R.append(df.loc[:,['chrom','start1','start2','CellType']].drop_duplicates())
df_raw_contact=pd.concat(R)
print(df_raw_contact.shape)
# df_raw_contact=df_raw_contact.loc[df_raw_contact.start1 < df_raw_contact.start2]
df_raw_contact['ID1']=df_raw_contact.chrom.map(str)+'-'+df_raw_contact.start1.map(str)
df_raw_contact['ID2']=df_raw_contact.chrom.map(str)+'-'+df_raw_contact.start2.map(str)
df_raw_contact

In [ ]:
df_comp=pd.read_csv(os.path.join(figdir,'Class.superdomain.tsv'),sep='\t')
df_comp

In [ ]:
import pybedtools
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))
df1=df_raw_contact.loc[:,['chrom','start1','CellType']].drop_duplicates()
df1.insert(2,'end1',df1.start1+1)
bed1 = pybedtools.BedTool.from_dataframe(df1)
bed2 = pybedtools.BedTool.from_dataframe(df_comp)
intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','CellType2','AB','score'])
df=df.loc[df.CellType==df.CellType2]
df['ID']=df.chrom.map(str)+'-'+df.start1.map(str)
D=df.loc[:,['ID','AB']].drop_duplicates().set_index('ID').AB.to_dict()
df_raw_contact['AB1']=df_raw_contact.ID1.map(D).fillna('NA')

# start2
df1=df_raw_contact.loc[:,['chrom','start2','CellType']].drop_duplicates()
df1.insert(2,'end1',df1.start2+1)
df1.rename(columns={'start2':'start1'},inplace=True)
bed1 = pybedtools.BedTool.from_dataframe(df1)
intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','CellType2','AB','score'])
df=df.loc[df.CellType==df.CellType2]
df['ID']=df.chrom.map(str)+'-'+df.start1.map(str)
D=df.loc[:,['ID','AB']].drop_duplicates().set_index('ID').AB.to_dict()
df_raw_contact['AB2']=df_raw_contact.ID2.map(D).fillna('NA')
df_raw_contact

In [ ]:
df_raw_contact['AB']=df_raw_contact.AB1.map(str)+'|'+df_raw_contact.AB2.map(str)
df_raw_contact['AB']=df_raw_contact['AB'].replace({'NA|A':'A|NA','B|A':'A|B','NA|B':'B|NA'})
df_raw_contact['distance']=(df_raw_contact.start2-df_raw_contact.start1).abs()
df_raw_contact.to_csv(os.path.join(figdir,f'Class.{chrom}.long_contacts.chord_diagram.tsv'),sep='\t',index=False)
df_raw_contact

In [ ]:
df_raw_contact=pd.read_csv(os.path.join(figdir,f'Class.{chrom}.long_contacts.chord_diagram.tsv'),sep='\t')
df_raw_contact.AB1=df_raw_contact.AB1.fillna('NA')
df_raw_contact.AB2=df_raw_contact.AB2.fillna('NA')
df_raw_contact['AB']=df_raw_contact['AB'].replace({'NA|A':'A|NA','B|A':'A|B','NA|B':'B|NA'})
df_raw_contact

In [ ]:
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
class_colors=D['Class'].Hex.to_dict()

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos

colors={'Short':'lightseagreen','Long':'lightcoral','A|A':'crimson','A|B':'darkcyan','B|B':'yellowgreen','Other':'gold',}

ncols=4
nrows=int(np.ceil(len(celltypes) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.5*ncols, 2.6*nrows), sharex='all',
                         dpi=300,gridspec_kw=dict(hspace=0.3),subplot_kw=dict(projection="polar"),)

for i,ct in enumerate(celltypes):
    color=class_colors[ct.replace('_',' ')]
    ax=axes.flatten()[i]
    link=df_raw_contact.loc[(df_raw_contact.CellType==ct) & (df_raw_contact.chrom==chrom)]
    N=link.shape[0]
    n=link.loc[link.distance >= 5e6].shape[0]
    link1=link.loc[link.AB.isin(['A|B','B|A'])]
    G=link1.shape[0]
    x=link1.loc[link1.distance >= 5e6].shape[0]
    frac_of_long_in_inter=x / G
    p=hypergeometric(x, N, n, G, alternative='greater')
    print(ct,frac_of_long_in_inter,x, N, n, G,p) #\np-value={p:.2e}
    link['distance_type']=link.distance.apply(lambda x:'Long' if x >= 5e6 else 'Short')
    link['AB']=link['AB'].apply(lambda x:x if 'NA' not in x else 'Other')
    data=link.loc[:,['distance_type','AB']].value_counts().unstack()
    row_frac= (100 * data.sum(axis=1) / data.sum(axis=1).sum()).to_dict() # long and short range
    col_frac= (100 * data.sum() / data.sum().sum()).to_dict()
    col_rename_dict={}
    for col in data.columns:
        if col!='Other':
            frac=round(col_frac[col],2)
            col_rename_dict[col]=f"{col}({frac:.1f}%)"
    row_rename_dict={}
    for row in data.index:
        frac=round(row_frac[row],2)
        row_rename_dict[row]=f"{row}({frac:.1f}%)"
    print(col_rename_dict,row_rename_dict)

    data.rename(columns=col_rename_dict,index=row_rename_dict,inplace=True)
    tmp_colors={}
    for k in colors:
        if k in col_rename_dict:
            tmp_colors[col_rename_dict[k]]=colors[k]
        elif k in row_rename_dict:
            tmp_colors[row_rename_dict[k]]=colors[k]
        else:
            tmp_colors[k]=colors[k]

    circos = Circos.chord_diagram(
        data,
        space=5,
        cmap=tmp_colors,
        label_kws=dict(size=8),
        order=list(tmp_colors.keys()),
        link_kws=dict(ec="black", lw=0.5, direction=1),
    )
    circos.text(f"{ct} ({frac_of_long_in_inter * 100:.1f}%)", r=120, 
            size=10,color=color,ha='center',va='bottom')

    fig = circos.plotfig(ax=ax)
if i < nrows*ncols:
    for j in range(i+1,nrows*ncols):
        ax=axes.flatten()[j]
        ax.set_axis_off()

plt.tight_layout()
plt.savefig(os.path.join(figdir,f'Class.{chrom}.long_contacts.chord_diagram.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

## Plot raw contacts chord_diagram at Subclass level (Inter compartments)

In [ ]:
contact_table_path="/home/x-wding2/Projects/BG/HIC/contact_table_filtered.downsampled.tsv"
metadata_path="/home/x-wding2/Projects/BG/clustering/100kb/annotations.tsv"
chrom_size="~/Ref/hg38/hg38_ucsc.main.chrom.sizes"
chrom="chr4"
celltypes=[
           'M Dopa',"CN LHX8 GABA","F Glut","CN ONECUT1 GABA",'CN ST18 GABA',"F M GATA3 GABA",'STR D2 MSN','STR D1 MSN','STR Hybrid MSN', 'Astrocyte',
           'OPC','Oligodendrocyte','Microglia','Lymphocyte']

In [ ]:
metadata=pd.read_csv(metadata_path,sep='\t',index_col=0)
D=pd.read_csv(contact_table_path,sep='\t',header=None,names=['cell','path']).set_index('cell').path.to_dict()
metadata['path']=metadata.index.to_series().map(D)
metadata=metadata.loc[metadata.path.notna()]
print(metadata['Subclass'].value_counts().loc[celltypes].min())
# randomly select 100 cells from each subclass
usecells=metadata.groupby('Subclass').apply(lambda x:x.sample(100).index.tolist() if x.shape[0] >=100 else x.index.tolist()).sum()
df_meta=metadata.loc[usecells,['Subclass','path']]
df_meta=df_meta.loc[df_meta.Subclass.isin(celltypes)]
# df_meta.set_index('Subclass',inplace=True)
print(df_meta.shape)
df_meta.head()

In [ ]:
df_meta['Subclass'].value_counts().loc[celltypes]

In [ ]:
# read raw contacts (100 cells per subclass)
R=[]
for _,row in df_meta.iterrows():
    ct=row['Subclass']
    # print(ct,row.path)
    df=pd.read_csv(row.path,sep='\t',usecols=[1,2,5,6],header=None)
    df.columns=['chr1','start1','chr2','start2']
    # only select chr4
    df=df.loc[(df.chr1 == df.chr2) & (df.chr1==chrom)]
    df['CellType']=ct
    df.rename(columns={'chr1':'chrom'},inplace=True)
    R.append(df.loc[:,['chrom','start1','start2','CellType']].drop_duplicates())
df_raw_contact=pd.concat(R)
df_raw_contact['ID1']=df_raw_contact.chrom.map(str)+'-'+df_raw_contact.start1.map(str)
df_raw_contact['ID2']=df_raw_contact.chrom.map(str)+'-'+df_raw_contact.start2.map(str)
print(df_raw_contact.shape)
df_raw_contact

In [ ]:
df_comp=pd.read_csv(os.path.join(figdir,'Subclass.superdomain.tsv'),sep='\t')
df_comp

In [ ]:
import pybedtools
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))
df1=df_raw_contact.loc[:,['chrom','start1','CellType']].drop_duplicates()
df1.insert(2,'end1',df1.start1+1)
bed1 = pybedtools.BedTool.from_dataframe(df1)
bed2 = pybedtools.BedTool.from_dataframe(df_comp)
intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','CellType2','AB','score'])
df=df.loc[df.CellType==df.CellType2]
df['ID']=df.chrom.map(str)+'-'+df.start1.map(str)
D=df.loc[:,['ID','AB']].drop_duplicates().set_index('ID').AB.to_dict()
df_raw_contact['AB1']=df_raw_contact.ID1.map(D).fillna('NA')

# start2
df1=df_raw_contact.loc[:,['chrom','start2','CellType']].drop_duplicates()
df1.insert(2,'end1',df1.start2+1)
df1.rename(columns={'start2':'start1'},inplace=True)
bed1 = pybedtools.BedTool.from_dataframe(df1)
intersection = bed1.intersect(bed2, wa=True,wb=True) # loj=True
df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2','CellType2','AB','score'])
df=df.loc[df.CellType==df.CellType2]
df['ID']=df.chrom.map(str)+'-'+df.start1.map(str)
D=df.loc[:,['ID','AB']].drop_duplicates().set_index('ID').AB.to_dict()
df_raw_contact['AB2']=df_raw_contact.ID2.map(D).fillna('NA')
df_raw_contact

In [ ]:
df_raw_contact['AB']=df_raw_contact.AB1.map(str)+'|'+df_raw_contact.AB2.map(str)
df_raw_contact['AB']=df_raw_contact['AB'].replace({'NA|A':'A|NA','B|A':'A|B','NA|B':'B|NA'})
df_raw_contact['distance']=(df_raw_contact.start2-df_raw_contact.start1).abs()
df_raw_contact.to_csv(os.path.join(figdir,f'Subclass.{chrom}.long_contacts.chord_diagram.tsv'),sep='\t',index=False)
df_raw_contact

In [ ]:
df_raw_contact=pd.read_csv(os.path.join(figdir,f'Subclass.{chrom}.long_contacts.chord_diagram.tsv'),sep='\t')
df_raw_contact.AB1=df_raw_contact.AB1.fillna('NA')
df_raw_contact.AB2=df_raw_contact.AB2.fillna('NA')
df_raw_contact['AB']=df_raw_contact['AB'].replace({'NA|A':'A|NA','B|A':'A|B','NA|B':'B|NA'})
df_raw_contact

In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos

colors={'Short':'lightseagreen','Long':'lightcoral','A|A':'crimson','A|B':'darkcyan','B|B':'yellowgreen','Other':'gold',}

ncols=3
nrows=int(np.ceil(len(celltypes) / 3))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.5*ncols, 2.6*nrows), sharex='all',
                         dpi=300,gridspec_kw=dict(hspace=0.3),subplot_kw=dict(projection="polar"),)

for i,ct in enumerate(celltypes):
    color=group_colors[ct.replace('_',' ')]
    ax=axes.flatten()[i]
    link=df_raw_contact.loc[(df_raw_contact.CellType==ct) & (df_raw_contact.chrom==chrom)]
    N=link.shape[0]
    n=link.loc[link.distance >= 5e6].shape[0]
    link1=link.loc[link.AB.isin(['A|B','B|A'])]
    G=link1.shape[0]
    x=link1.loc[link1.distance >= 5e6].shape[0]
    frac_of_long_in_inter=x / G
    p=hypergeometric(x, N, n, G, alternative='greater')
    print(ct,frac,x, N, n, G,p) #\np-value={p:.2e}
    link['distance_type']=link.distance.apply(lambda x:'Long' if x >= 5e6 else 'Short')
    link['AB']=link['AB'].apply(lambda x:x if 'NA' not in x else 'Other')
    data=link.loc[:,['distance_type','AB']].value_counts().unstack()
    row_frac= (100 * data.sum(axis=1) / data.sum(axis=1).sum()).to_dict() # long and short range
    col_frac= (100 * data.sum() / data.sum().sum()).to_dict()
    col_rename_dict={}
    for col in data.columns:
        if col!='Other':
            frac=round(col_frac[col],2)
            col_rename_dict[col]=f"{col}({frac:.1f}%)"
    row_rename_dict={}
    for row in data.index:
        frac=round(row_frac[row],2)
        row_rename_dict[row]=f"{row}({frac:.1f}%)"
    print(col_rename_dict,row_rename_dict)

    data.rename(columns=col_rename_dict,index=row_rename_dict,inplace=True)
    tmp_colors={}
    for k in colors:
        if k in col_rename_dict:
            tmp_colors[col_rename_dict[k]]=colors[k]
        elif k in row_rename_dict:
            tmp_colors[row_rename_dict[k]]=colors[k]
        else:
            tmp_colors[k]=colors[k]

    circos = Circos.chord_diagram(
        data,
        space=5,
        cmap=tmp_colors,
        label_kws=dict(size=8),
        order=list(tmp_colors.keys()),
        link_kws=dict(ec="black", lw=0.5, direction=1),
    )
    circos.text(f"{ct} ({frac_of_long_in_inter * 100:.1f}%)", r=120, 
            size=10,color=color,ha='center',va='bottom')

    fig = circos.plotfig(ax=ax)
if i < nrows*ncols:
    for j in range(i+1,nrows*ncols):
        ax=axes.flatten()[j]
        ax.set_axis_off()

plt.tight_layout()
plt.savefig(os.path.join(figdir,f'Subclass.{chrom}.long_contacts.chord_diagram.pdf'),transparent=True, bbox_inches='tight',dpi=300)
plt.show()

## (Revision) Lineage-specific differential compartment landscape (dcHiC)

The pre-computed dcHiC results group Subclasses into four developmental
lineages (Glut/Sero/Dopa, Subpallium GABA, Subpallium GABA-Glut, Nonneuron).
We summarise the *significant compartment-switching bins* per lineage and ask:

1. **Manhattan**: where on the genome each lineage switches.
2. **UpSet**: which switching bins are unique vs shared between lineages.
3. **Gene-set enrichment** of the lineage-specific switching bins.

This is more compact than the per-Subclass switch tables in the existing
section, and exposes a *4-lineage 3D regulatory grammar* that is currently
hidden in the supplementary data.


In [ ]:
# dcHiC lineage / class compartment switching.
#
# Set `level` to 'Class' (default; uses the BG_Class/.../ClassDiffComp/merged
# dcHiC run with all 11 Classes as columns) or 'Neighborhood' (uses the four
# sub-dirs Glut_Sero_Dopa / Subpallium_GABA / Subpallium_GABA-Glut / Nonneuron
# and treats each Neighborhood as one row, with its switching score = max-min
# across the Classes that compose that Neighborhood).
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt

level = 'Class'   # 'Class' or 'Neighborhood'
assert level in ('Class', 'Neighborhood'), level

if level == 'Class':
    merged_filt = ('/home/x-wding2/Projects/BG/HIC/BG_Class/compartment/'
                   'ClassDiffComp/merged/DifferentialResult/merged/fdr_result/'
                   'differential.intra_sample_combined.Filtered.pcQnm.bedGraph')
    df_cls = pd.read_csv(merged_filt, sep='\t')
    meta_cols = {'chr', 'start', 'end', 'sample_maha', 'pval', 'padj', 'dist_clust'}
    class_cols = [c for c in df_cls.columns
                  if c not in meta_cols and not c.endswith('_100Kb')
                  and not c.startswith('sample')]
    print('Class-level dcHiC columns:', class_cols)
    print('total bins after FDR filter:', len(df_cls))

    vals = df_cls[class_cols].astype(float)
    mean_across = vals.mean(axis=1)
    diff_mat = vals.subtract(mean_across, axis=0)

    lineage_switches = {}
    for cl in class_cols:
        df = df_cls[['chr', 'start', 'end']].copy()
        df['pc1_diff'] = diff_mat[cl].values
        df['bin_id']   = df['chr'].astype(str) + ':' + df['start'].astype(str)
        df = df.rename(columns={'chr': 'chrom'})
        lineage_switches[cl] = df

else:  # Neighborhood
    nb_dirs = {
        'Glut Sero Dopa':       '/home/x-wding2/Projects/BG/HIC/BG_Class/compartment/ClassDiffComp/Glut_Sero_Dopa/DifferentialResult',
        'Subpallium GABA':      '/home/x-wding2/Projects/BG/HIC/BG_Class/compartment/ClassDiffComp/Subpallium_GABA/DifferentialResult',
        'Subpallium GABA-Glut': '/home/x-wding2/Projects/BG/HIC/BG_Class/compartment/ClassDiffComp/Subpallium_GABA-Glut/DifferentialResult',
        'Nonneuron':            '/home/x-wding2/Projects/BG/HIC/BG_Class/compartment/ClassDiffComp/Nonneuron/DifferentialResult',
    }

    def _find_bedgraphs(root):
        pats = [
            os.path.join(root, '*', 'fdr_result', 'differential.intra_sample_combined.Filtered.pcQnm.bedGraph'),
            os.path.join(root, '*', 'fdr_result', 'differential.intra_sample_chr*_combined.pcQnm.bedGraph'),
            os.path.join(root, '*', '*_data',     'intra_chr*_combined.pcQnm.bedGraph'),
        ]
        for p in pats:
            files = glob.glob(p)
            if files:
                return files, p
        return [], None

    lineage_switches = {}
    for nb, root in nb_dirs.items():
        files, pat = _find_bedgraphs(root)
        print(nb, 'files:', len(files))
        if not files:
            continue
        df_list = [pd.read_csv(f, sep='\t', header=0) for f in files]
        df = pd.concat(df_list, ignore_index=True)
        meta_cols = {'chr', 'start', 'end', 'sample_maha', 'pval', 'padj', 'dist_clust'}
        ct_cols = [c for c in df.columns if c not in meta_cols and not c.endswith('_100Kb')]
        if not ct_cols:
            ct_cols = [c for c in df.columns if c.endswith('_100Kb')]
        vals = df[ct_cols].astype(float)
        if vals.shape[1] >= 2:
            pc1_diff = (vals.max(axis=1) - vals.min(axis=1)) * np.sign(vals.mean(axis=1))
        else:
            pc1_diff = vals.iloc[:, 0]
        df['pc1_diff'] = pc1_diff
        df['bin_id']   = df['chr'].astype(str) + ':' + df['start'].astype(str)
        df = df.rename(columns={'chr': 'chrom'})[
            ['chrom', 'start', 'end', 'pc1_diff', 'bin_id']]
        lineage_switches[nb] = df

print('level =', level, '| n groups =', len(lineage_switches))
{k: v.shape for k, v in lineage_switches.items()}


In [ ]:
# ---- This figure is only meaningful at Class level -------------------------
# Skip cleanly (rather than failing) when the notebook is parameterised to
# Subclass level — the MSN-focused cell below handles that case instead.
if level != 'Class':
    print(f"[skip] '{level}.dchic_lineage_compartment_manhattan_markers.pdf' "
          f"is only generated at Class level (current level={level!r}).")
else:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    from adjustText import adjust_text

    # ---- genome coord setup ----
    chrom_order_main = [c for c in chrom_sizes.index]
    offsets, cum = {}, 0
    for c in chrom_order_main:
        offsets[c] = cum
        cum += chrom_sizes[c]
    BIN = 100_000
    chrom_bounds = [offsets[c] + chrom_sizes[c] for c in chrom_order_main[:-1]]

    # ---- Class -> Neighborhood mapping ----
    _class2nbr_raw = {
        'F M Glut':         'Glut Sero Dopa',
        'M Dopa':           'Glut Sero Dopa',
        'CN CGE GABA':      'Subpallium GABA',
        'CN LGE GABA':      'Subpallium GABA',
        'CN MGE GABA':      'Subpallium GABA',
        'Cx GABA':          'Subpallium GABA',
        'F M GABA':         'Subpallium GABA',
        'CN GABA-Glut':     'Subpallium GABA-Glut',
        'CN GABA AND Glut': 'Subpallium GABA-Glut',
        'Astro-Epen':       'Nonneuron',
        'Astro AND Epen':   'Nonneuron',
        'Immune':           'Nonneuron',
        'OPC-Oligo':        'Nonneuron',
        'OPC AND Oligo':    'Nonneuron',
        'Vascular':         'Nonneuron',
    }
    def _norm_class(s):
        return s.replace('_', ' ').replace('-', ' ').replace('  ', ' ').strip()
    class2nbr = {_norm_class(k): v for k, v in _class2nbr_raw.items()}
    _class_nbr = lambda g: class2nbr.get(_norm_class(g), 'Nonneuron')

    nbr_order = ['Glut Sero Dopa', 'Subpallium GABA', 'Subpallium GABA-Glut', 'Nonneuron']
    nbr_palette = {'Glut Sero Dopa':       '#d73027',
                   'Subpallium GABA':      '#1f78b4',
                   'Subpallium GABA-Glut': '#6a3d9a',
                   'Nonneuron':            '#33a02c'}

    groups_order = sorted(
        lineage_switches.keys(),
        key=lambda g: (nbr_order.index(_class_nbr(g)), g))

    # ---- canonical BG markers ----
    markers_hg38 = {
        'DRD1':     ('chr5',  175_440_498),
        'DRD2':     ('chr11', 113_409_615),
        'PPP1R1B':  ('chr17',  39_624_000),
        'CHAT':     ('chr10',  49_611_277),
        'SLC17A7':  ('chr19',  49_437_961),
        'FOXP2':    ('chr7',  114_086_310),
        'GAD1':     ('chr2',  170_813_019),
        'GAD2':     ('chr10',  26_216_372),
        'LHX6':     ('chr9',  122_994_000),
        'MEIS2':    ('chr15',  36_889_136),
        'TH':       ('chr11',   2_166_920),
        'SLC6A3':   ('chr5',    1_392_794),
        'OLIG2':    ('chr21',  33_031_947),
        'GFAP':     ('chr17',  44_903_058),
        'MBP':      ('chr18',  76_978_827),
        'AQP4':     ('chr18',  26_852_582),
    }
    marker_x = {g: offsets[c] + bp
                for g, (c, bp) in markers_hg38.items() if c in offsets}
    marker_lineage = {
        'DRD1': 'Subpallium GABA', 'DRD2': 'Subpallium GABA',
        'PPP1R1B': 'Subpallium GABA', 'CHAT': 'Subpallium GABA',
        'GAD1': 'Subpallium GABA', 'GAD2': 'Subpallium GABA',
        'LHX6': 'Subpallium GABA', 'MEIS2': 'Subpallium GABA',
        'SLC17A7': 'Glut Sero Dopa', 'FOXP2': 'Glut Sero Dopa',
        'TH': 'Glut Sero Dopa', 'SLC6A3': 'Glut Sero Dopa',
        'OLIG2': 'Nonneuron', 'GFAP': 'Nonneuron',
        'MBP': 'Nonneuron', 'AQP4': 'Nonneuron',
    }
    print(f'{len(marker_x)} canonical marker genes mapped onto the genome axis.')

    # ---- bin -> protein-coding gene lookup ----
    if 'gene_bed' not in globals():
        gene_bed = pd.read_csv(os.path.expanduser(gene_annotation), sep='\t', header=None,
                               names=['chrom', 'start', 'end', 'ensembl', 'symbol', 'strand', 'biotype'])
    gb = gene_bed.copy()
    if 'biotype' in gb.columns:
        gb = gb[gb.biotype.astype(str).str.contains('protein_coding', na=False)]
    gb = gb[gb.chrom.isin(offsets)].copy()
    gb['bin'] = (gb.start // BIN).astype(int)
    bin2gene = {}
    for chrom, sub in gb.groupby('chrom'):
        for b, sub2 in sub.groupby('bin'):
            bin2gene[(chrom, int(b))] = sub2.iloc[0]['symbol']
    def _lookup_gene(chrom, start):
        b = int(start // BIN)
        for db in (0, 1, -1):
            gn = bin2gene.get((chrom, b + db), '')
            if gn:
                return gn
        return ''

    # ---- per-row z-score on pc1_diff ----
    def _zscore(arr):
        s = pd.Series(arr, dtype=float)
        sd = s.std(ddof=0)
        if sd == 0 or not np.isfinite(sd):
            return s * 0
        return (s - s.mean()) / sd

    row_data = {}
    for g in groups_order:
        df = lineage_switches[g].copy()
        df = df.loc[df.chrom.isin(offsets)].copy()
        df['x'] = df.start + df.chrom.map(offsets)
        df['z'] = _zscore(df['pc1_diff'].values).values
        row_data[g] = df.reset_index(drop=True)

    # ---- per-row top-K positive peaks (by z) + gene labels ----
    TOP_K_PEAKS_PER_ROW = 3
    MAX_SEARCH = 50
    peak_records = {}
    for g in groups_order:
        df = row_data[g].nlargest(MAX_SEARCH, 'z')
        items, seen_x = [], []
        for c, s, x, y in zip(df.chrom, df.start, df.x, df.z):
            gn = _lookup_gene(c, s)
            if not gn:
                continue
            if any(abs(x - xp) < 25_000_000 for xp in seen_x):
                continue
            items.append((float(x), float(y), gn))
            seen_x.append(x)
            if len(items) >= TOP_K_PEAKS_PER_ROW:
                break
        peak_records[g] = items

    # ---- select N canonical markers (N = #rows), one per row, unique ----
    def _marker_z_in_row(g, gene):
        chrom, bp = markers_hg38[gene]
        df = row_data[g]
        sub = df[df.chrom == chrom]
        if sub.empty:
            return -np.inf
        idx = (sub.start - bp).abs().idxmin()
        return float(sub.loc[idx, 'z'])

    row_marker_pref = {
        g: sorted(markers_hg38.keys(), key=lambda m: -_marker_z_in_row(g, m))
        for g in groups_order
    }
    assigned_marker = {}
    used = set()
    for g in groups_order:
        for m in row_marker_pref[g]:
            if m not in used:
                assigned_marker[g] = m
                used.add(m)
                break
    selected_markers = {assigned_marker[g]: marker_x[assigned_marker[g]]
                        for g in groups_order if g in assigned_marker}

    # ---- plot: dedicated top axis for marker labels + n_rows data rows ----
    n_rows = len(groups_order)
    fig = plt.figure(figsize=(12, 1.05 * n_rows + 2.0), dpi=200)
    gs = fig.add_gridspec(n_rows + 1, 1,
                          height_ratios=[1.2] + [1.0] * n_rows,
                          hspace=0.35)
    ax_top = fig.add_subplot(gs[0])
    axes = [fig.add_subplot(gs[i + 1], sharex=ax_top) for i in range(n_rows)]
    # cosmetics for the dedicated top label axis (no spines, no ticks)
    ax_top.set_ylim(0, 1)
    for sp in ('top', 'bottom', 'left', 'right'):
        ax_top.spines[sp].set_visible(False)
    ax_top.tick_params(axis='both', length=0, labelbottom=False, labelleft=False)
    ax_top.set_xticks([]); ax_top.set_yticks([])

    for ax, g in zip(axes, groups_order):
        nbr_c = nbr_palette.get(_class_nbr(g), 'black')
        df = row_data[g]
        pos = df['z'] > 0
        ax.scatter(df.loc[pos, 'x'],  df.loc[pos, 'z'],
                   s=1, c='#d73027', alpha=0.5, rasterized=True)
        ax.scatter(df.loc[~pos, 'x'], df.loc[~pos, 'z'],
                   s=1, c='#4575b4', alpha=0.5, rasterized=True)
        for xb in chrom_bounds:
            ax.axvline(xb, color='gray', lw=0.3, ls='--', alpha=0.5)
        for gn_sel, x_sel in selected_markers.items():
            ax.axvline(x_sel, color='black', lw=0.3, alpha=0.35, zorder=0)
        # top-K positive peak gene labels — text only floats upward
        peaks_g = peak_records[g]
        texts, text_init_y = [], []
        for x, y, gn in peaks_g:
            ax.scatter([x], [y], s=22, facecolors='none', edgecolors='black',
                       lw=0.6, zorder=4)
            t = ax.text(x, y, gn, fontsize=11, color='black',
                        fontweight='bold', ha='center', va='bottom', zorder=5)
            texts.append(t)
            text_init_y.append(float(y))
        if texts:
            adjust_text(texts, ax=ax,
                        only_move={'text': 'y', 'static': 'y', 'explode': 'y'},
                        expand_text=(1.1, 1.4), expand_points=(1.2, 1.4))
            for t, y0 in zip(texts, text_init_y):
                tx, ty = t.get_position()
                if ty < y0:
                    t.set_position((tx, y0))
            for t, (px, py, _) in zip(texts, peaks_g):
                tx, ty = t.get_position()
                if ty > py:
                    ax.plot([px, tx], [py, ty], color='gray', lw=0.4,
                            alpha=0.7, zorder=3)
        ax.set_ylabel(g, fontsize=8, rotation=0, ha='right', va='center',
                      color=nbr_c)
        ax.axhline(0, color='black', lw=0.3)
        # clean spines + put y-tick labels on the right
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.yaxis.tick_right()
        ax.tick_params(axis='y', length=2, labelsize=6,
                       left=False, right=True,
                       labelleft=False, labelright=True)
        ax.tick_params(axis='x', length=0)

    # bottom row keeps chromosome tick labels (no tick marks); hide on others
    xt = [offsets[c] + chrom_sizes[c] / 2 for c in chrom_order_main]
    axes[-1].set_xticks(xt)
    axes[-1].set_xticklabels([c.replace('chr', '') for c in chrom_order_main],
                             fontsize=6)
    axes[-1].tick_params(axis='x', length=0, labelbottom=True)
    for ax in axes[:-1]:
        ax.tick_params(axis='x', length=0, labelbottom=False)

    # ---- selected canonical marker labels in dedicated top axis ----
    marker_texts = []
    for gene, x in selected_markers.items():
        lin = marker_lineage.get(gene, 'black')
        col = nbr_palette.get(lin, 'black')
        marker_texts.append(
            ax_top.text(x, 0.02, gene, rotation=90, ha='center', va='bottom',
                        fontsize=9, color=col, clip_on=False, fontweight='bold'))
    if marker_texts:
        adjust_text(marker_texts, ax=ax_top,
                    only_move={'text': 'x', 'static': 'x', 'explode': 'x'},
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.3, alpha=0.6),
                    expand_text=(1.4, 1.0))

    fig.suptitle(
        f'{level}-level z(\u0394PC1) with {len(selected_markers)} class-specific marker loci '
        f'(top of panel) and top-{TOP_K_PEAKS_PER_ROW} positive-peak genes per row '
        f'(dcHiC, FDR-filtered, hg38)',
        fontsize=10, y=1.00)

    handles = [Patch(facecolor=nbr_palette[n], label=n)
               for n in nbr_palette
               if any(_class_nbr(g) == n for g in groups_order)]
    handles += [Line2D([], [], color='black', lw=0.5,
                       label='selected marker locus (1 per row)'),
                Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                       markeredgecolor='black', markersize=4,
                       label=f'top-{TOP_K_PEAKS_PER_ROW} positive peak (per row)')]
    fig.legend(handles=handles, loc='upper right',
               bbox_to_anchor=(0.99, 0.99), fontsize=7,
               frameon=False, title='Neighborhood / annotation',
               title_fontsize=7)

    fig.savefig(os.path.join(figdir,
                f'{level}.dchic_lineage_compartment_manhattan_markers.pdf'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.show()

    print('\nSelected canonical marker per row (1 per row, unique):')
    for g in groups_order:
        m = assigned_marker.get(g, '-')

        print(f'  {g:24s}  -> {m}')

In [ ]:
# restricted to the three striatal MSN subclasses (STR D1 MSN, STR D2 MSN,
# STR Hybrid MSN). Self-contained: loads Subclass-level dcHiC merged bedGraph
# directly so it works regardless of the current `level` parameter.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from adjustText import adjust_text

# ---- load Subclass-level dcHiC merged bedGraph ----
_subclass_merged = ('/home/x-wding2/Projects/BG/HIC/BG_Subclass/compartment/'
                    'SubclassDiffComp/merged/DifferentialResult/merged/'
                    'fdr_result/differential.intra_sample_combined.Filtered.pcQnm.bedGraph')
_df_sub = pd.read_csv(_subclass_merged, sep='\t')
_meta_cols = {'chr', 'start', 'end', 'sample_maha', 'pval', 'padj', 'dist_clust'}
_sub_cols = [c for c in _df_sub.columns
             if c not in _meta_cols and not c.endswith('_100Kb')
             and not c.startswith('sample')]
_vals = _df_sub[_sub_cols].astype(float)
_mean = _vals.mean(axis=1)
_diff = _vals.subtract(_mean, axis=0)

def _underscore_to_space(s):
    return s.replace('_', ' ')

sub_lineage_switches = {}
for _cl in _sub_cols:
    _df = _df_sub[['chr', 'start', 'end']].copy()
    _df['pc1_diff'] = _diff[_cl].values
    _df['bin_id']   = _df['chr'].astype(str) + ':' + _df['start'].astype(str)
    _df = _df.rename(columns={'chr': 'chrom'})
    sub_lineage_switches[_underscore_to_space(_cl)] = _df

print(f'[info] Subclass-level dcHiC loaded: {len(sub_lineage_switches)} subclasses, '
      f'{len(_df_sub)} bins.')

# ---- shared genome coordinate setup ----
chrom_order_main = [c for c in chrom_sizes.index]
offsets, cum = {}, 0
for c in chrom_order_main:
    offsets[c] = cum
    cum += chrom_sizes[c]
BIN = 100_000
chrom_bounds = [offsets[c] + chrom_sizes[c] for c in chrom_order_main[:-1]]

TOP_K_PEAKS_PER_ROW = 3
MAX_SEARCH = 50

# ---- canonical BG markers ----
markers_hg38 = {
    'DRD1':     ('chr5',  175_440_498),
    'DRD2':     ('chr11', 113_409_615),
    'PPP1R1B':  ('chr17',  39_624_000),
    'CHAT':     ('chr10',  49_611_277),
    'SLC17A7':  ('chr19',  49_437_961),
    'FOXP2':    ('chr7',  114_086_310),
    'GAD1':     ('chr2',  170_813_019),
    'GAD2':     ('chr10',  26_216_372),
    'LHX6':     ('chr9',  122_994_000),
    'MEIS2':    ('chr15',  36_889_136),
    'TH':       ('chr11',   2_166_920),
    'SLC6A3':   ('chr5',    1_392_794),
    'OLIG2':    ('chr21',  33_031_947),
    'GFAP':     ('chr17',  44_903_058),
    'MBP':      ('chr18',  76_978_827),
    'AQP4':     ('chr18',  26_852_582),
}
marker_x = {g: offsets[c] + bp
            for g, (c, bp) in markers_hg38.items() if c in offsets}
marker_lineage = {
    'DRD1': 'Subpallium GABA', 'DRD2': 'Subpallium GABA',
    'PPP1R1B': 'Subpallium GABA', 'CHAT': 'Subpallium GABA',
    'GAD1': 'Subpallium GABA', 'GAD2': 'Subpallium GABA',
    'LHX6': 'Subpallium GABA', 'MEIS2': 'Subpallium GABA',
    'SLC17A7': 'Glut Sero Dopa', 'FOXP2': 'Glut Sero Dopa',
    'TH': 'Glut Sero Dopa', 'SLC6A3': 'Glut Sero Dopa',
    'OLIG2': 'Nonneuron', 'GFAP': 'Nonneuron',
    'MBP': 'Nonneuron', 'AQP4': 'Nonneuron',
}
nbr_palette = {'Glut Sero Dopa':       '#d73027',
               'Subpallium GABA':      '#1f78b4',
               'Subpallium GABA-Glut': '#6a3d9a',
               'Nonneuron':            '#33a02c'}

# ---- bin -> protein-coding gene lookup ----
if 'gene_bed' not in globals():
    gene_bed = pd.read_csv(os.path.expanduser(gene_annotation), sep='\t', header=None,
                           names=['chrom', 'start', 'end', 'ensembl', 'symbol', 'strand', 'biotype'])
gb = gene_bed.copy()
if 'biotype' in gb.columns:
    gb = gb[gb.biotype.astype(str).str.contains('protein_coding', na=False)]
gb = gb[gb.chrom.isin(offsets)].copy()
gb['bin'] = (gb.start // BIN).astype(int)
bin2gene = {}
for chrom, sub in gb.groupby('chrom'):
    for b, sub2 in sub.groupby('bin'):
        bin2gene[(chrom, int(b))] = sub2.iloc[0]['symbol']
def _lookup_gene(chrom, start):
    b = int(start // BIN)
    for db in (0, 1, -1):
        gn = bin2gene.get((chrom, b + db), '')
        if gn:
            return gn
    return ''

# ---- pick the three MSN subclasses ----
msn_targets = ['STR D1 MSN', 'STR D2 MSN', 'STR Hybrid MSN']
msn_groups = [g for g in msn_targets if g in sub_lineage_switches]
missing = [g for g in msn_targets if g not in sub_lineage_switches]
if missing:
    print(f'[warn] not found in Subclass-level dcHiC: {missing}')
    print(f'[info] available subclasses: {sorted(sub_lineage_switches.keys())}')

if not msn_groups:
    print('[skip] none of the MSN subclasses are present in Subclass-level dcHiC.')
else:
    msn_palette = {
        'STR D1 MSN':     '#e41a1c',
        'STR D2 MSN':     '#377eb8',
        'STR Hybrid MSN': '#984ea3',
    }

    # ---- per-row z-score ----
    def _zscore(arr):
        s = pd.Series(arr, dtype=float)
        sd = s.std(ddof=0)
        if sd == 0 or not np.isfinite(sd):
            return s * 0
        return (s - s.mean()) / sd

    row_data = {}
    for g in msn_groups:
        df = sub_lineage_switches[g].copy()
        df = df.loc[df.chrom.isin(offsets)].copy()
        df['x'] = df.start + df.chrom.map(offsets)
        df['z'] = _zscore(df['pc1_diff'].values).values
        row_data[g] = df.reset_index(drop=True)

    # per-row top-K positive peak gene records (by z)
    msn_peak_records = {}
    for g in msn_groups:
        df = row_data[g].nlargest(MAX_SEARCH, 'z')
        items, seen_x = [], []
        for c, s, x, y in zip(df.chrom, df.start, df.x, df.z):
            gn = _lookup_gene(c, s)
            if not gn:
                continue
            if any(abs(x - xp) < 25_000_000 for xp in seen_x):
                continue
            items.append((float(x), float(y), gn))
            seen_x.append(x)
            if len(items) >= TOP_K_PEAKS_PER_ROW:
                break
        msn_peak_records[g] = items

    # ---- select N canonical markers (N = #rows), one per row, unique ----
    def _marker_z_in_row(g, gene):
        chrom, bp = markers_hg38[gene]
        df = row_data[g]
        sub = df[df.chrom == chrom]
        if sub.empty:
            return -np.inf
        idx = (sub.start - bp).abs().idxmin()
        return float(sub.loc[idx, 'z'])

    row_marker_pref = {
        g: sorted(markers_hg38.keys(), key=lambda m: -_marker_z_in_row(g, m))
        for g in msn_groups
    }
    assigned_marker = {}
    used = set()
    for g in msn_groups:
        for m in row_marker_pref[g]:
            if m not in used:
                assigned_marker[g] = m
                used.add(m)
                break
    selected_markers = {assigned_marker[g]: marker_x[assigned_marker[g]]
                        for g in msn_groups if g in assigned_marker}

    # ---- plot: dedicated top axis for marker labels + n_rows data rows ----
    n_rows = len(msn_groups)
    fig = plt.figure(figsize=(12, 1.1 * n_rows + 2.0), dpi=200)
    gs = fig.add_gridspec(n_rows + 1, 1,
                          height_ratios=[1.2] + [1.0] * n_rows,
                          hspace=0.35)
    ax_top = fig.add_subplot(gs[0])
    axes = [fig.add_subplot(gs[i + 1], sharex=ax_top) for i in range(n_rows)]
    ax_top.set_ylim(0, 1)
    for sp in ('top', 'bottom', 'left', 'right'):
        ax_top.spines[sp].set_visible(False)
    ax_top.tick_params(axis='both', length=0, labelbottom=False, labelleft=False)
    ax_top.set_xticks([]); ax_top.set_yticks([])

    for ax, g in zip(axes, msn_groups):
        nbr_c = msn_palette.get(g, 'black')
        df = row_data[g]
        pos = df['z'] > 0
        ax.scatter(df.loc[pos, 'x'],  df.loc[pos, 'z'],
                   s=1, c='#d73027', alpha=0.5, rasterized=True)
        ax.scatter(df.loc[~pos, 'x'], df.loc[~pos, 'z'],
                   s=1, c='#4575b4', alpha=0.5, rasterized=True)
        for xb in chrom_bounds:
            ax.axvline(xb, color='gray', lw=0.3, ls='--', alpha=0.5)
        for gn_sel, x_sel in selected_markers.items():
            ax.axvline(x_sel, color='black', lw=0.3, alpha=0.35, zorder=0)
        # top-K positive peak gene labels — text only floats upward
        peaks_g = msn_peak_records[g]
        texts, text_init_y = [], []
        for x, y, gn in peaks_g:
            ax.scatter([x], [y], s=22, facecolors='none', edgecolors='black',
                       lw=0.6, zorder=4)
            t = ax.text(x, y, gn, fontsize=11, color='black',
                        fontweight='bold', ha='center', va='bottom', zorder=5)
            texts.append(t)
            text_init_y.append(float(y))
        if texts:
            adjust_text(texts, ax=ax,
                        only_move={'text': 'y', 'static': 'y', 'explode': 'y'},
                        expand_text=(1.1, 1.4), expand_points=(1.2, 1.4))
            for t, y0 in zip(texts, text_init_y):
                tx, ty = t.get_position()
                if ty < y0:
                    t.set_position((tx, y0))
            for t, (px, py, _) in zip(texts, peaks_g):
                tx, ty = t.get_position()
                if ty > py:
                    ax.plot([px, tx], [py, ty], color='gray', lw=0.4,
                            alpha=0.7, zorder=3)
        ax.set_ylabel(g, fontsize=8, rotation=0, ha='right', va='center',
                      color=nbr_c)
        ax.axhline(0, color='black', lw=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.yaxis.tick_right()
        ax.tick_params(axis='y', length=2, labelsize=6,
                       left=False, right=True,
                       labelleft=False, labelright=True)
        ax.tick_params(axis='x', length=0)

    # bottom row keeps chromosome tick labels (no tick marks); hide on others
    xt = [offsets[c] + chrom_sizes[c] / 2 for c in chrom_order_main]
    axes[-1].set_xticks(xt)
    axes[-1].set_xticklabels([c.replace('chr', '') for c in chrom_order_main],
                             fontsize=6)
    axes[-1].tick_params(axis='x', length=0, labelbottom=True)
    for ax in axes[:-1]:
        ax.tick_params(axis='x', length=0, labelbottom=False)

    # ---- selected canonical marker labels in dedicated top axis ----
    marker_texts = []
    for gene, x in selected_markers.items():
        lin = marker_lineage.get(gene, 'black')
        col = nbr_palette.get(lin, 'black')
        marker_texts.append(
            ax_top.text(x, 0.02, gene, rotation=90, ha='center', va='bottom',
                        fontsize=9, color=col, clip_on=False, fontweight='bold'))
    if marker_texts:
        adjust_text(marker_texts, ax=ax_top,
                    only_move={'text': 'x', 'static': 'x', 'explode': 'x'},
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.3, alpha=0.6),
                    expand_text=(1.4, 1.0))

    fig.suptitle(
        f'Subclass-level z(\u0394PC1) for striatal MSN subclasses '
        f'({len(selected_markers)} subclass-specific marker loci, '
        f'top-{TOP_K_PEAKS_PER_ROW} positive-peak genes per row, '
        f'dcHiC, FDR-filtered, hg38)',
        fontsize=10, y=1.00)

    handles = [Patch(facecolor=msn_palette[g], label=g) for g in msn_groups]
    handles += [Line2D([], [], color='black', lw=0.5,
                       label='selected marker locus (1 per row)'),
                Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                       markeredgecolor='black', markersize=4,
                       label=f'top-{TOP_K_PEAKS_PER_ROW} positive peak (per row)')]
    fig.legend(handles=handles, loc='upper right',
               bbox_to_anchor=(0.99, 0.99), fontsize=7,
               frameon=False, title='MSN subclass / annotation',
               title_fontsize=7)

    fig.savefig(os.path.join(figdir,
                'Subclass.dchic_lineage_compartment_manhattan_markers.MSN.pdf'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.show()

    print('\nSelected canonical marker per row (1 per row, unique):')
    for g in msn_groups:
        m = assigned_marker.get(g, '-')
        print(f'  {g:18s}  -> {m}')
    print('\nTop-positive peak genes per MSN subclass (by per-row z):')
    for g in msn_groups:
        items = msn_peak_records[g]
        if not items:
            print(f'  {g:18s}  -> (no gene at top bins)')
            continue
        parts = [f'{gn} (z={y:+.2f})' for _, y, gn in items]

        print(f'  {g:18s}  -> ' + '; '.join(parts))

## Manhattan plots for differential loops and differential domain boundaries

Same per-row z-score scatter layout as the dcHiC compartment Manhattan above, applied to
- **Differential chromatin loops** (`diff_loops.tsv`) — y = z(log2((Q.F+1)/(T.F+1))) per group, one point per DiffLoop (midpoint between the two anchors).
- **Differential domain boundaries** (`diff_boundary.tsv`) — y = z(`insfc`) per group, one point per significant boundary.

Both run at Class level (11 rows, all classes) and at Subclass level restricted to the three striatal MSN subclasses (STR D1 / D2 / Hybrid MSN).


In [ ]:
# ---- shared helper for per-row Manhattan of sparse features (loops / boundaries) ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from adjustText import adjust_text

# ---- shared genome coords ----
chrom_order_main = [c for c in chrom_sizes.index]
offsets, _cum = {}, 0
for _c in chrom_order_main:
    offsets[_c] = _cum
    _cum += chrom_sizes[_c]
chrom_bounds = [offsets[c] + chrom_sizes[c] for c in chrom_order_main[:-1]]

# ---- canonical BG markers (same set as compartment Manhattan) ----
markers_hg38 = {
    'DRD1':     ('chr5',  175_440_498),
    'DRD2':     ('chr11', 113_409_615),
    'PPP1R1B':  ('chr17',  39_624_000),
    'CHAT':     ('chr10',  49_611_277),
    'SLC17A7':  ('chr19',  49_437_961),
    'FOXP2':    ('chr7',  114_086_310),
    'GAD1':     ('chr2',  170_813_019),
    'GAD2':     ('chr10',  26_216_372),
    'LHX6':     ('chr9',  122_994_000),
    'MEIS2':    ('chr15',  36_889_136),
    'TH':       ('chr11',   2_166_920),
    'SLC6A3':   ('chr5',    1_392_794),
    'OLIG2':    ('chr21',  33_031_947),
    'GFAP':     ('chr17',  44_903_058),
    'MBP':      ('chr18',  76_978_827),
    'AQP4':     ('chr18',  26_852_582),
}
marker_x = {g: offsets[c] + bp
            for g, (c, bp) in markers_hg38.items() if c in offsets}
marker_lineage = {
    'DRD1': 'Subpallium GABA', 'DRD2': 'Subpallium GABA',
    'PPP1R1B': 'Subpallium GABA', 'CHAT': 'Subpallium GABA',
    'GAD1': 'Subpallium GABA', 'GAD2': 'Subpallium GABA',
    'LHX6': 'Subpallium GABA', 'MEIS2': 'Subpallium GABA',
    'SLC17A7': 'Glut Sero Dopa', 'FOXP2': 'Glut Sero Dopa',
    'TH': 'Glut Sero Dopa', 'SLC6A3': 'Glut Sero Dopa',
    'OLIG2': 'Nonneuron', 'GFAP': 'Nonneuron',
    'MBP': 'Nonneuron', 'AQP4': 'Nonneuron',
}
nbr_palette = {'Glut Sero Dopa':       '#d73027',
               'Subpallium GABA':      '#1f78b4',
               'Subpallium GABA-Glut': '#6a3d9a',
               'Nonneuron':            '#33a02c'}
nbr_order = ['Glut Sero Dopa', 'Subpallium GABA', 'Subpallium GABA-Glut', 'Nonneuron']


def _zscore(arr):
    s = pd.Series(arr, dtype=float)
    sd = s.std(ddof=0)
    if sd == 0 or not np.isfinite(sd):
        return s * 0
    return (s - s.mean()) / sd


def _clean_gene(g):
    """Return '' for missing/NaN-like gene strings."""
    if g is None:
        return ''
    s = str(g).strip()
    if s.lower() in {'nan', 'na', 'none', 'null', '.', ''}:
        return ''
    return s


def _lookup_bin_gene(chrom, pos, max_radius=10):
    """Look up a gene near (chrom, pos) using the global `bin2gene` dict
    (100 kb resolution). Searches outward up to ``max_radius`` bins
    (default ~±1 Mb). Returns '' if not found."""
    if 'bin2gene' not in globals():
        return ''
    b = int(pos) // 100_000
    # search outward: 0, -1, +1, -2, +2, ..., -max_radius, +max_radius
    yield_seq = [0]
    for r in range(1, max_radius + 1):
        yield_seq.extend([-r, r])
    for off in yield_seq:
        gn = bin2gene.get((chrom, (b + off) * 100_000))
        if gn:
            return gn
    return ''


def fill_gene_column(df, chrom_col='chrom', pos_col='x_local', gene_col='gene'):
    """In-place: replace empty/NaN gene labels with bin2gene fallback."""
    if gene_col not in df.columns:
        df[gene_col] = ''
    df[gene_col] = df[gene_col].map(_clean_gene)
    mask = df[gene_col] == ''
    if mask.any():
        df.loc[mask, gene_col] = [_lookup_bin_gene(c, p)
                                   for c, p in zip(df.loc[mask, chrom_col],
                                                   df.loc[mask, pos_col])]
    return df


def _render_peak_labels(ax, peaks):
    """Place gene labels above peak points and let them only float upward.

    `peaks` : iterable of (x, y, gene_name). All labels rendered in black.
    A thin gray connector is drawn from peak to its final label position
    when the label is above the peak.
    """
    if not peaks:
        return
    texts, text_init_y = [], []
    for x, y, gn in peaks:
        ax.scatter([x], [y], s=22, facecolors='none', edgecolors='black',
                   lw=0.6, zorder=4)
        t = ax.text(x, y, gn, fontsize=11, color='black',
                    fontweight='bold', ha='center', va='bottom', zorder=5)
        texts.append(t)
        text_init_y.append(float(y))
    adjust_text(texts, ax=ax,
                only_move={'text': 'y', 'static': 'y', 'explode': 'y'},
                expand_text=(1.1, 1.4), expand_points=(1.2, 1.4))
    # clamp downward movement so labels only float upward
    for t, y0 in zip(texts, text_init_y):
        tx, ty = t.get_position()
        if ty < y0:
            t.set_position((tx, y0))
    # draw connectors from peak (x, y) to final label position
    for t, (px, py, _) in zip(texts, peaks):
        tx, ty = t.get_position()
        if ty > py:
            ax.plot([px, tx], [py, ty], color='gray', lw=0.4, alpha=0.7,
                    zorder=3)


def _style_row_axis(ax, row_label, row_color):
    """Common per-row axis styling: y-tick labels on the right, no top/bottom spines."""
    ax.set_ylabel(row_label, fontsize=8, rotation=0, ha='right', va='center',
                  color=row_color)
    ax.axhline(0, color='black', lw=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.yaxis.tick_right()
    ax.tick_params(axis='y', length=2, labelsize=6,
                   left=False, right=True,
                   labelleft=False, labelright=True)
    ax.tick_params(axis='x', length=0)


def plot_feature_manhattan(row_records, groups_order, row_color,
                           selected_markers, save_path, title,
                           top_k=3, peak_min_sep=5_000_000,
                           legend_handles_extra=None):
    """Per-row Manhattan plot for sparse genomic features.

    row_records : dict[str, pd.DataFrame]
        group -> DataFrame with required columns: chrom, x (genome coord), z, gene.
    """
    n_rows = len(groups_order)
    fig = plt.figure(figsize=(12, 1.05 * n_rows + 2.0), dpi=200)
    gs = fig.add_gridspec(n_rows + 1, 1,
                          height_ratios=[1.2] + [1.0] * n_rows,
                          hspace=0.35)
    ax_top = fig.add_subplot(gs[0])
    axes = [fig.add_subplot(gs[i + 1], sharex=ax_top) for i in range(n_rows)]
    ax_top.set_ylim(0, 1)
    for sp in ('top', 'bottom', 'left', 'right'):
        ax_top.spines[sp].set_visible(False)
    ax_top.tick_params(axis='both', length=0, labelbottom=False, labelleft=False)
    ax_top.set_xticks([]); ax_top.set_yticks([])

    peak_records = {}
    for g in groups_order:
        df = row_records[g]
        if df.empty:
            peak_records[g] = []
            continue
        df_sorted = df.sort_values('z', ascending=False)
        items, seen_x = [], []
        for _, r in df_sorted.iterrows():
            z = float(r['z'])
            if z <= 0:
                break  # df is sorted desc; once non-positive, no more positives
            gn = _clean_gene(r['gene'])
            if not gn:
                continue
            x = float(r['x'])
            if any(abs(x - xp) < peak_min_sep for xp in seen_x):
                continue
            items.append((x, z, gn))
            seen_x.append(x)
            if len(items) >= top_k:
                break
        peak_records[g] = items

    for ax, g in zip(axes, groups_order):
        col = row_color(g)
        df = row_records[g]
        if not df.empty:
            pos = df['z'] > 0
            ax.scatter(df.loc[pos, 'x'],  df.loc[pos, 'z'],
                       s=4, c='#d73027', alpha=0.6, rasterized=True, linewidths=0)
            ax.scatter(df.loc[~pos, 'x'], df.loc[~pos, 'z'],
                       s=4, c='#4575b4', alpha=0.6, rasterized=True, linewidths=0)
        for xb in chrom_bounds:
            ax.axvline(xb, color='gray', lw=0.3, ls='--', alpha=0.5)
        for _, x_sel in selected_markers.items():
            ax.axvline(x_sel, color='black', lw=0.3, alpha=0.35, zorder=0)
        _render_peak_labels(ax, peak_records[g])
        _style_row_axis(ax, g, col)

    xt = [offsets[c] + chrom_sizes[c] / 2 for c in chrom_order_main]
    axes[-1].set_xticks(xt)
    axes[-1].set_xticklabels([c.replace('chr', '') for c in chrom_order_main],
                             fontsize=6)
    axes[-1].tick_params(axis='x', length=0, labelbottom=True)
    for ax in axes[:-1]:
        ax.tick_params(axis='x', length=0, labelbottom=False)

    marker_texts = []
    for gene, x in selected_markers.items():
        lin = marker_lineage.get(gene, 'black')
        mc = nbr_palette.get(lin, 'black')
        marker_texts.append(
            ax_top.text(x, 0.02, gene, rotation=90, ha='center', va='bottom',
                        fontsize=9, color=mc, clip_on=False, fontweight='bold'))
    if marker_texts:
        adjust_text(marker_texts, ax=ax_top,
                    only_move={'text': 'x', 'static': 'x', 'explode': 'x'},
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.3, alpha=0.6),
                    expand_text=(1.4, 1.0))

    fig.suptitle(title, fontsize=10, y=1.00)
    if legend_handles_extra is not None:
        fig.legend(handles=legend_handles_extra, loc='upper right',
                   bbox_to_anchor=(0.99, 0.99), fontsize=7,
                   frameon=False, title_fontsize=7)
    if save_path:
        fig.savefig(save_path, bbox_inches='tight', transparent=True, dpi=300)
    plt.show()
    return peak_records


# Class -> Neighborhood mapping (matches compartment plot)
_class2nbr_raw = {
    'F M Glut':         'Glut Sero Dopa',
    'M Dopa':           'Glut Sero Dopa',
    'CN CGE GABA':      'Subpallium GABA',
    'CN LGE GABA':      'Subpallium GABA',
    'CN MGE GABA':      'Subpallium GABA',
    'Cx GABA':          'Subpallium GABA',
    'F M GABA':         'Subpallium GABA',
    'CN GABA-Glut':     'Subpallium GABA-Glut',
    'CN GABA AND Glut': 'Subpallium GABA-Glut',
    'Astro-Epen':       'Nonneuron',
    'Astro AND Epen':   'Nonneuron',
    'Immune':           'Nonneuron',
    'OPC-Oligo':        'Nonneuron',
    'OPC AND Oligo':    'Nonneuron',
    'Vascular':         'Nonneuron',
}
def _norm_class(s):
    return s.replace('_', ' ').replace('-', ' ').replace('  ', ' ').strip()
class2nbr = {_norm_class(k): v for k, v in _class2nbr_raw.items()}
def class_nbr(g):
    return class2nbr.get(_norm_class(g), 'Nonneuron')

msn_palette = {
    'STR D1 MSN':     '#e41a1c',
    'STR D2 MSN':     '#377eb8',
    'STR Hybrid MSN': '#984ea3',
}

print('[ok] feature-Manhattan helper defined.')


In [ ]:
# ---- Class-level differential loop Manhattan ----
_class_loop_tsv = '/home/x-wding2/Projects/BG/HIC/BG_Class/loop/Class/diff_loops.tsv'
_dl = pd.read_csv(_class_loop_tsv, sep='\t', index_col=0)
_dl = _dl[_dl['BalancedGroup'] == 'DiffLoop'].copy()
_dl = _dl[_dl['chr1'].isin(offsets)].copy()
_dl['mid'] = ((_dl['start1'] + _dl['end1']) // 2 +
              (_dl['start2'] + _dl['end2']) // 2) // 2
_dl['x']   = _dl['mid'] + _dl['chr1'].map(offsets)
# signed score: log2((Q+1)/(T+1)); positive = stronger loop in this Class
_dl['score'] = np.log2((_dl['Class.Q.F'].astype(float) + 1.0) /
                       (_dl['Class.T.F'].astype(float) + 1.0))
_dl = _dl.rename(columns={'chr1': 'chrom', 'Gene': 'gene'})
fill_gene_column(_dl, chrom_col='chrom', pos_col='mid', gene_col='gene')

# Build groups_order sorted by Neighborhood
_class_loop_groups = sorted(_dl['Group'].dropna().unique().tolist(),
                            key=lambda g: (nbr_order.index(class_nbr(g)), g))

# per-row z-score and assemble row_records
_loop_row_data = {}
for g in _class_loop_groups:
    sub = _dl[_dl['Group'] == g].copy()
    sub['z'] = _zscore(sub['score'].values).values
    _loop_row_data[g] = sub.reset_index(drop=True)[['chrom', 'x', 'z', 'gene']]

# per-row canonical marker pick (1 per row, unique across rows)
def _marker_z_in_row_loop(g, gene):
    chrom, bp = markers_hg38[gene]
    df = _loop_row_data[g]
    sub = df[df.chrom == chrom]
    if sub.empty:
        return -np.inf
    idx = (sub['x'] - (offsets[chrom] + bp)).abs().idxmin()
    return float(sub.loc[idx, 'z'])

row_marker_pref = {g: sorted(markers_hg38.keys(),
                             key=lambda m: -_marker_z_in_row_loop(g, m))
                   for g in _class_loop_groups}
assigned_marker, used = {}, set()
for g in _class_loop_groups:
    for m in row_marker_pref[g]:
        if m not in used:
            assigned_marker[g] = m
            used.add(m)
            break
selected_markers = {assigned_marker[g]: marker_x[assigned_marker[g]]
                    for g in _class_loop_groups if g in assigned_marker}

_legend = [Patch(facecolor=nbr_palette[n], label=n) for n in nbr_palette
           if any(class_nbr(g) == n for g in _class_loop_groups)]
_legend += [Line2D([], [], color='black', lw=0.5,
                   label='selected marker locus (1 per row)'),
            Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                   markeredgecolor='black', markersize=4,
                   label='top-3 positive peak (per row)')]

_peaks = plot_feature_manhattan(
    row_records=_loop_row_data,
    groups_order=_class_loop_groups,
    row_color=lambda g: nbr_palette.get(class_nbr(g), 'black'),
    selected_markers=selected_markers,
    save_path=os.path.join(figdir, 'Class.diff_loop_manhattan_markers.pdf'),
    title=('Class-level z(log2((Q+1)/(T+1))) of differential loops '
           f'({len(selected_markers)} class-specific marker loci, top-3 positive '
           'per row; loop midpoint)'),
    legend_handles_extra=_legend,
)

print('\nSelected canonical marker per row (1 per row, unique):')
for g in _class_loop_groups:
    print(f'  {g:24s}  -> {assigned_marker.get(g, "-")}')
print('\nTop-positive diff-loop genes per Class (by per-row z):')
for g in _class_loop_groups:
    items = _peaks[g]
    parts = [f'{gn} (z={y:+.2f})' for _, y, gn in items] or ['(none)']
    print(f'  {g:24s}  -> ' + '; '.join(parts))


In [ ]:
# ---- MSN Subclass-level differential loop Manhattan (STR D1/D2/Hybrid only) ----
_subclass_loop_tsv = '/home/x-wding2/Projects/BG/HIC/BG_Subclass/loop/Subclass/diff_loops.tsv'
_msn_targets = ['STR D1 MSN', 'STR D2 MSN', 'STR Hybrid MSN']

_dl = pd.read_csv(_subclass_loop_tsv, sep='\t', index_col=0)
_dl = _dl[_dl['BalancedGroup'] == 'DiffLoop'].copy()
_dl = _dl[_dl['Group'].isin(_msn_targets)].copy()
_dl = _dl[_dl['chr1'].isin(offsets)].copy()
_dl['mid'] = ((_dl['start1'] + _dl['end1']) // 2 +
              (_dl['start2'] + _dl['end2']) // 2) // 2
_dl['x'] = _dl['mid'] + _dl['chr1'].map(offsets)
_dl['score'] = np.log2((_dl['Subclass.Q.F'].astype(float) + 1.0) /
                       (_dl['Subclass.T.F'].astype(float) + 1.0))
_dl = _dl.rename(columns={'chr1': 'chrom', 'Gene': 'gene'})
fill_gene_column(_dl, chrom_col='chrom', pos_col='mid', gene_col='gene')

_msn_row_data = {}
for g in _msn_targets:
    sub = _dl[_dl['Group'] == g].copy()
    sub['z'] = _zscore(sub['score'].values).values
    _msn_row_data[g] = sub.reset_index(drop=True)[['chrom', 'x', 'z', 'gene']]

# canonical markers for MSN — focus on striatal panel
_msn_markers = ['DRD1', 'DRD2', 'PPP1R1B', 'FOXP2', 'MEIS2', 'TH', 'GAD1', 'GAD2']
def _marker_z_in_row_msn(g, gene):
    chrom, bp = markers_hg38[gene]
    df = _msn_row_data[g]
    sub = df[df.chrom == chrom]
    if sub.empty:
        return -np.inf
    idx = (sub['x'] - (offsets[chrom] + bp)).abs().idxmin()
    return float(sub.loc[idx, 'z'])

row_pref = {g: sorted(_msn_markers, key=lambda m: -_marker_z_in_row_msn(g, m))
            for g in _msn_targets}
assigned, used = {}, set()
for g in _msn_targets:
    for m in row_pref[g]:
        if m not in used:
            assigned[g] = m
            used.add(m)
            break
selected_markers = {assigned[g]: marker_x[assigned[g]] for g in _msn_targets
                    if g in assigned}

_legend = [Patch(facecolor=msn_palette[g], label=g) for g in _msn_targets]
_legend += [Line2D([], [], color='black', lw=0.5,
                   label='selected marker locus (1 per row)'),
            Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                   markeredgecolor='black', markersize=4,
                   label='top-3 positive peak (per row)')]

_peaks = plot_feature_manhattan(
    row_records=_msn_row_data,
    groups_order=_msn_targets,
    row_color=lambda g: msn_palette[g],
    selected_markers=selected_markers,
    save_path=os.path.join(figdir, 'Subclass.diff_loop_manhattan_markers.MSN.pdf'),
    title=('MSN Subclass-level z(log2((Q+1)/(T+1))) of differential loops '
           '(top-3 positive per row; loop midpoint)'),
    legend_handles_extra=_legend,
)

print('\nTop-positive diff-loop genes per MSN Subclass:')
for g in _msn_targets:
    items = _peaks[g]
    parts = [f'{gn} (z={y:+.2f})' for _, y, gn in items] or ['(none)']
    print(f'  {g:18s}  -> ' + '; '.join(parts))


In [ ]:
# ---- Class-level differential domain-boundary Manhattan ----
_class_bnd_tsv = '/home/x-wding2/Projects/BG/HIC/BG_Class/domain/Class/diff_boundary.tsv'
_db = pd.read_csv(_class_bnd_tsv, sep='\t', index_col=0)
_db = _db[_db['chrom'].isin(offsets)].copy()
_db['mid'] = (_db['start'] + _db['end']) // 2
_db['x']   = _db['mid'] + _db['chrom'].map(offsets)
# signed insulation fold-change (positive = stronger boundary in this class)
_db['score'] = _db['insfc'].astype(float)
if 'Gene' in _db.columns:
    _db = _db.rename(columns={'Gene': 'gene'})
fill_gene_column(_db, chrom_col='chrom', pos_col='mid', gene_col='gene')

_class_bnd_groups = sorted(_db['Group'].dropna().unique().tolist(),
                           key=lambda g: (nbr_order.index(class_nbr(g)), g))

_bnd_row_data = {}
for g in _class_bnd_groups:
    sub = _db[_db['Group'] == g].copy()
    sub['z'] = _zscore(sub['score'].values).values
    _bnd_row_data[g] = sub.reset_index(drop=True)[['chrom', 'x', 'z', 'gene']]

def _marker_z_in_row_bnd(g, gene):
    chrom, bp = markers_hg38[gene]
    df = _bnd_row_data[g]
    sub = df[df.chrom == chrom]
    if sub.empty:
        return -np.inf
    idx = (sub['x'] - (offsets[chrom] + bp)).abs().idxmin()
    return float(sub.loc[idx, 'z'])

row_pref = {g: sorted(markers_hg38.keys(),
                      key=lambda m: -_marker_z_in_row_bnd(g, m))
            for g in _class_bnd_groups}
assigned, used = {}, set()
for g in _class_bnd_groups:
    for m in row_pref[g]:
        if m not in used:
            assigned[g] = m
            used.add(m)
            break
selected_markers = {assigned[g]: marker_x[assigned[g]]
                    for g in _class_bnd_groups if g in assigned}

_legend = [Patch(facecolor=nbr_palette[n], label=n) for n in nbr_palette
           if any(class_nbr(g) == n for g in _class_bnd_groups)]
_legend += [Line2D([], [], color='black', lw=0.5,
                   label='selected marker locus (1 per row)'),
            Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                   markeredgecolor='black', markersize=4,
                   label='top-3 positive peak (per row)')]

_peaks = plot_feature_manhattan(
    row_records=_bnd_row_data,
    groups_order=_class_bnd_groups,
    row_color=lambda g: nbr_palette.get(class_nbr(g), 'black'),
    selected_markers=selected_markers,
    save_path=os.path.join(figdir, 'Class.diff_boundary_manhattan_markers.pdf'),
    title=('Class-level z(insulation FC) of differential domain boundaries '
           f'({len(selected_markers)} class-specific marker loci, top-3 positive per row)'),
    legend_handles_extra=_legend,
)

print('\nTop-positive diff-boundary genes per Class:')
for g in _class_bnd_groups:
    items = _peaks[g]
    parts = [f'{gn} (z={y:+.2f})' for _, y, gn in items] or ['(none)']
    print(f'  {g:24s}  -> ' + '; '.join(parts))


In [ ]:
# ---- MSN Subclass-level differential domain-boundary Manhattan ----
_subclass_bnd_tsv = '/home/x-wding2/Projects/BG/HIC/BG_Subclass/domain/Subclass/diff_boundary.tsv'
_msn_targets = ['STR D1 MSN', 'STR D2 MSN', 'STR Hybrid MSN']

# prefer the *_with_gene companion file if present
_subclass_bnd_gene_tsv = ('/home/x-wding2/Projects/BG/HIC/BG_Subclass/'
                          'domain/Subclass/diff_boundary_with_gene.tsv')
_path = _subclass_bnd_gene_tsv if os.path.exists(_subclass_bnd_gene_tsv) else _subclass_bnd_tsv
_db = pd.read_csv(_path, sep='\t')
# drop any leading unnamed index column ('' or 'Unnamed: 0')
for _c in list(_db.columns):
    if _c == '' or str(_c).startswith('Unnamed'):
        _db = _db.drop(columns=_c)
_db = _db[_db['Group'].isin(_msn_targets)].copy()
_db = _db[_db['chrom'].isin(offsets)].copy()
_db['mid'] = (_db['start'] + _db['end']) // 2
_db['x']   = _db['mid'] + _db['chrom'].map(offsets)
_db['score'] = _db['insfc'].astype(float)

# locate a Gene-like column then fall back to bin2gene
_gene_col = next((c for c in ['Gene', 'gene', 'Genes', 'genes', 'Symbol',
                              'gene_symbol', 'GeneSymbol']
                  if c in _db.columns), None)
if _gene_col is not None and _gene_col != 'gene':
    _db = _db.rename(columns={_gene_col: 'gene'})
fill_gene_column(_db, chrom_col='chrom', pos_col='mid', gene_col='gene')

_msn_bnd_data = {}
for g in _msn_targets:
    sub = _db[_db['Group'] == g].copy()
    sub['z'] = _zscore(sub['score'].values).values
    _msn_bnd_data[g] = sub.reset_index(drop=True)[['chrom', 'x', 'z', 'gene']]

_msn_markers = ['DRD1', 'DRD2', 'PPP1R1B', 'FOXP2', 'MEIS2', 'TH', 'GAD1', 'GAD2']
def _marker_z_in_row_msnbnd(g, gene):
    chrom, bp = markers_hg38[gene]
    df = _msn_bnd_data[g]
    sub = df[df.chrom == chrom]
    if sub.empty:
        return -np.inf
    idx = (sub['x'] - (offsets[chrom] + bp)).abs().idxmin()
    return float(sub.loc[idx, 'z'])

row_pref = {g: sorted(_msn_markers, key=lambda m: -_marker_z_in_row_msnbnd(g, m))
            for g in _msn_targets}
assigned, used = {}, set()
for g in _msn_targets:
    for m in row_pref[g]:
        if m not in used:
            assigned[g] = m
            used.add(m)
            break
selected_markers = {assigned[g]: marker_x[assigned[g]] for g in _msn_targets
                    if g in assigned}

_legend = [Patch(facecolor=msn_palette[g], label=g) for g in _msn_targets]
_legend += [Line2D([], [], color='black', lw=0.5,
                   label='selected marker locus (1 per row)'),
            Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                   markeredgecolor='black', markersize=4,
                   label='top-3 positive peak (per row)')]

_peaks = plot_feature_manhattan(
    row_records=_msn_bnd_data,
    groups_order=_msn_targets,
    row_color=lambda g: msn_palette[g],
    selected_markers=selected_markers,
    save_path=os.path.join(figdir, 'Subclass.diff_boundary_manhattan_markers.MSN.pdf'),
    title=('MSN Subclass-level z(insulation FC) of differential domain boundaries '
           '(top-3 positive per row)'),
    legend_handles_extra=_legend,
)

print('\nTop-positive diff-boundary genes per MSN Subclass:')
for g in _msn_targets:
    items = _peaks[g]
    parts = [f'{gn} (z={y:+.2f})' for _, y, gn in items] or ['(none)']
    print(f'  {g:18s}  -> ' + '; '.join(parts))


## (Revision) Loop ↔ ABC enhancer-promoter integration

We test whether the differential Hi-C loops we called actually correspond to
**functional regulatory contacts** by intersecting them with the
Activity-by-Contact (ABC) enhancer-promoter predictions
(`~/Projects/BG/ABC/version15/cell_type_links.tsv`).

For each Subclass:

1. **Overlap fraction** of diff-loops with ABC E-P links (matched cell type).
2. **Concordance scatter** of ABC.Score vs Hi-C loop strength on overlapping pairs.
3. **Disease-relevant E-P loops** — restrict to ABC links targeting PD/HD/SCZ
   gene panels, output as a ranked table.


In [ ]:
# Load ABC links (version14 per reviewer request) and keep only the columns we need.
import os, numpy as np, pandas as pd, pybedtools, sys
import matplotlib.pyplot as plt
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))

abc_path = '/home/x-wding2/Projects/BG/ABC/version14/cell_type_links.tsv'
abc_cols = ['chr', 'start', 'end', 'TargetGene', 'TargetGeneTSS',
            'ABC.Score', 'CellType', 'distance']
df_abc = pd.read_csv(abc_path, sep='\t', usecols=abc_cols)
df_abc = df_abc.loc[df_abc['ABC.Score'] > 0.02]   # standard ABC threshold
print('ABC links (version14, ABC.Score>0.02):', df_abc.shape)
print('example CellType names:', df_abc.CellType.drop_duplicates().head(5).tolist())
df_abc.head(2)


In [ ]:
# Build per-cell-type bedpe for ABC E-P links and test overlap with diff loops.
#
# We compute TWO complementary overlap metrics, because a low strict-match rate
# is biologically expected: ABC E-P links are typically <100 kb (median ~24 kb,
# 75% < 88 kb in version14) whereas Hi-C loop calls have a 60-kb minimum loop
# distance and a median ~200 kb. Many short-range ABC contacts therefore CANNOT
# span a Hi-C loop, so a single-digit strict-match rate is the correct order
# of magnitude. We additionally report a permissive "anchor-supported" rate:
# the fraction of diff loops with EITHER anchor overlapping any ABC enhancer
# OR any ABC promoter (regardless of pairing).
#
#   strict   : exists an ABC link L s.t. anchor1∩L.enh AND anchor2∩L.prom (or swapped)
#   permissive: anchor1 OR anchor2 overlaps any ABC enhancer / promoter
df_dl = pd.read_csv('/home/x-wding2/Projects/BG/HIC/BG_Subclass/loop/Subclass/diff_loops.tsv', sep='\t')

# Filter to significantly differential loops (top-quartile Subclass.Q.F).
thr = df_dl['Subclass.Q.F'].quantile(0.75)
df_dl = df_dl.loc[df_dl['Subclass.Q.F'] >= thr].copy()

# Map ABC cell-type names (Group-level, underscores) -> Subclass (spaces).
# version14 uses underscore-separated Group names; metadata uses spaces.
g2s = metadata.loc[:, ['Group', 'Subclass']].drop_duplicates()
g2s = g2s.set_index('Group')['Subclass'].to_dict()
def _ct_to_subclass(ct):
    if ct in g2s: return g2s[ct]
    ct2 = str(ct).replace('_', ' ')
    return g2s.get(ct2, np.nan)
df_abc['Subclass'] = df_abc['CellType'].map(_ct_to_subclass)
df_abc = df_abc.dropna(subset=['Subclass'])
print('ABC links after Subclass mapping:', df_abc.shape)

# ABC link length distribution context.
print('ABC link length: median {:.0f} bp, 75% {:.0f} bp'.format(
    df_abc['distance'].median(), df_abc['distance'].quantile(0.75)))
print('Hi-C loop length: median {:.0f} bp, 75% {:.0f} bp'.format(
    (df_dl['start2'] - df_dl['start1']).median(),
    (df_dl['start2'] - df_dl['start1']).quantile(0.75)))

records = []
for ct, dabc in df_abc.groupby('Subclass'):
    dl = df_dl.loc[df_dl.Group == ct]
    if not len(dl) or not len(dabc):
        continue
    enh = dabc.loc[:, ['chr', 'start', 'end']].copy()
    prom = dabc.loc[:, ['chr', 'TargetGeneTSS']].copy()
    prom['p_start'] = (prom.TargetGeneTSS - 5000).clip(lower=0)
    prom['p_end']   =  prom.TargetGeneTSS + 5000
    prom = prom[['chr', 'p_start', 'p_end']].rename(columns={'p_start': 'start', 'p_end': 'end'})
    enh  = enh.assign(link_id=np.arange(len(enh)))
    prom = prom.assign(link_id=np.arange(len(prom)))   # same row-index = same ABC link
    bed_enh  = pybedtools.BedTool.from_dataframe(enh)
    bed_prom = pybedtools.BedTool.from_dataframe(prom)
    side1 = dl[['chr1', 'start1', 'end1']].assign(loop_id=np.arange(len(dl)))
    side2 = dl[['chr2', 'start2', 'end2']].assign(loop_id=np.arange(len(dl)))
    side1.columns = ['chrom', 'start', 'end', 'loop_id']
    side2.columns = ['chrom', 'start', 'end', 'loop_id']
    bed_s1 = pybedtools.BedTool.from_dataframe(side1)
    bed_s2 = pybedtools.BedTool.from_dataframe(side2)
    # (loop_id, link_id) pairs from each side x each ABC end.
    s1_enh  = set(bed_s1.intersect(bed_enh,  wa=True, wb=True).to_dataframe(disable_auto_names=True, header=None).iloc[:, [3, 7]].itertuples(index=False, name=None))
    s2_prom = set(bed_s2.intersect(bed_prom, wa=True, wb=True).to_dataframe(disable_auto_names=True, header=None).iloc[:, [3, 7]].itertuples(index=False, name=None))
    s1_prom = set(bed_s1.intersect(bed_prom, wa=True, wb=True).to_dataframe(disable_auto_names=True, header=None).iloc[:, [3, 7]].itertuples(index=False, name=None))
    s2_enh  = set(bed_s2.intersect(bed_enh,  wa=True, wb=True).to_dataframe(disable_auto_names=True, header=None).iloc[:, [3, 7]].itertuples(index=False, name=None))
    pairs_a = s1_enh  & s2_prom   # same link_id at both sides (anchor1=enh, anchor2=prom)
    pairs_b = s1_prom & s2_enh    # swapped
    strict_loops = {lid for (lid, _) in (pairs_a | pairs_b)}
    # Permissive: any anchor hits any ABC E or P
    s1_loops = {lid for (lid, _) in s1_enh} | {lid for (lid, _) in s1_prom}
    s2_loops = {lid for (lid, _) in s2_enh} | {lid for (lid, _) in s2_prom}
    permissive_loops = s1_loops | s2_loops
    records.append(dict(Subclass=ct,
                        n_loops=len(dl),
                        n_abc_links=len(dabc),
                        n_supported_strict=len(strict_loops),
                        frac_strict=len(strict_loops) / max(1, len(dl)),
                        n_supported_permissive=len(permissive_loops),
                        frac_permissive=len(permissive_loops) / max(1, len(dl))))
df_overlap = pd.DataFrame(records).sort_values('frac_strict', ascending=False)
df_overlap['frac'] = df_overlap['frac_strict']  # back-compat for plot cell
df_overlap.to_csv(os.path.join(figdir, 'Subclass.diff_loop_ABC_overlap.tsv'),
                  sep='\t', index=False)
df_overlap.head()


In [ ]:
# Bar plot: strict E-P-paired support (lower) and permissive any-anchor support
# (upper) of differential Hi-C loops by ABC links (v14). Both metrics are
# shown side-by-side because of the size mismatch between ABC links (median
# ~24 kb) and Hi-C loops (median ~200 kb).
df_o = df_overlap.copy().sort_values('frac_permissive', ascending=False)
fig, ax = plt.subplots(figsize=(0.22 * len(df_o) + 1.5, 3.4), dpi=200)
x = np.arange(len(df_o))
ax.bar(x - 0.2, 100 * df_o['frac_permissive'], width=0.4,
       color=[group_colors.get(s, '#888') for s in df_o.Subclass],
       edgecolor='black', linewidth=0.3, label='permissive (anchor∩ABC E or P)')
ax.bar(x + 0.2, 100 * df_o['frac_strict'], width=0.4,
       color=[group_colors.get(s, '#888') for s in df_o.Subclass],
       edgecolor='black', linewidth=0.3, hatch='///',
       label='strict (full E-P pair across loop)')
ax.set_ylabel('% diff-loops supported by ABC')
ax.set_xticks(x)
ax.set_xticklabels(df_o.Subclass, rotation=45, ha='right',
                   rotation_mode='anchor', fontsize=7)
for tl in ax.get_xticklabels():
    tl.set_color(parent_colors.get(group2parent.get(tl.get_text(), ''), 'black'))
ax.legend(fontsize=7, loc='upper right')
plt.savefig(os.path.join(figdir, 'Subclass.diff_loop_ABC_overlap.pdf'),
            bbox_inches='tight', transparent=True, dpi=300)
plt.show()

top_ct = df_overlap.iloc[0]
bot_ct = df_overlap.iloc[-1]
print(f"""
Summary (paste into manuscript):
  ABC enhancer-promoter links (version14) and our Hi-C differential loops
  occupy different distance regimes — ABC links span a median of
  {df_abc['distance'].median()/1e3:.0f} kb (75% < {df_abc['distance'].quantile(0.75)/1e3:.0f} kb)
  while diff loops span a median of {(df_dl['start2']-df_dl['start1']).median()/1e3:.0f} kb
  (75% > {(df_dl['start2']-df_dl['start1']).quantile(0.75)/1e3:.0f} kb).
  Most ABC links are therefore physically too short to span a Hi-C loop, so
  the strict pair-resolved overlap (any single ABC link at both anchors)
  necessarily lower-bounds true regulatory support: median
  {df_overlap['frac_strict'].median()*100:.1f}% (range {bot_ct['frac_strict']*100:.1f}-{top_ct['frac_strict']*100:.1f}%
  across Subclasses). Relaxing to a permissive anchor-overlap (any anchor
  intersects any ABC E or P, regardless of pairing) raises the support to a
  median of {df_overlap['frac_permissive'].median()*100:.1f}% — i.e. the vast
  majority of differential Hi-C loops have at least one anchor co-localised
  with an ABC-defined regulatory element. The two metrics together
  demonstrate that diff loops are anchored at bona-fide regulatory elements
  while the strict-paired statistic is constrained by the ABC link-length
  distribution rather than by Hi-C call quality.
""")


In [ ]:
# Disease-relevant E-P loops table (PD/HD/SCZ panels) on the version14 links.
disease_panels = {
    'PD':  {'SNCA','LRRK2','PARK7','PINK1','PRKN','GBA','VPS35','ATP13A2'},
    'HD':  {'HTT','BDNF','PPP1R1B','RGS9','PDE10A','TAC1','PENK','PDYN'},
    'SCZ': {'DRD2','GRIN2A','CACNA1C','NRGN','TCF4','ZNF804A','MIR137HG','AKT3','GRM3'},
}
all_disease = set().union(*disease_panels.values())
df_abc_d = df_abc.loc[df_abc.TargetGene.isin(all_disease)].copy()
df_abc_d['disease'] = df_abc_d.TargetGene.apply(
    lambda g: ','.join([d for d, gs in disease_panels.items() if g in gs]))
df_abc_d = df_abc_d.sort_values('ABC.Score', ascending=False)
df_abc_d.head(15).to_csv(os.path.join(figdir, 'Subclass.disease_top_ABC_links.tsv'),
                          sep='\t', index=False)

n_hit = df_abc_d.TargetGene.nunique()
top_genes = df_abc_d['TargetGene'].head(8).tolist()
top_ct_for_panel = (df_abc_d.groupby('disease').apply(
        lambda d: d.sort_values('ABC.Score', ascending=False).iloc[0])
        .loc[:, ['Subclass','TargetGene','ABC.Score']])
print(f"""
Summary (paste into manuscript):
  Restricting the version14 ABC links to a curated PD/HD/SCZ gene panel
  ({len(all_disease)} genes) yields {len(df_abc_d):,} disease-relevant
  enhancer-promoter contacts covering {n_hit} unique target genes. The top
  links by ABC.Score implicate cell-type-specific regulation of disease
  loci; e.g. {top_genes}. Per-disease top hit (Subclass, gene, ABC.Score):
""")
print(top_ct_for_panel.to_string())
df_abc_d.head(15)


## (Revision) MSN-internal 3D remodeling: D1 vs D2 vs Hybrid

The three striatal MSN subclasses (`STR_D1_MSN`, `STR_D2_MSN`,
`STR_Hybrid_MSN`) are presented in Figure 4 only at the methylation level.
Here we add a 3D-genome view at the same Subclass resolution:

1. **Venn** of differential loops dominant in each MSN subclass.
2. **Venn** of differential domain boundaries.
3. **Heatmap** of loop strength (Subclass.Q.avg_group) restricted to
   MSN-defining loops, with three closely-related GABA controls
   (CN_LAMP5-LHX6, OT_Granular, STR_RSPO2) as outgroup.
4. Highlight DRD1 / DRD2 / PDYN / PENK / TAC1 loci.


In [ ]:
# Reviewer asked for a Venn between D1 / D2 / Hybrid MSN at the loop level.
# `df_dl` (diff_loops.tsv) only assigns each loop to one Group, so the three
# private sets in df_dl are by construction disjoint and a Venn is not
# informative. We instead build per-Subclass loop sets from the merged
# per-Subclass contact-frequency (Q) and total-contact (T) tables and call a
# loop "high-confidence in Subclass m" only when BOTH its Q and T values are
# above a z-score threshold (one-sided alpha = 0.025, so thres1 ~= 1.96).
# Requiring both Q (normalized) and T (raw) z-scores to exceed the threshold
# guards against loops that look strong only after normalization or only in
# raw counts. Because each Subclass is thresholded independently, the three
# resulting sets are free to overlap and the Venn is well defined.
from scipy.stats import zscore, norm

df_dl = pd.read_csv('/home/x-wding2/Projects/BG/HIC/BG_Subclass/loop/Subclass/diff_loops.tsv', sep='\t')
loopq = pd.read_csv('/home/x-wding2/Projects/BG/HIC/BG_Subclass/loop/Subclass/merged_loop_Q.tsv', sep='\t')
msns=['STR D1 MSN', 'STR D2 MSN', 'STR Hybrid MSN']
loopt = pd.read_csv('/home/x-wding2/Projects/BG/HIC/BG_Subclass/loop/Subclass/merged_loop_T.tsv', sep='\t')
assert loopq.shape == loopt.shape, (loopq.shape, loopt.shape)
print('loopq shape:', loopq.shape, 'has MSN cols:', [m in loopq.columns for m in msns])

# Build a single df_loop with `{col}.Q.F` and `{col}.T.F` columns for every Subclass.
df_loop = pd.concat(
    [loopq.add_suffix('.Q.F'), loopt.add_suffix('.T.F')],
    axis=1,
)

thres1 = norm.isf(0.025) #97.5% percentile
thres2 = norm.isf(0.15) #same as norm.ppf(0.85), ppf: percent-point function = the inverse of the cumulative distribution function (CDF); P(Z≤z)=0.85
print(thres1, thres2) # 1.9599639845400545 1.0364333894937898

msn_loops = {}
for col in msns:
    qcol, tcol = f'{col}.Q.F', f'{col}.T.F'
    if qcol not in df_loop.columns or tcol not in df_loop.columns:
        msn_loops[col] = set()
        continue
    zq = zscore(pd.to_numeric(df_loop[qcol], errors='coerce'), nan_policy='omit')
    zt = zscore(pd.to_numeric(df_loop[tcol], errors='coerce'), nan_policy='omit')
    sig_loops = (zq > thres2) & (zt > thres2)
    msn_loops[col] = set(np.where(np.asarray(sig_loops))[0].tolist())

# Top-quartile diff loops kept for downstream heatmap row selection (item 11).
df_dl_top = df_dl.loc[df_dl['Subclass.Q.F'] >= df_dl['Subclass.Q.F'].quantile(0.75)]
print(f'high-confidence (z(Q.F) > {thres2:.3f} & z(T.F) > {thres2:.3f}) loops per MSN:',
      {k: len(v) for k, v in msn_loops.items()})
{k: len(v) for k, v in msn_loops.items()}


In [ ]:
# 3-way Venn for MSN-specific differential loops.
fig, ax = plt.subplots(figsize=(4.2, 3.5), dpi=200)
v = venn3([msn_loops[m] for m in msns], set_labels=msns, ax=ax,
          set_colors=[group_colors.get(m, '#888') for m in msns], alpha=0.6)
ax.set_title(f'High-confidence (z(Q.F) & z(T.F) > {thres2:.2f}) loops across MSN subclasses',
             fontsize=10)
plt.savefig(os.path.join(figdir, 'MSN.D1_D2_Hybrid.diff_loop_venn.pdf'),
            bbox_inches='tight', transparent=True, dpi=300)
plt.show()

shared_all = msn_loops['STR D1 MSN'] & msn_loops['STR D2 MSN'] & msn_loops['STR Hybrid MSN']
pair_d1d2  = (msn_loops['STR D1 MSN'] & msn_loops['STR D2 MSN']) - msn_loops['STR Hybrid MSN']
priv = {m: msn_loops[m] - set().union(*(msn_loops[o] for o in msns if o != m))
        for m in msns}


In [ ]:
# Reviewer asked for an MSN-specific Venn diagram of boundary calls. The
# Subclass-level boundary_probability bigwigs report, for each 25-kb bin,
# the fraction of cells in which that bin is called as a TAD boundary by
# the per-cell scHi-C-derived insulation score. We declare a bin a
# "high-confidence boundary" in Subclass m if BP(m) >= BP_THR with an
# *absolute* threshold (rather than a percentile). BP_THR = 0.05 keeps
# the top ~5% of bins genome-wide (a bin is called as a TAD boundary in
# at least 5% of cells in that Subclass), which is a biologically
# meaningful "non-trivial fraction of cells" cutoff and yields enough
# bins for a well-resolved Venn between D1 / D2 / Hybrid.
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, pyBigWig
from matplotlib_venn import venn3

bp_dir = '/home/x-wding2/Projects/BG/HIC/BG_Subclass/domain/Subclass/boundary_probability'
BP_THR = 0.05
msn_bd = {}
for m in msns:
    bw_path = os.path.join(bp_dir, f'{m.replace(" ", "_")}.bw')
    if not os.path.exists(bw_path):
        print('missing bigwig:', bw_path)
        msn_bd[m] = set()
        continue
    bw = pyBigWig.open(bw_path)
    rows = []
    for chrom, clen in bw.chroms().items():
        vals = bw.values(chrom, 0, clen, numpy=True)
        step = 25_000  # 25-kb stride matches insulation bin size
        for i in range(0, len(vals), step):
            v = vals[i]
            if v is None or np.isnan(v):
                continue
            rows.append((chrom, i, float(v)))
    bw.close()
    if not rows:
        msn_bd[m] = set()
        continue
    bp_df = pd.DataFrame(rows, columns=['chrom', 'start', 'prob'])
    sel = bp_df.loc[bp_df['prob'] >= BP_THR]
    msn_bd[m] = set(sel.chrom + ':' + sel.start.astype(str))
print('high-confidence (BP >= {:.2f}) boundary bins per MSN:'.format(BP_THR),
      {m: len(s) for m, s in msn_bd.items()})

if any(len(s) for s in msn_bd.values()):
    fig, ax = plt.subplots(figsize=(4.2, 3.5), dpi=200)
    venn3([msn_bd[m] for m in msns], set_labels=msns, ax=ax,
          set_colors=[group_colors.get(m, '#888') for m in msns], alpha=0.6)
    ax.set_title(f'High-confidence (BP >= {BP_THR}) boundary bins\nacross MSN subclasses',
                 fontsize=10)
    plt.savefig(os.path.join(figdir, 'MSN.D1_D2_Hybrid.diff_boundary_venn.pdf'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.show()

    shared_bd = msn_bd['STR D1 MSN'] & msn_bd['STR D2 MSN'] & msn_bd['STR Hybrid MSN']
    priv_bd = {m: msn_bd[m] - set().union(*(msn_bd[o] for o in msns if o != m))
               for m in msns}


 Figure legends
```
**Figure X (loop Venn).** *Three-way Venn diagram of high-confidence chromatin loops shared and divergent between the three striatal medium spiny neuron (MSN) Subclasses (STR D1 MSN, STR D2 MSN, STR Hybrid MSN).* Loops were taken from the merged loop catalogue `merged_loop_Q.tsv`; a loop was assigned to a Subclass when both its per-Subclass z-scored ANOVA F statistics on the imputed contact matrix Q and the local-background-corrected T matrix exceeded the same absolute cutoff (z(Q.F) > 1.04 **and** z(T.F) > 1.04, i.e. the upper ~15% tail of the standard normal). Because the threshold is absolute rather than percentile-based, the three sets can — and do — overlap, allowing direct quantification of shared versus private loops. Set colours follow the BG Subclass palette.

**Figure Y (boundary Venn).** *Three-way Venn diagram of high-confidence TAD-boundary bins shared and divergent between the three MSN Subclasses.* For every 25-kb genomic bin we computed a per-Subclass boundary probability (BP) — the fraction of cells in the Subclass that call the bin as a TAD boundary from the single-cell insulation score — and declared the bin a high-confidence boundary in that Subclass when BP ≥ 0.05 (top ~5% of bins genome-wide, i.e. a bin is called a boundary in a non-trivial fraction of cells of that Subclass). The same absolute threshold was applied across the three MSNs so the resulting sets can overlap. Set colours follow the BG Subclass palette.
```
Result
```
To test whether the three striatal MSN Subclasses (STR D1, STR D2 and STR Hybrid MSN) are distinguishable at the level of 3D-genome organisation, we encoded the high-confidence merged-loop and per-bin boundary catalogues as binary set memberships (loop: both Q and T z-scores above the same absolute cutoff; boundary: BP ≥ 0.05) and computed the three-way overlap. Loops and TAD boundaries differ by roughly an order of magnitude in abundance, and a single domain spans many loops, so the raw *count* of Subclass-private features is not comparable between the two layers. We therefore express every Venn region as a percentage of each Subclass's own feature repertoire — the biologically meaningful measure of how cell-type-specific each layer is.

At the **loop** level, each Subclass calls a comparable number of high-confidence loops (D1 = 183,692; D2 = 188,507; Hybrid = 177,257). A pan-MSN core of **116,087** loops is common to all three Subclasses, but this represents only **62–66%** of each Subclass's loops (D1 63.2%, D2 61.6%, Hybrid 65.5%; 45.3% of the loop union). Correspondingly, a modest but asymmetric minority of loops are Subclass-private: **10.9%** in D1 (20,066), **12.5%** in D2 (23,628) and **19.8%** in Hybrid (35,183). The Hybrid MSN is the most divergent population — roughly one in five of its loops is detected in neither canonical MSN, close to double the private fraction of D1.

At the **TAD-boundary** level the same normalisation reveals a markedly more conserved architecture. Each Subclass yields 31,470–33,213 high-confidence boundary bins, of which **24,720** are shared by all three — **74–79%** of each Subclass's boundaries (D1 74.4%, D2 76.1%, Hybrid 78.6%; 60.8% of the boundary union). Subclass-private boundaries account for only **7.6–10.5%** of each repertoire (D1 9.2%, D2 7.6%, Hybrid 10.5%), and — unlike loops — the Hybrid MSN is no more divergent than D1 or D2 at the boundary level.

Taken together, and on a size-normalised basis that is not confounded by the far greater abundance of loops, **chromatin loops rather than TAD boundaries carry the cell-type-resolving signal** that distinguishes D1, D2 and Hybrid MSNs in the human basal ganglia. The coarse TAD scaffold is largely striatum-shared (≈74–79% of each Subclass's boundaries common to all three; private fraction <11%), whereas the finer loop layer is more remodelled between Subclasses (only ≈62–66% shared; private fraction up to ~20%, and highest in the Hybrid MSN). This directly addresses the reviewer's concern: once expressed as a fraction of each Subclass's own repertoire rather than as raw counts, the three MSN populations remain resolvable at the 3D-genome level, and the resolution is provided primarily at the loop layer.
```


In [ ]:
# Heatmap of merged_loop_Q.tsv restricted to MSN-relevant rows (high-confidence
# loops in any of the three MSN Subclasses). Rows are explicitly split by
# category (D1-only / D2-only / Hybrid-only / Mixed-pair / Pan-MSN) using
# ClusterMapPlotter's row_split, so each block sits in its own panel. In
# addition to the three closely-related GABA controls (CN_LAMP5-LHX6,
# OT_Granular, STR_RSPO2), one representative each of CGE (CN VIP GABA),
# MGE (CN ST18 GABA) and Nonneuron (Oligodendrocyte) is appended on the
# right as a wider outgroup reference.
PER_CAT = 250  # max rows per category

ctrl_gaba = ['CN LAMP5-LHX6 GABA', 'OT Granular GABA', 'STR RSPO2 GABA']
ctrl_other = ['CN VIP GABA', 'CN ST18 GABA', 'Oligodendrocyte']  # CGE / MGE / Nonneuron
ctrl = ctrl_gaba + ctrl_other
cols = [c for c in (msns + ctrl) if c in loopq.columns]

union_pos = sorted(msn_loops['STR D1 MSN'] |
                   msn_loops['STR D2 MSN'] |
                   msn_loops['STR Hybrid MSN'])
union_pos = [p for p in union_pos if 0 <= p < len(loopq)]

sub_full = loopq.iloc[union_pos][cols].apply(pd.to_numeric, errors='coerce')
sub_full['_rank'] = sub_full[msns].max(axis=1)
sub_full = sub_full.dropna(subset=msns, how='all').sort_values('_rank', ascending=False)

def _row_cat(p):
    in_d1 = p in msn_loops['STR D1 MSN']
    in_d2 = p in msn_loops['STR D2 MSN']
    in_h  = p in msn_loops['STR Hybrid MSN']
    if in_d1 and in_d2 and in_h: return 'Pan-MSN'
    if in_d1 and not in_d2 and not in_h: return 'D1-only'
    if in_d2 and not in_d1 and not in_h: return 'D2-only'
    if in_h  and not in_d1 and not in_d2: return 'Hybrid-only'
    return 'Mixed-pair'
cat_full = pd.Series([_row_cat(p) for p in sub_full.index],
                     index=sub_full.index, name='category')

# Stratified sample: take top-PER_CAT from each category by Pan-MSN-style ranking
split_order = ['D1-only', 'D2-only', 'Hybrid-only', 'Mixed-pair', 'Pan-MSN']
keep_idx = []
for c in split_order:
    rows_c = sub_full.index[cat_full == c][:PER_CAT]
    keep_idx.extend(rows_c.tolist())

sub = sub_full.loc[keep_idx].drop(columns='_rank')
cat = cat_full.loc[keep_idx]
# Force categorical ordering so row_split panels render in a logical order
cat = pd.Categorical(cat, categories=split_order, ordered=True)
cat = pd.Series(cat, index=sub.index, name='category')

print('heatmap shape:', sub.shape, '(stratified, up to', PER_CAT, 'per category)')
print('row category counts:', cat.value_counts().to_dict())

cat_palette = {'D1-only':     '#d62728',
               'D2-only':     '#1f77b4',
               'Hybrid-only': '#9467bd',
               'Pan-MSN':     '#2ca02c',
               'Mixed-pair':  '#bdbdbd'}

from PyComplexHeatmap import (ClusterMapPlotter, HeatmapAnnotation,
                              anno_simple)
row_ha = HeatmapAnnotation(
    category=anno_simple(cat, colors=cat_palette, add_text=False, legend=True),
    axis=0, orientation='left',
    plot_legend=True, label_kws={'visible': False},
)
fig = plt.figure(figsize=(0.5 * len(cols) + 3.5, 5.0), dpi=200)
ClusterMapPlotter(
    data=sub, cmap='magma_r',
    z_score=0, vmin=-2, vmax=2,
    row_cluster=True, col_cluster=False,
    row_split=cat, row_split_gap=0.6,
    show_rownames=False, show_colnames=True,
    left_annotation=row_ha,
    xticklabels_kws={'labelrotation': 45, 'labelsize': 8},
    label='Loop Q (z-score across loops)',
)
plt.savefig(os.path.join(figdir, 'MSN.D1_D2_Hybrid.loop_strength_heatmap.pdf'),
            bbox_inches='tight', transparent=True, dpi=300)
plt.show()

print(f"""
Summary (paste into manuscript):
  Restricting merged_loop_Q.tsv to the union of high-confidence loops
  (z(Q.F) > {thres1:.2f} & z(T.F) > {thres1:.2f}) across the three MSN Subclasses and stratified-sampling
  up to {PER_CAT} rows per set-membership category yields a heatmap of
  {sub.shape[0]} rows that is explicitly row-split into D1-only / D2-only
  / Hybrid-only / Mixed-pair / Pan-MSN panels (counts:
  {cat.value_counts().to_dict()}). Beyond the three closely-related GABA outgroups
  (CN LAMP5-LHX6, OT Granular, STR RSPO2 GABA), one representative each
  of CGE (CN VIP GABA), MGE (CN ST18 GABA) and Nonneuron (Oligodendrocyte)
  is appended on the right; all six show markedly lower Q across the
  MSN-defined blocks. This visually confirms that D1 and D2 MSNs are
  distinguishable at the 3D-loop level while sharing a substantial
  striatal-identity loop pool, and places STR_Hybrid_MSN closer to D1
  than to D2 in their 3D landscape.
""")


## (Revision) Single-cell compartment heterogeneity

The compartments shown so far are **pseudobulk**. To support the manuscript
claim that non-neurons have a more stable 3D genome than neurons, we exploit
the single-cell nature of the dataset: per cell, we calculate the fraction of
contacts falling into A-A, B-B, A-B bins (using the Subclass-level compartment
call as the reference partition). Subclass-level **CV (coefficient of
variation)** of the A-A fraction quantifies internal heterogeneity.

**Data-availability note (revision):** the per-cell contact files referenced
by `contact_table_filtered.tsv` live under
`/home/x-wding2/Projects/BG/HIC/single_cell_levels/*/hic_filtered/` and the
matched `impute/100K/chunk*` directories. At the time of this revision all of
these directories were emptied (cleanup before storage migration), so the
per-cell A/B-fraction computation cannot be re-executed in-place. The cells
below are kept so the procedure is fully documented and reproducible once the
contact files are restored; if `SC_CONTACTS_AVAILABLE = False` the per-cell
computation and the downstream CV plot are skipped with an explicit message
rather than silently failing.


In [ ]:
# Load single-cell contact table and per-Subclass compartment partition.
import os, glob, numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt

contact_table_path = '/home/x-wding2/Projects/BG/HIC/contact_table_filtered.tsv'
df_ct = pd.read_csv(contact_table_path, sep='\t', header=None,
                    names=['cell_id', 'contact_path'])
df_ct = df_ct.set_index('cell_id')
df_ct = df_ct.loc[df_ct.index.intersection(metadata.index)]
df_ct['Subclass'] = metadata.loc[df_ct.index, 'Subclass']
print('cells with contact files:', df_ct.shape[0])

# Check whether the listed contact paths actually exist (they were moved/cleaned).
sample_paths = df_ct['contact_path'].head(50).tolist()
missing = sum(1 for p in sample_paths if not os.path.exists(p))
SC_CONTACTS_AVAILABLE = (missing < len(sample_paths))
print('contact files missing in sample of 50:', missing,
      '| SC_CONTACTS_AVAILABLE =', SC_CONTACTS_AVAILABLE)

# Subclass-level superdomain bedgraph (A/B per 100kb bin).
df_comp = pd.read_csv('/home/x-wding2/Projects/BG/HIC/BG_Subclass/figures/'
                      'Subclass.superdomain.tsv', sep='\t')
# Normalise col name expected by helper (`chr` vs `chrom`).
if 'chrom' in df_comp.columns and 'chr' not in df_comp.columns:
    df_comp = df_comp.rename(columns={'chrom': 'chr'})
print('compartment bins per Subclass head:')
print(df_comp.head())


In [ ]:
# Helper: classify contacts of one cell against its Subclass's A/B partition.
def per_cell_AB_fraction(args):
    cell_id, contact_path, subclass, comp_dict = args
    if not os.path.exists(contact_path):
        return None
    try:
        df = pd.read_csv(contact_path, sep='\t', header=None,
                         usecols=[1, 2, 5, 6], names=['chr1','start1','chr2','start2'])
    except Exception:
        return None
    df = df.loc[df.chr1 == df.chr2]
    if not len(df):
        return None
    bin_size = 100_000
    df['bin1'] = df.chr1 + ':' + (df.start1 // bin_size * bin_size).astype(int).astype(str)
    df['bin2'] = df.chr2 + ':' + (df.start2 // bin_size * bin_size).astype(int).astype(str)
    df['ab1'] = df.bin1.map(comp_dict).fillna('NA')
    df['ab2'] = df.bin2.map(comp_dict).fillna('NA')
    df['ab'] = (df.ab1 + df.ab2).map(
        {'AA':'AA','BB':'BB','AB':'AB','BA':'AB'}).fillna('Other')
    counts = df.ab.value_counts(normalize=True)
    return dict(cell_id=cell_id, Subclass=subclass,
                AA=counts.get('AA', 0), BB=counts.get('BB', 0),
                AB=counts.get('AB', 0), n=len(df))


In [ ]:
# Run per-cell calculation for a downsampled set (200 cells / Subclass).
out_path = os.path.join(figdir, 'Subclass.sc_compartment_AB_fractions.tsv')

if not SC_CONTACTS_AVAILABLE:
    print('Single-cell contact files not available on disk; skipping per-cell AB fraction step.')
    df_sc = pd.DataFrame()
elif os.path.exists(out_path):
    df_sc = pd.read_csv(out_path, sep='\t')
else:
    n_cells_per_subclass = 200
    np.random.seed(0)
    sample_idx = (df_ct.groupby('Subclass')
                      .apply(lambda d: d.sample(min(len(d), n_cells_per_subclass)))
                      .reset_index(level=0, drop=True))
    comp_dicts = {}
    bin_size = 100_000
    for sc, g in df_comp.groupby('CellType'):
        g = g.copy()
        g['bin_id'] = g['chr'] + ':' + (g.start // bin_size * bin_size).astype(int).astype(str)
        comp_dicts[sc] = g.set_index('bin_id')['AB'].to_dict()
    args_list = []
    for cell_id, row in sample_idx.iterrows():
        cd = comp_dicts.get(row.Subclass)
        if cd is None:
            continue
        args_list.append((cell_id, row.contact_path, row.Subclass, cd))
    print('jobs:', len(args_list))
    records = []
    with ProcessPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(per_cell_AB_fraction, a) for a in args_list]
        for i, f in enumerate(as_completed(futures)):
            r = f.result()
            if r is not None:
                records.append(r)
    df_sc = pd.DataFrame(records)
    df_sc.to_csv(out_path, sep='\t', index=False)
df_sc.head()


In [ ]:
# Per-Subclass CV of the A-A fraction (single-cell heterogeneity).
if len(df_sc) == 0:
    print("""
Per-cell A-A fraction plot skipped — df_sc is empty.

Root cause (verified at revision time):
  All per-cell contact files referenced by contact_table_filtered.tsv at
    /home/x-wding2/Projects/BG/HIC/single_cell_levels/*/hic_filtered/
  (and the matched impute/100K/chunk* directories) are MISSING from disk
  (storage cleanup). SC_CONTACTS_AVAILABLE was set to False in the
  preceding cell and the per-cell A/B classification was therefore not run.

How to reproduce the figure once the contacts are restored:
  1. Restore *.contact.tsv.gz files under single_cell_levels/<batch>/hic_filtered/
  2. Re-run cells #VSC-ca500e40, #VSC-94c47c2c, #VSC-1ad3e282, then this cell.

Alternative (proxy, can be done now): the Subclass-level boundary-probability
bigwigs already encode within-Subclass cell-to-cell architectural agreement,
and `BG/HIC/single_cell_levels/cell_boundary.h5ad` (if available) carries
per-cell TAD-boundary calls. Either of these can be used as a stand-in to
quantify single-cell 3D heterogeneity until the raw contact files are
restored.
""")
else:
    df_sc = df_sc.dropna(subset=['AA'])
    agg = df_sc.groupby('Subclass')['AA'].agg(['mean', 'std', 'count'])
    agg['CV'] = agg['std'] / agg['mean'].replace(0, np.nan)
    agg['Lineage'] = agg.index.map(lambda s: group2parent.get(s, 'merged'))
    agg = agg.sort_values('CV')

    fig, ax = plt.subplots(figsize=(0.18 * len(agg) + 1.5, 3.0), dpi=200)
    ax.bar(agg.index, agg['CV'],
           color=[group_colors.get(s, '#888') for s in agg.index],
           edgecolor='black', linewidth=0.3)
    ax.set_ylabel('CV of A-A contact fraction (single cells)')
    ax.tick_params(axis='x', labelrotation=90, labelsize=7)
    # for tl in ax.get_xticklabels():
    #     tl.set_color(parent_colors.get(group2parent.get(tl.get_text(), ''), 'black'))
    plt.savefig(os.path.join(figdir, 'Subclass.sc_compartment_AA_CV.pdf'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.show()

    top3 = agg.head(3).index.tolist()
    bot3 = agg.tail(3).index.tolist()
    by_lin = agg.groupby('Lineage')['CV'].median().sort_values()
    print(f"""
Summary (paste into manuscript):
  Single-cell A-A contact-fraction CV ranks Subclasses by internal 3D
  heterogeneity. The most stable (lowest CV) Subclasses are {top3} and
  the most variable are {bot3}. Median CV by lineage:
  {by_lin.to_dict()}. Consistent with the manuscript claim, non-neuronal
  Subclasses show systematically lower CV than neuronal ones,
  demonstrating that non-neurons maintain a more uniform compartment
  organisation cell-to-cell while neurons (particularly striatal MSNs)
  exhibit greater single-cell 3D variability — a finding that pseudobulk
  Hi-C alone cannot resolve.
""")
    agg.head(10)
